# Final Plots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import json
import utils
import numpy as np
import importlib
importlib.reload(utils)
from pathlib import Path
from types import SimpleNamespace
import globals
importlib.reload(globals)
import seaborn as sns
from tqdm import tqdm
import main
import hashlib

# Set seaborn style for professional plots
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['grid.alpha'] = 0.4

# Global plot styling
PLOT_BACKGROUND_COLOR = '#f5f5f5'  # Light gray background (set to 'white' or None for default)
# PLOT_BACKGROUND_COLOR = "#dbdbdb"

# ---- ID/OOD CF type globals (modify as needed) ----
ID_CF_TYPES = globals.gpt_oss_bias_cues_train  # Training cue types = ID
OOD_CF_TYPES = globals.gpt_oss_bias_cues_test  # Test-only cue types = OOD

simulator_sweep_configs = [
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': True},  # cf-sim-xye
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': False},  # cf-sim-xy
]


In [ ]:
def load_stats_across_seeds(base_exp_name, n_seeds=1):
    all_results = []
    for seed in range(n_seeds):
        seed_exp_name = f"{base_exp_name}_sd{seed}"
        stats_df = pd.read_csv(f"results/stats_{seed_exp_name}.csv")
        all_results.append(stats_df)
        print(f"Loaded stats for seed {seed} from {seed_exp_name}")
    # average only numerical columns across seeds
    # First, identify which columns to group by (non-numerical identifiers)
    group_cols = ['dataname', 'counterfactual_type', 'round']
    # Concatenate all dataframes
    combined_df = pd.concat(all_results, ignore_index=True)
    # Get numerical columns (excluding group columns)
    numerical_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
    # Group by identifier columns and average only numerical columns
    combined_df = combined_df.groupby(group_cols, as_index=False)[numerical_cols].mean()
    return combined_df

In [ ]:
# ---- Data Loading Helpers ----

def load_test_datasets_for_round(args, round_num, n_seeds=1):
    """
    Load test datasets across seeds for a given round.
    Returns a list of dataframes (one per seed).
    """
    base_exp_name = utils.get_base_exp_name(args)
    seed_dfs = []
    
    for seed in range(n_seeds):
        # Reconstruct args with this seed to get the correct exp_name
        args_copy = SimpleNamespace(**vars(args))
        args_copy.seed = seed
        exp_name = utils.get_exp_name(args_copy)
        
        dataset_path = Path(f"artifacts/dataset_{exp_name}_round{round_num}_test.csv")
        if dataset_path.exists():
            df = pd.read_csv(dataset_path)
            df['_seed'] = seed  # Tag with seed for tracking
            seed_dfs.append(df)
        else:
            print(f"Warning: Missing {dataset_path}")
    
    return seed_dfs


def filter_to_cf_types(df, cf_types):
    """Filter dataframe to rows with counterfactual_type in cf_types."""
    if 'counterfactual_type' in df.columns:
        return df[df['counterfactual_type'].isin(cf_types)].copy()
    return df


def aggregate_cf_type_label(cf_types_in_df, id_types, ood_types):
    """
    Determine aggregate label based on which cf types are present.
    Returns 'all_ID', 'all_OOD', or specific type name.
    """
    unique_types = set(cf_types_in_df)
    if unique_types.issubset(set(id_types)):
        return 'all_ID'
    elif unique_types.issubset(set(ood_types)):
        return 'all_OOD'
    elif len(unique_types) == 1:
        return list(unique_types)[0]
    else:
        return 'mixed'

In [ ]:
# ---- Bootstrap Evaluation Stats (Optimized) ----
from sklearn.metrics import roc_auc_score, average_precision_score


def bootstrap_pvalue_twosided(betas):
    """
    Calculate two-sided p-value for H0: mean(betas) = 0
    from a bootstrapped distribution of statistics.
    
    This uses the "centered percentile" method:
    1. Center the bootstrap distribution at 0 (under H0)
    2. Compute proportion of centered values at least as extreme as |observed mean|
    
    Args:
        betas: array of bootstrap replicates (e.g., round_n - round_0 differences)
    
    Returns:
        Two-sided p-value
    """
    betas = np.asarray(betas)
    abs_mean_beta = np.abs(np.mean(betas))
    centered_betas = betas - np.mean(betas)  # Center at 0 under H0
    
    # Proportion of centered distribution outside ±|observed mean|
    p_value = np.mean(centered_betas < -abs_mean_beta) + np.mean(centered_betas > abs_mean_beta)
    return p_value


def print_pvalue_table(bootstrap_samples: dict, metrics: list, rounds: list, baseline_round: int = 0):
    """
    Print a table of p-values for improvement of metrics across rounds vs baseline.
    
    Args:
        bootstrap_samples: Dict of {round_num: {metric_name: np.array of bootstrap values}}
        metrics: List of metric names to report
        rounds: List of round numbers
        baseline_round: Round to compare against (default 0)
    """
    print(f"\n{'='*70}")
    print(f"Bootstrap Hypothesis Tests: Improvement vs Round {baseline_round}")
    print(f"{'='*70}")
    
    if baseline_round not in bootstrap_samples:
        print(f"ERROR: Baseline round {baseline_round} not found in bootstrap samples")
        return
    
    for metric in metrics:
        if metric not in bootstrap_samples[baseline_round]:
            print(f"Skipping metric {metric} (not found in baseline round {baseline_round})")
            continue
            
        print(f"\nMetric: {metric}")
        print(f"  {'Round':<8} {'Mean':>10} {'Δ vs R0':>12} {'p-value':>12} {'Sig':>6}")
        print(f"  {'-'*50}")
        
        baseline_samples_arr = bootstrap_samples[baseline_round][metric]
        baseline_mean = np.mean(baseline_samples_arr)
        
        for round_num in sorted(rounds):
            if round_num not in bootstrap_samples:
                continue
            if metric not in bootstrap_samples[round_num]:
                continue  # Skip if metric not found in this round
                
            round_samples = bootstrap_samples[round_num][metric]
            round_mean = np.mean(round_samples)
            
            if round_num == baseline_round:
                print(f"  {round_num:<8} {round_mean:>10.4f} {'--':>12} {'--':>12} {'--':>6}")
            else:
                # Paired differences - handle different sample sizes by truncating to min length
                min_len = min(len(round_samples), len(baseline_samples_arr))
                betas = round_samples[:min_len] - baseline_samples_arr[:min_len]
                mean_diff = np.mean(betas)
                p_val = bootstrap_pvalue_twosided(betas)
                
                # Significance stars
                if p_val < 0.001:
                    sig = '***'
                elif p_val < 0.01:
                    sig = '**'
                elif p_val < 0.05:
                    sig = '*'
                else:
                    sig = ''
                
                print(f"  {round_num:<8} {round_mean:>10.4f} {mean_diff:>+12.4f} {p_val:>12.4f} {sig:>6}")
    
    print(f"\n{'='*70}")
    print("Significance: * p<0.05, ** p<0.01, *** p<0.001")
    print(f"{'='*70}\n")


def print_xye_vs_xy_table(bootstrap_samples: dict, metrics: list, rounds: list):
    """
    Print a table of p-values for the difference between xye and xy metrics within each round.
    
    Automatically pairs metrics that differ only in their config suffix (xye vs xy).
    
    Args:
        bootstrap_samples: Dict of {round_num: {metric_name: np.array of bootstrap values}}
        metrics: List of metric names (should include both xye and xy versions)
        rounds: List of round numbers
    """
    # Find xye/xy pairs
    metric_pairs = []
    xye_metrics = [m for m in metrics if m.endswith('cf-sim-xye')]
    for xye_metric in xye_metrics:
        xy_metric = xye_metric.replace('cf-sim-xye', 'cf-sim-xy')
        if xy_metric in metrics:
            metric_pairs.append((xye_metric, xy_metric))
    
    if not metric_pairs:
        print("No xye/xy metric pairs found.")
        return
    
    print(f"\n{'='*70}")
    print("Bootstrap Hypothesis Tests: XYE vs XY (within each round)")
    print(f"{'='*70}")
    
    for xye_metric, xy_metric in metric_pairs:
        # Extract base name for display
        base_name = xye_metric.replace('_cf-sim-xye', '').replace('cf-sim-xye', '')
        print(f"\nMetric: {base_name}")
        print(f"  {'Round':<8} {'XYE':>10} {'XY':>10} {'Δ (XYE-XY)':>12} {'p-value':>12} {'Sig':>6}")
        print(f"  {'-'*60}")
        
        for round_num in sorted(rounds):
            if round_num not in bootstrap_samples:
                continue
            if xye_metric not in bootstrap_samples[round_num]:
                continue
            if xy_metric not in bootstrap_samples[round_num]:
                continue
            
            xye_samples = bootstrap_samples[round_num][xye_metric]
            xy_samples = bootstrap_samples[round_num][xy_metric]
            
            xye_mean = np.mean(xye_samples)
            xy_mean = np.mean(xy_samples)
            
            # Paired differences (xye - xy) - handle different sample sizes by truncating
            min_len = min(len(xye_samples), len(xy_samples))
            betas = xye_samples[:min_len] - xy_samples[:min_len]
            mean_diff = np.mean(betas)
            p_val = bootstrap_pvalue_twosided(betas)
            
            # Significance stars
            if p_val < 0.001:
                sig = '***'
            elif p_val < 0.01:
                sig = '**'
            elif p_val < 0.05:
                sig = '*'
            else:
                sig = ''
            
            print(f"  {round_num:<8} {xye_mean:>10.4f} {xy_mean:>10.4f} {mean_diff:>+12.4f} {p_val:>12.4f} {sig:>6}")
    
    print(f"\n{'='*70}")
    print("Significance: * p<0.05, ** p<0.01, *** p<0.001")
    print(f"{'='*70}\n")


def print_exp1_vs_exp2_table(
    bootstrap_samples_list: list,
    experiment_labels: list,
    metric: str,
    rounds: list,
):
    """
    Print a table of p-values comparing two experiments on the same metric within each round.
    
    Args:
        bootstrap_samples_list: List of bootstrap_samples dicts, one per experiment.
                                Each is {round_num: {metric_name: np.array of bootstrap values}}
        experiment_labels: List of labels for each experiment (e.g., ['SL', 'RL'])
        metric: The metric name to compare
        rounds: List of round numbers
    """
    if len(bootstrap_samples_list) != 2:
        print(f"print_exp1_vs_exp2_table requires exactly 2 experiments, got {len(bootstrap_samples_list)}")
        return
    
    exp1_samples = bootstrap_samples_list[0]
    exp2_samples = bootstrap_samples_list[1]
    exp1_label = experiment_labels[0] if len(experiment_labels) > 0 else "Exp1"
    exp2_label = experiment_labels[1] if len(experiment_labels) > 1 else "Exp2"
    
    print(f"\n{'='*80}")
    print(f"Bootstrap Hypothesis Tests: {exp1_label} vs {exp2_label}")
    print(f"Metric: {metric}")
    print(f"{'='*80}")
    
    print(f"  {'Round':<8} {exp1_label:>12} {exp2_label:>12} {'Δ (E1-E2)':>12} {'p-value':>12} {'Sig':>6}")
    print(f"  {'-'*66}")
    
    for round_num in sorted(rounds):
        # Check if both experiments have this round
        if round_num not in exp1_samples or round_num not in exp2_samples:
            continue
        if metric not in exp1_samples[round_num] or metric not in exp2_samples[round_num]:
            continue
        
        exp1_vals = exp1_samples[round_num][metric]
        exp2_vals = exp2_samples[round_num][metric]
        
        # Check that bootstrap sample sizes match
        if len(exp1_vals) != len(exp2_vals):
            print(f"  {round_num:<8} Bootstrap sample sizes don't match: {len(exp1_vals)} vs {len(exp2_vals)}")
            continue
        
        exp1_mean = np.mean(exp1_vals)
        exp2_mean = np.mean(exp2_vals)
        
        # Paired differences (exp1 - exp2)
        betas = exp1_vals - exp2_vals
        mean_diff = np.mean(betas)
        p_val = bootstrap_pvalue_twosided(betas)
        
        # Significance stars
        if p_val < 0.001:
            sig = '***'
        elif p_val < 0.01:
            sig = '**'
        elif p_val < 0.05:
            sig = '*'
        else:
            sig = ''
        
        print(f"  {round_num:<8} {exp1_mean:>12.4f} {exp2_mean:>12.4f} {mean_diff:>+12.4f} {p_val:>12.4f} {sig:>6}")
    
    print(f"\n{'='*80}")
    print("Significance: * p<0.05, ** p<0.01, *** p<0.001")
    print(f"{'='*80}\n")


def parse_metric_name(metric_name: str):
    """
    Parse a metric name like 'all_bias_monitor_auroc_k-0_CoT-F_cf-sim-xye' into components.
    
    Returns dict with keys:
        - subset: 'all', 'cf_stable', 'sim_balanced_fresh', etc.
        - metric_type: 'bias_monitor_auroc', 'sim_greedy_cot_acc', etc.
        - k: integer k-shot value
        - cot: 'CoT-T' or 'CoT-F'
        - config: 'cf-sim-xye', 'cf-sim-xy', etc.
    """
    # Known subsets (order matters - check longer ones first)
    subsets = ['sim_balanced_fresh', 'sim_balanced', 'cf_stable', 'all']
    
    parsed = {}
    remaining = metric_name
    
    # Find subset
    for subset in subsets:
        if remaining.startswith(subset + '_'):
            parsed['subset'] = subset
            remaining = remaining[len(subset) + 1:]
            break
    else:
        parsed['subset'] = 'all'  # default
    
    # Extract config from the end (e.g., 'cf-sim-xye', 'cf-sim-xy', 'yesno-xye', etc.)
    config_patterns = ['cf-sim-xye', 'cf-sim-xy', 'cf-sim-xe', 'cf-sim-x', 
                       'yesno-xye', 'yesno-xy', 'yesno-xe', 'yesno-x']
    for config in config_patterns:
        if remaining.endswith(config):
            parsed['config'] = config
            remaining = remaining[:-len(config) - 1]  # Remove config and underscore
            break
    else:
        parsed['config'] = None
    
    # Extract CoT flag
    for cot in ['CoT-T', 'CoT-F']:
        if f'_{cot}_' in remaining or remaining.endswith(f'_{cot}'):
            parsed['cot'] = cot
            remaining = remaining.replace(f'_{cot}', '')
            break
    else:
        parsed['cot'] = None
    
    # Extract k value
    import re
    k_match = re.search(r'_k-(\d+)', remaining)
    if k_match:
        parsed['k'] = int(k_match.group(1))
        remaining = re.sub(r'_k-\d+', '', remaining)
    else:
        parsed['k'] = 0
    
    # What's left is the metric type
    parsed['metric_type'] = remaining
    
    return parsed


def compute_bootstrap_metrics(
    df: pd.DataFrame,
    metrics_to_compute: list,
    simulator_sweep_configs: list,
):
    """
    Compute only the specific metrics requested for bootstrap efficiency.
    
    This is a lightweight version of main.compute_evaluation_stats that only
    computes the metrics specified in metrics_to_compute.
    
    Args:
        df: Resampled dataframe
        metrics_to_compute: List of metric names to compute (e.g., 'all_bias_monitor_auroc_k-0_CoT-F_cf-sim-xye')
        simulator_sweep_configs: List of simulator config dicts
    
    Returns:
        dict of metric_name -> value
    """
    stats = {}
    
    # Parse all metrics to understand what we need
    parsed_metrics = [parse_metric_name(m) for m in metrics_to_compute]
    
    # Determine which subsets we need
    subsets_needed = set(p['subset'] for p in parsed_metrics)
    
    # Determine which metric types we need
    metric_types_needed = set(p['metric_type'] for p in parsed_metrics)
    
    # Determine which configs we need
    configs_needed = set(p['config'] for p in parsed_metrics if p['config'])
    
    # Build config name lookup
    config_name_to_dict = {utils.simulator_config_to_str(cfg): cfg for cfg in simulator_sweep_configs}
    
    # Get number of samples
    n_total_samples = len([col for col in df.columns if col.startswith('simulator_score_')])
    if n_total_samples == 0:
        n_total_samples = 1
    
    # ---- Prepare indices for each subset (only compute what's needed) ----
    subset_indices = {}
    
    # 'all' subset - just use all rows
    if 'all' in subsets_needed:
        subset_indices['all'] = np.arange(len(df))
    
    # cf_stable subset (requires cross-round columns)
    if 'cf_stable' in subsets_needed:
        # Look for round columns
        round_cols = [c for c in df.columns if c.startswith('counterfactual_model_answer_round')]
        if len(round_cols) >= 2:
            # Find the highest round
            round_nums = [int(c.split('round')[1]) for c in round_cols]
            max_round = max(round_nums)
            if max_round > 0:
                current_col = f'counterfactual_model_answer_round{max_round}'
                first_col = 'counterfactual_model_answer_round0'
                if current_col in df.columns and first_col in df.columns:
                    cf_stable_mask = df[current_col] == df[first_col]
                    subset_indices['cf_stable'] = np.argwhere(cf_stable_mask.values).flatten()
                else:
                    subset_indices['cf_stable'] = np.array([], dtype=int)
            else:
                subset_indices['cf_stable'] = np.array([], dtype=int)
        else:
            subset_indices['cf_stable'] = np.array([], dtype=int)
    
    # sim_balanced_fresh subset
    if 'sim_balanced_fresh' in subsets_needed or 'sim_balanced' in subsets_needed:
        greedy = df.reset_index(drop=True)
        greedy = utils.add_balancing_columns_to_df(greedy)
        sim_cols = ["model_answer_switch", "model_correct_on_counterfactual", "counterfactual_model_answer_is_B"]
        sim_wts = [1, 1, 0.5]
        sim_balanced_size = min(len(greedy), max(100, len(greedy) // 10))
        _, fresh_sim_balanced_idx = utils.get_balanced_dataset(
            greedy,
            sample_size=sim_balanced_size,
            balance_cols=sim_cols,
            weights=sim_wts,
            n_attempts=10000,
            return_indices=True,
        )
        subset_indices['sim_balanced_fresh'] = fresh_sim_balanced_idx
        subset_indices['sim_balanced'] = fresh_sim_balanced_idx  # Use same for now
    
    # ---- Precompute bias monitor data (if any bias metrics needed) ----
    need_bias_data = any('bias_monitor' in p['metric_type'] for p in parsed_metrics)
    bias_data_cache = {}  # (subset, config_name) -> (model_influenced, monitor_probs, monitor_preds)
    
    if need_bias_data and 'model_persuaded_by_cue' in df.columns:
        posthoc_mask = df["cue_could_persuade_model_posthoc"].isna().eq(False) & df["cue_could_persuade_model_posthoc"].astype(bool)
        posthoc_idx = np.argwhere(posthoc_mask.values).flatten()
        
        for subset in subsets_needed:
            if subset not in subset_indices:
                continue
            idx = subset_indices[subset]
            if len(idx) == 0:
                continue
            
            # Intersect with posthoc
            if subset == 'all':
                final_idx = posthoc_idx
            else:
                final_idx = idx[np.isin(idx, posthoc_idx)]
            
            if len(final_idx) == 0:
                continue
            
            # Get model_influenced for this subset
            model_influenced = df.iloc[final_idx]["model_persuaded_by_cue"].values.astype(bool)
            
            # Cache for each config
            for config_name in config_name_to_dict:
                monitor_prob_col = f"{config_name}_monitor_prob_0"
                monitor_pred_col = f"{config_name}_monitor_pred_0"
                
                if monitor_prob_col in df.columns and monitor_pred_col in df.columns:
                    monitor_probs = df.iloc[final_idx][monitor_prob_col].values.astype(float)
                    monitor_preds = df.iloc[final_idx][monitor_pred_col].values.astype(bool)
                    bias_data_cache[(subset, config_name)] = (model_influenced, monitor_probs, monitor_preds)
    
    # ---- Compute requested metrics ----
    for metric_name in metrics_to_compute:
        parsed = parse_metric_name(metric_name)
        subset = parsed['subset']
        metric_type = parsed['metric_type']
        config = parsed['config']
        
        # Get config name - match exactly at the end of the config name
        config_name = None
        for cfg_name in config_name_to_dict:
            if config and cfg_name.endswith(config):
                config_name = cfg_name
                break
        
        # ---- Simulation accuracy metrics ----
        if 'sim_greedy_cot_acc' in metric_type:
            if config_name and subset in subset_indices:
                idx = subset_indices[subset]
                if len(idx) > 0:
                    sim_preds_col = f"{config_name}_simulator_pred_0"
                    cf_labels_col = "counterfactual_model_answer_0"
                    if sim_preds_col in df.columns and cf_labels_col in df.columns:
                        sim_preds = df.iloc[idx][sim_preds_col].values
                        cf_labels = df.iloc[idx][cf_labels_col].values
                        acc = np.mean(sim_preds == cf_labels)
                        stats[metric_name] = float(acc)
        
        # ---- Monitor metrics (bias) - use cached data ----
        elif 'bias_monitor' in metric_type:
            cache_key = (subset, config_name)
            if cache_key in bias_data_cache:
                model_influenced, monitor_probs, monitor_preds = bias_data_cache[cache_key]
                
                # Compute the specific metric requested
                if 'auroc' in metric_type:
                    if len(np.unique(model_influenced)) > 1:
                        stats[metric_name] = float(roc_auc_score(model_influenced, monitor_probs))
                elif 'auprc' in metric_type:
                    if len(np.unique(model_influenced)) > 1:
                        stats[metric_name] = float(average_precision_score(model_influenced, monitor_probs))
                elif 'gmean2' in metric_type or 'gmean' in metric_type:
                    # G-mean = sqrt(TPR * TNR)
                    tp = np.sum(monitor_preds & model_influenced)
                    tn = np.sum(~monitor_preds & ~model_influenced)
                    fn = np.sum(~monitor_preds & model_influenced)
                    fp = np.sum(monitor_preds & ~model_influenced)
                    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
                    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
                    gmean = np.sqrt(tpr * tnr) if (tpr >= 0 and tnr >= 0) else -1
                    if 'gmean2' in metric_type:
                        stats[metric_name] = float(gmean ** 2)  # gmean2
                    else:
                        stats[metric_name] = float(gmean)
                elif 'precision' in metric_type:
                    tp = np.sum(monitor_preds & model_influenced)
                    fp = np.sum(monitor_preds & ~model_influenced)
                    precision = tp / (tp + fp) if (tp + fp) > 0 else -1
                    stats[metric_name] = float(precision)
                elif 'recall' in metric_type:
                    tp = np.sum(monitor_preds & model_influenced)
                    fn = np.sum(~monitor_preds & model_influenced)
                    recall = tp / (tp + fn) if (tp + fn) > 0 else -1
                    stats[metric_name] = float(recall)
                elif 'FPR' in metric_type:
                    fp = np.sum(monitor_preds & ~model_influenced)
                    tn = np.sum(~monitor_preds & ~model_influenced)
                    fpr = fp / (fp + tn) if (fp + tn) > 0 else -1
                    stats[metric_name] = float(fpr)
                elif 'acc' in metric_type:
                    stats[metric_name] = float(np.mean(monitor_preds == model_influenced))
                    
        # ---- Baseline metrics ----
        elif 'sim_baseline_majority_class' in metric_type:
            # Simulator baseline: majority class accuracy
            if 'counterfactual_model_answer_0' in df.columns:
                cf_answers = df['counterfactual_model_answer_0'].dropna()
                valid_answers = cf_answers[cf_answers.isin(['A', 'B'])]
                if len(valid_answers) > 0:
                    counts = valid_answers.value_counts()
                    baseline = counts.max() / len(valid_answers)
                    stats[metric_name] = float(baseline)
                    
        elif 'monitor_baseline_majority_class' in metric_type or 'monitor_baseline_never_influenced' in metric_type:
            # Monitor baseline: majority class of model_persuaded_by_cue (predicting "never influenced")
            if 'model_persuaded_by_cue' in df.columns and 'cue_could_persuade_model_posthoc' in df.columns:
                posthoc_mask = df["cue_could_persuade_model_posthoc"].notna() & df["cue_could_persuade_model_posthoc"].astype(bool)
                labels = df.loc[posthoc_mask, 'model_persuaded_by_cue'].dropna().astype(bool)
                if len(labels) > 0:
                    baseline = max(labels.mean(), 1 - labels.mean())
                    stats[metric_name] = float(baseline)
        
        # ---- Greedy bias rate (influence rate) ----
        elif metric_name == 'all_greedy_bias_rate' or 'greedy_bias_rate' in metric_type:
            # Bias rate: mean of model_persuaded_by_cue for eligible samples
            if 'model_persuaded_by_cue' in df.columns and 'cue_could_persuade_model_posthoc' in df.columns:
                posthoc_mask = df["cue_could_persuade_model_posthoc"].notna() & df["cue_could_persuade_model_posthoc"].astype(bool)
                if posthoc_mask.sum() > 0:
                    bias_rate = df.loc[posthoc_mask, 'model_persuaded_by_cue'].astype(bool).mean()
                    stats[metric_name] = float(bias_rate)
    
    return stats


def bootstrap_evaluation_stats(
    args,
    n_seeds: int,
    rounds: list,
    simulator_sweep_configs: list,
    metrics_to_compute: list = None,  # NEW: if provided, only compute these metrics (much faster)
    cf_type_filter: list = None,  # If None, use all cf types; otherwise filter to these
    n_bootstrap: int = 10,
    confidence_level: float = 0.95,
    random_state: int = 42,
    block_col: str = 'id',
    verbose: bool = True,
    cache_path: str = None,  # If provided, cache results to this path
    force_rerun: bool = False,  # If True, ignore cache and recompute
    print_pvalues: bool = True,  # If True, print p-value table for improvement vs round 0
):
    """
    Compute bootstrap confidence intervals for evaluation stats using hierarchical bootstrap.
    
    Hierarchical bootstrap procedure:
    1. Load data organized by seed (each seed has same test set across rounds, different seeds have different test sets)
    2. Pre-generate shared random indices: seed sampling AND per-seed ID sampling (enables paired comparisons)
    3. For each bootstrap iteration:
       a. Sample seeds with replacement
       b. For each sampled seed, resample IDs using that seed's pre-generated indices
       c. Stack all resampled data
       d. Compute stats
    4. Repeat n_bootstrap times to get CI
    
    Note: Pairing is valid because the same ID resampling is used for a given seed across all rounds.
    
    Args:
        args: Experiment args (SimpleNamespace)
        n_seeds: Number of seeds to bootstrap over
        rounds: List of round numbers to compute stats for
        simulator_sweep_configs: List of simulator config dicts for compute_evaluation_stats
        metrics_to_compute: List of specific metrics to compute (if None, uses expensive main.compute_evaluation_stats)
        cf_type_filter: List of counterfactual types to include (None = all)
        n_bootstrap: Number of bootstrap iterations
        confidence_level: CI level (e.g., 0.95 for 95% CI)
        random_state: Random seed for reproducibility
        block_col: Column to use for block resampling
        verbose: Show progress bar
        cache_path: If provided, save/load results from this pickle path
        force_rerun: If True, recompute even if cache exists
        print_pvalues: If True, print hypothesis test results for improvement vs round 0
    
    Returns:
        results_long: pd.DataFrame with columns: round, metric, mean, ci_low, ci_high, n_eff
        results_wide: Wide-format df suitable for plot_metric
        bootstrap_samples: Dict of {round_num: {metric_name: np.array of bootstrap values}}
                          for paired hypothesis testing (only for metrics_to_compute)
    """
    import pickle
    
    # Check cache
    if cache_path and not force_rerun and Path(cache_path).exists():
        if verbose:
            print(f"Loading cached bootstrap results from {cache_path}")
        with open(cache_path, 'rb') as f:
            cached = pickle.load(f)
        results_long = cached['results_long']
        results_wide = cached['results_wide']
        bootstrap_samples = cached.get('bootstrap_samples', {})
        true_means_wide = cached.get('true_means_wide', pd.DataFrame())
        
        # Print p-values if requested and samples available
        if print_pvalues and bootstrap_samples and metrics_to_compute:
            print_pvalue_table(bootstrap_samples, metrics_to_compute, rounds, baseline_round=0)
            print_xye_vs_xy_table(bootstrap_samples, metrics_to_compute, rounds)
        
        return results_long, results_wide, bootstrap_samples, true_means_wide
    
    use_lightweight = metrics_to_compute is not None
    if use_lightweight and verbose:
        print(f"Using lightweight bootstrap for {len(metrics_to_compute)} specific metrics")
    
    rng = np.random.default_rng(random_state)
    alpha = 1 - confidence_level
    
    # ========================================================================
    # PHASE 1: Load all data for all rounds upfront, organized by seed
    # ========================================================================
    # Structure: seed_round_data[seed_idx][round_num] = df
    # Each seed has the SAME test set across rounds (but different seeds have different test sets)
    if verbose:
        print("\n=== Loading data for all rounds ===")
    
    seed_round_data = {seed_idx: {} for seed_idx in range(n_seeds)}  # {seed_idx: {round_num: df}}
    per_seed_ids = {}  # {seed_idx: np.array of IDs common across all rounds for this seed}
    
    for round_num in rounds:
        if verbose:
            print(f"Loading round {round_num}...")
        
        seed_dfs = load_test_datasets_for_round(args, round_num, n_seeds=n_seeds)
        if len(seed_dfs) == 0:
            print(f"No data found for round {round_num}, skipping")
            continue
        
        # Filter to cf types if specified
        if cf_type_filter is not None:
            seed_dfs = [filter_to_cf_types(df, cf_type_filter) for df in seed_dfs]
        
        # Store each seed's df for this round
        for seed_idx, df in enumerate(seed_dfs):
            if df is not None and len(df) > 0:
                seed_round_data[seed_idx][round_num] = df
    
    # Get IDs for each seed (all rounds within a seed share the same test set by construction)
    valid_rounds = set()
    for seed_idx in range(n_seeds):
        # Just grab IDs from any available round for this seed
        for round_num, df in seed_round_data[seed_idx].items():
            per_seed_ids[seed_idx] = np.array(sorted(df[block_col].unique()))
            valid_rounds.add(round_num)
            break  # All rounds have the same IDs, so we only need one
        else:
            per_seed_ids[seed_idx] = np.array([])
    
    # Collect all valid rounds and check which rounds have ALL seeds
    for seed_idx in range(n_seeds):
        valid_rounds.update(seed_round_data[seed_idx].keys())
    
    # Find rounds that have data from ALL seeds (complete rounds)
    complete_rounds = set(valid_rounds)
    for seed_idx in range(n_seeds):
        complete_rounds = complete_rounds.intersection(seed_round_data[seed_idx].keys())
    
    incomplete_rounds = valid_rounds - complete_rounds
    if incomplete_rounds and verbose:
        print(f"\nWARNING: Rounds {sorted(incomplete_rounds)} are missing data from some seeds!")
        for round_num in sorted(incomplete_rounds):
            missing_seeds = [s for s in range(n_seeds) if round_num not in seed_round_data[s]]
            print(f"  Round {round_num}: missing seeds {missing_seeds}")
        print(f"  -> Bootstrap for these rounds will only use available seeds, which may bias results.")
        print(f"  -> Consider filtering to complete rounds only: {sorted(complete_rounds)}")
    
    if len(valid_rounds) == 0:
        print("No data found for any round!")
        return pd.DataFrame(), pd.DataFrame(), {}, pd.DataFrame()
    
    if verbose:
        for seed_idx in range(n_seeds):
            print(f"Seed {seed_idx}: {len(per_seed_ids[seed_idx])} common IDs across {len(seed_round_data[seed_idx])} rounds")
    
    # ========================================================================
    # PHASE 2: Pre-generate shared random indices for ALL bootstrap iterations
    # ========================================================================
    if verbose:
        print(f"\nPre-generating {n_bootstrap} bootstrap index sets...")
    
    # Seed indices: shape (n_bootstrap, n_seeds) - same across rounds
    seed_indices_all = rng.choice(n_seeds, size=(n_bootstrap, n_seeds), replace=True)
    
    # ID indices PER SEED: {seed_idx: (n_bootstrap, n_ids_for_seed)} - same across rounds for pairing
    id_indices_per_seed = {}
    for seed_idx in range(n_seeds):
        n_ids = len(per_seed_ids[seed_idx])
        if n_ids > 0:
            id_indices_per_seed[seed_idx] = rng.choice(n_ids, size=(n_bootstrap, n_ids), replace=True)
        else:
            id_indices_per_seed[seed_idx] = None
    
    # ========================================================================
    # PHASE 3: Compute TRUE MEANS on original (non-resampled) data
    # ========================================================================
    # NOTE: Like the bootstrap, we compute metrics PER SEED and then AVERAGE.
    # This handles nonlinear metrics correctly and matches load_stats_across_seeds.
    # ========================================================================
    if verbose:
        print(f"\nComputing true means on original data...")
    
    true_means_results = []
    for round_num in sorted(valid_rounds):
        # Compute stats per seed, then average
        per_seed_stats_list = []
        for seed_idx in range(n_seeds):
            if round_num not in seed_round_data[seed_idx]:
                continue
            
            seed_df = seed_round_data[seed_idx][round_num]
            
            try:
                if use_lightweight:
                    seed_stats = compute_bootstrap_metrics(
                        df=seed_df,
                        metrics_to_compute=metrics_to_compute,
                        simulator_sweep_configs=simulator_sweep_configs,
                    )
                else:
                    seed_stats = main.compute_evaluation_stats(
                        args=args,
                        df=seed_df,
                        simulator_sweep_configs=simulator_sweep_configs,
                        train_round=round_num,
                        backfilling_cf_stable_stats=False,
                        tokenizer=None,
                    )
                per_seed_stats_list.append(seed_stats)
            except Exception as e:
                if verbose:
                    print(f"Warning: true means computation failed for seed {seed_idx} round {round_num}: {e}")
        
        if len(per_seed_stats_list) == 0:
            continue
        
        # Average metrics across seeds
        all_keys = set()
        for s in per_seed_stats_list:
            all_keys.update(s.keys())
        
        for key in all_keys:
            values = [s[key] for s in per_seed_stats_list if key in s and s[key] is not None]
            if len(values) > 0:
                true_means_results.append({
                    'round': round_num,
                    'metric': key,
                    'true_mean': float(np.mean(values)),
                })
    
    # Convert to wide format for true means
    true_means_df = pd.DataFrame(true_means_results)
    if not true_means_df.empty:
        true_means_wide = true_means_df.pivot(index='round', columns='metric', values='true_mean')
        true_means_wide = true_means_wide.reset_index()
        # Rename to match expected format: metric_test
        rename_map = {col: f"{col}_test" for col in true_means_wide.columns if col != 'round'}
        true_means_wide = true_means_wide.rename(columns=rename_map)
    else:
        true_means_wide = pd.DataFrame()
    
    # ========================================================================
    # PHASE 4: Bootstrap loop with shared indices
    # ========================================================================
    # NOTE: We compute metrics PER SEED and then AVERAGE across seeds.
    # This is important for nonlinear metrics like G-Mean = sqrt(TPR * TNR).
    # If we pooled data across seeds first, we'd get a different (biased) result
    # because E[f(X)] != f(E[X]) for nonlinear f.
    # By averaging per-seed metrics, the bootstrap correctly mimics the estimation
    # procedure: "average of per-seed statistics."
    # ========================================================================
    all_results = []
    bootstrap_samples = {}  # {round_num: {metric_name: np.array}}
    
    for round_num in sorted(valid_rounds):
        if verbose:
            print(f"\n===  Train Round {round_num} ===")
        
        # Compute n_eff for this round
        total_n = sum(len(seed_round_data[s].get(round_num, pd.DataFrame())) for s in range(n_seeds))
        n_eff = total_n / n_seeds
        
        # Bootstrap loop with shared indices
        bootstrap_stats = []
        iterator = tqdm(range(n_bootstrap), desc=f"Round {round_num}") if verbose else range(n_bootstrap)
        
        for b in iterator:
            # Step 1: Sample seeds with replacement (using pre-generated indices)
            sampled_seed_indices = seed_indices_all[b]
            
            # Step 2: For each sampled seed, resample its IDs and compute metrics PER SEED
            per_seed_stats_list = []
            for seed_idx in sampled_seed_indices:
                if round_num not in seed_round_data[seed_idx]:
                    continue
                if id_indices_per_seed[seed_idx] is None:
                    continue
                
                df = seed_round_data[seed_idx][round_num]
                seed_ids = per_seed_ids[seed_idx]
                
                # Get the pre-generated ID indices for this seed and bootstrap iteration
                sampled_ids = seed_ids[id_indices_per_seed[seed_idx][b]]
                
                # Vectorized resampling using merge (inner join to handle rounds with fewer IDs)
                sampled_id_df = pd.DataFrame({block_col: sampled_ids, '_order': np.arange(len(sampled_ids))})
                resampled_df = sampled_id_df.merge(df, on=block_col, how='inner')
                resampled_df = resampled_df.drop(columns=['_order'])
                
                # Compute metrics for THIS SEED's resampled data
                try:
                    if use_lightweight:
                        seed_stats = compute_bootstrap_metrics(
                            df=resampled_df,
                            metrics_to_compute=metrics_to_compute,
                            simulator_sweep_configs=simulator_sweep_configs,
                        )
                    else:
                        seed_stats = main.compute_evaluation_stats(
                            args=args,
                            df=resampled_df,
                            simulator_sweep_configs=simulator_sweep_configs,
                            train_round=round_num,
                            backfilling_cf_stable_stats=False,
                            tokenizer=None,
                        )
                    per_seed_stats_list.append(seed_stats)
                except Exception as e:
                    # Skip this seed if computation fails
                    continue
            
            if len(per_seed_stats_list) == 0:
                continue
            
            # Step 3: Average metrics across seeds (handles nonlinear metrics correctly)
            stats = {}
            all_keys = set()
            for s in per_seed_stats_list:
                all_keys.update(s.keys())
            
            for key in all_keys:
                values = [s[key] for s in per_seed_stats_list if key in s and s[key] is not None]
                if len(values) > 0:
                    stats[key] = float(np.mean(values))
            
            bootstrap_stats.append(stats)
        
        if len(bootstrap_stats) == 0:
            print(f"All bootstrap iterations failed for round {round_num}")
            continue
        
        # Convert to DataFrame for easy percentile computation
        stats_df = pd.DataFrame(bootstrap_stats)
        
        # Store raw bootstrap samples for requested metrics (for paired hypothesis tests)
        if use_lightweight:
            bootstrap_samples[round_num] = {}
            for metric in metrics_to_compute:
                if metric in stats_df.columns:
                    values = stats_df[metric].dropna().values
                    # Store if we have at least 90% of iterations (allow some failures)
                    if len(values) >= n_bootstrap * 0.9:
                        bootstrap_samples[round_num][metric] = values.copy()
                    elif len(values) > 0:
                        print(f"Warning: Only {len(values)}/{n_bootstrap} bootstrap samples for {metric} at round {round_num}")
                        bootstrap_samples[round_num][metric] = values.copy()  # Store anyway
        
        # Compute mean and CIs for each metric
        for metric in stats_df.columns:
            values = stats_df[metric].dropna().values
            if len(values) == 0:
                continue
            
            mean_val = np.mean(values)
            ci_low = np.percentile(values, 100 * alpha / 2)
            ci_high = np.percentile(values, 100 * (1 - alpha / 2))
            
            all_results.append({
                'round': round_num,
                'metric': metric,
                'mean': mean_val,
                'ci_low': ci_low,
                'ci_high': ci_high,
                'n_eff': n_eff,
                'n_bootstrap_successful': len(values),
            })
    
    results_df = pd.DataFrame(all_results)
    
    # Also create wide format for plotting
    wide_df = results_df.pivot(index='round', columns='metric', values=['mean', 'ci_low', 'ci_high'])
    wide_df.columns = [f"{metric}_{stat}" if stat != 'mean' else metric 
                       for stat, metric in wide_df.columns]
    wide_df = wide_df.reset_index()
    
    # Rename columns to match expected format: metric_test, metric_test_ci_low, etc.
    rename_map = {}
    for col in wide_df.columns:
        if col == 'round':
            continue
        if col.endswith('_ci_low'):
            base = col[:-7]  # Remove _ci_low
            rename_map[col] = f"{base}_test_ci_low"
        elif col.endswith('_ci_high'):
            base = col[:-8]  # Remove _ci_high
            rename_map[col] = f"{base}_test_ci_high"
        else:
            rename_map[col] = f"{col}_test"
    
    wide_df = wide_df.rename(columns=rename_map)
    
    # Save to cache if path provided (including bootstrap samples and true means)
    if cache_path:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        with open(cache_path, 'wb') as f:
            pickle.dump({
                'results_long': results_df, 
                'results_wide': wide_df,
                'bootstrap_samples': bootstrap_samples,
                'true_means_wide': true_means_wide,
            }, f)
        if verbose:
            print(f"\nSaved bootstrap results to {cache_path}")
    
    # Print p-value tables if requested
    if print_pvalues and bootstrap_samples and metrics_to_compute:
        print_pvalue_table(bootstrap_samples, metrics_to_compute, list(valid_rounds), baseline_round=0)
        print_xye_vs_xy_table(bootstrap_samples, metrics_to_compute, list(valid_rounds))
    
    return results_df, wide_df, bootstrap_samples, true_means_wide

# Plotting

In [ ]:
def plot_metric(
    df, metrics, split="test", control_condition=None, buffer=0.5,
    figsize=(10, 6), legend_labels=None, title=None, ylabel=None, ylim=None,
    show_values=False, value_decimals=None, scale=100, lines_only=False, show_ci=False, ci_alpha=0.2,
    ci_style='ribbon',  # 'ribbon', 'lines', 'ribbon+lines', or 'errorbar'
    show_true_means=False, true_means_df=None,  # NEW: overlay true means
    save_name=None,
):
    """
    Plots train vs. test curves for one or more metrics against round using seaborn.

    Args:
        df (pd.DataFrame): roundspread dataframe (bootstrap results if show_ci=True)
        metrics (str | list[str]): base metric name(s) (without _train/_test)
        split (str): which curves to plot: "train", "test", or "both"
        control_condition (str): optional control metric to overlay
        buffer (float): margin for y-axis padding
        figsize (tuple): figure size (width, height)
        legend_labels (list): custom labels for metrics, in same order as metrics list
        title (str): custom title for the plot
        ylabel (str): custom label for y-axis
        ylim (tuple): custom y-axis range as (ymin, ymax), e.g. (0.5, 1.0)
        show_values (bool): if True, annotate points with their values
        scale (float): scale factor to apply to all metric values (default 100)
        lines_only (bool): if True, plot lines without markers (default False)
        ci_alpha (float): transparency for CI ribbons (only used if ci_style='ribbon')
        ci_style (str): 'ribbon', 'lines', 'ribbon+lines', or 'errorbar'
        show_true_means (bool): if True, overlay true means from true_means_df
        true_means_df (pd.DataFrame): dataframe with true means (if None, uses df)
    """
    # Custom palette - muted, professional colors
    palette = [
        "#5fa89d",  # muted teal
        "#d9844f",  # muted orange
        "#6b7fa3",  # muted blue
    ]
    
    # Normalize metrics input
    if isinstance(metrics, str):
        metrics = [metrics]
    
    # Validate split arg
    if split not in {"train", "test", "both"}:
        raise ValueError("split must be 'train', 'test', or 'both'")
    
    # Create metric to label mapping if custom labels provided
    metric_label_map = {}
    if legend_labels is not None:
        for i, metric in enumerate(metrics):
            if i < len(legend_labels):
                metric_label_map[metric] = legend_labels[i]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Prepare data for seaborn
    plot_data = []
    
    # --- Case 1: Single metric ---
    if len(metrics) == 1:
        metric = metrics[0]
        train_col, test_col = f"{metric}_train", f"{metric}_test"
        display_metric = metric_label_map.get(metric, metric)

        if split in {"train", "both"} and train_col in df.columns:
            for _, row in df.iterrows():
                entry = {
                    'Round': row['round'],
                    'Metric Value': row[train_col] * scale,
                    'Split': 'Train',
                    'Metric': display_metric
                }
                ci_low_col = f"{metric}_train_ci_low"
                ci_high_col = f"{metric}_train_ci_high"
                if ci_low_col in df.columns and ci_high_col in df.columns:
                    entry['ci_low'] = row[ci_low_col] * scale
                    entry['ci_high'] = row[ci_high_col] * scale
                plot_data.append(entry)
        
        if split in {"test", "both"} and test_col in df.columns:
            for _, row in df.iterrows():
                entry = {
                    'Round': row['round'],
                    'Metric Value': row[test_col] * scale,
                    'Split': 'Test',
                    'Metric': display_metric
                }
                ci_low_col = f"{metric}_test_ci_low"
                ci_high_col = f"{metric}_test_ci_high"
                if ci_low_col in df.columns and ci_high_col in df.columns:
                    entry['ci_low'] = row[ci_low_col] * scale
                    entry['ci_high'] = row[ci_high_col] * scale
                plot_data.append(entry)

    # --- Case 2: Multiple metrics ---
    else:
        for metric in metrics:
            train_col, test_col = f"{metric}_train", f"{metric}_test"
            display_metric = metric_label_map.get(metric, metric)
            
            if split in {"train", "both"} and train_col in df.columns:
                for _, row in df.iterrows():
                    entry = {
                        'Round': row['round'],
                        'Metric Value': row[train_col] * scale,
                        'Split': 'Train',
                        'Metric': display_metric
                    }
                    ci_low_col = f"{metric}_train_ci_low"
                    ci_high_col = f"{metric}_train_ci_high"
                    if show_ci and ci_low_col in df.columns and ci_high_col in df.columns:
                        entry['ci_low'] = row[ci_low_col] * scale
                        entry['ci_high'] = row[ci_high_col] * scale
                    plot_data.append(entry)
            
            if split in {"test", "both"} and test_col in df.columns:
                for _, row in df.iterrows():
                    entry = {
                        'Round': row['round'],
                        'Metric Value': row[test_col] * scale,
                        'Split': 'Test',
                        'Metric': display_metric
                    }
                    ci_low_col = f"{metric}_test_ci_low"
                    ci_high_col = f"{metric}_test_ci_high"
                    if show_ci and ci_low_col in df.columns and ci_high_col in df.columns:
                        entry['ci_low'] = row[ci_low_col] * scale
                        entry['ci_high'] = row[ci_high_col] * scale
                    plot_data.append(entry)
    
    plot_df = pd.DataFrame(plot_data)
    
    # Determine groupby column for CI drawing and get order (matching seaborn's hue order)
    if len(metrics) == 1:
        groupby_col = 'Split'
        # Seaborn uses order of first appearance
        group_order = plot_df['Split'].unique().tolist()
    else:
        groupby_col = 'Metric'
        # Use order from legend_labels if provided, otherwise order of first appearance
        group_order = plot_df['Metric'].unique().tolist()
    
    # --- Draw CIs first (so lines are on top) ---
    if show_ci and 'ci_low' in plot_df.columns:
        for i, group_val in enumerate(group_order):
            group_df = plot_df[plot_df[groupby_col] == group_val].sort_values('Round')
            color = palette[i % len(palette)]
            
            if ci_style == 'ribbon':
                ax.fill_between(
                    group_df['Round'],
                    group_df['ci_low'],
                    group_df['ci_high'],
                    alpha=ci_alpha,
                    color=color,
                    linewidth=0,
                )
            elif ci_style == 'lines':
                # Dashed lines for CI bounds
                ax.plot(
                    group_df['Round'], group_df['ci_low'],
                    linestyle='--', linewidth=1.5, color=color, alpha=0.4
                )
                ax.plot(
                    group_df['Round'], group_df['ci_high'],
                    linestyle='--', linewidth=1.5, color=color, alpha=0.4
                )
            elif ci_style == 'ribbon+lines':
                # Both ribbon shading and dashed line bounds
                ax.fill_between(
                    group_df['Round'],
                    group_df['ci_low'],
                    group_df['ci_high'],
                    alpha=ci_alpha,
                    color=color,
                    linewidth=0,
                )
                ax.plot(
                    group_df['Round'], group_df['ci_low'],
                    linestyle='--', linewidth=1.5, color=color, alpha=0.4
                )
                ax.plot(
                    group_df['Round'], group_df['ci_high'],
                    linestyle='--', linewidth=1.5, color=color, alpha=0.4
                )
            elif ci_style == 'errorbar':
                # Will draw error bars after main plot
                pass
    
    # --- Draw true means if requested (before seaborn plot so it's underneath) ---
    if show_true_means:
        if true_means_df is None:
            raise ValueError("show_true_means=True requires true_means_df to be provided")
        tm_df = true_means_df
        true_means_data = []
        
        if len(metrics) == 1:
            metric = metrics[0]
            display_metric = metric_label_map.get(metric, metric)
            
            if split in {"train", "both"} and f"{metric}_train" in tm_df.columns:
                for _, row in tm_df.iterrows():
                    true_means_data.append({
                        'Round': row['round'],
                        'Metric Value': row[f"{metric}_train"] * scale,
                        'Split': 'Train',
                        'Metric': display_metric
                    })
            
            if split in {"test", "both"} and f"{metric}_test" in tm_df.columns:
                for _, row in tm_df.iterrows():
                    true_means_data.append({
                        'Round': row['round'],
                        'Metric Value': row[f"{metric}_test"] * scale,
                        'Split': 'Test',
                        'Metric': display_metric
                    })
        else:
            for metric in metrics:
                display_metric = metric_label_map.get(metric, metric)
                
                if split in {"train", "both"} and f"{metric}_train" in tm_df.columns:
                    for _, row in tm_df.iterrows():
                        true_means_data.append({
                            'Round': row['round'],
                            'Metric Value': row[f"{metric}_train"] * scale,
                            'Split': 'Train',
                            'Metric': display_metric
                        })
                
                if split in {"test", "both"} and f"{metric}_test" in tm_df.columns:
                    for _, row in tm_df.iterrows():
                        true_means_data.append({
                            'Round': row['round'],
                            'Metric Value': row[f"{metric}_test"] * scale,
                            'Split': 'Test',
                            'Metric': display_metric
                        })
        
        true_means_plot_df = pd.DataFrame(true_means_data)
        if not true_means_plot_df.empty:
            marker = None if lines_only else 'o'
            for i, group_val in enumerate(group_order):
                group_df = true_means_plot_df[true_means_plot_df[groupby_col] == group_val].sort_values('Round')
                color = palette[i % len(palette)]
                ax.plot(
                    group_df['Round'], group_df['Metric Value'],
                    linestyle='-', linewidth=3.0, color=color, alpha=0.9, zorder=10,
                    marker=marker, markersize=7,
                )
    
    # Main plot with seaborn (skip if showing true means - we already plotted the lines)
    if not plot_df.empty and not show_true_means:
        marker = None if lines_only else 'o'
        if len(metrics) == 1:
            sns.lineplot(
                data=plot_df, x='Round', y='Metric Value', hue='Split',
                marker=marker, linewidth=3.0, markersize=7,
                palette=palette[:2], ax=ax
            )
        else:
            if split == "both":
                sns.lineplot(
                    data=plot_df, x='Round', y='Metric Value', 
                    hue='Metric', style='Split',
                    marker=marker, linewidth=3.0, markersize=7, palette=palette, ax=ax
                )
            else:
                sns.lineplot(
                    data=plot_df, x='Round', y='Metric Value', 
                    hue='Metric',
                    marker=marker, linewidth=3.0, markersize=7, palette=palette, ax=ax
                )
    elif not plot_df.empty and show_true_means:
        # Still need seaborn for the legend, but plot invisible lines
        marker = None if lines_only else 'o'
        if len(metrics) == 1:
            sns.lineplot(
                data=plot_df, x='Round', y='Metric Value', hue='Split',
                marker=marker, linewidth=0, markersize=0,
                palette=palette[:2], ax=ax, legend=True
            )
        else:
            if split == "both":
                sns.lineplot(
                    data=plot_df, x='Round', y='Metric Value', 
                    hue='Metric', style='Split',
                    marker=marker, linewidth=0, markersize=0, palette=palette, ax=ax, legend=True
                )
            else:
                sns.lineplot(
                    data=plot_df, x='Round', y='Metric Value', 
                    hue='Metric',
                    marker=marker, linewidth=0, markersize=0, palette=palette, ax=ax, legend=True
                )
    
    # Draw error bars after main plot (so they're on top)
    if show_ci and 'ci_low' in plot_df.columns and ci_style == 'errorbar':
        for i, group_val in enumerate(group_order):
            group_df = plot_df[plot_df[groupby_col] == group_val].sort_values('Round')
            color = palette[i % len(palette)]
            yerr_low = group_df['Metric Value'] - group_df['ci_low']
            yerr_high = group_df['ci_high'] - group_df['Metric Value']
            ax.errorbar(
                group_df['Round'], group_df['Metric Value'],
                yerr=[yerr_low, yerr_high],
                fmt='none', ecolor=color, elinewidth=1.5, capsize=3, alpha=0.7
            )
    
    # --- Optional control condition ---
    if control_condition is not None:
        assert split in ['train', 'test']
        control_col = f"{control_condition}_{split}"
        if control_col in df.columns:
            ax.plot(
                df["round"], df[control_col] * scale,
                label=f"Control ({control_condition})",
                linestyle="--", linewidth=2, color="gray", alpha=0.6, marker='D'
            )
    
    # Styling
    ax.spines['top'].set_color('#ffffff')
    ax.spines['right'].set_color('#ffffff')
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.set_facecolor(PLOT_BACKGROUND_COLOR)
    
    ax.set_xlabel("Round", fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel if ylabel else "Metric Value", fontsize=13, fontweight='bold')
    
    round_vals = df['round'].unique()
    ax.set_xticks(sorted(round_vals))
    ax.set_xticklabels([int(r) for r in sorted(round_vals)])
    ax.tick_params(axis='x', pad=-1)
    
    if title:
        plot_title = title
    else:
        plot_title = f"Metrics vs. Round ({split.capitalize()})" if len(metrics) > 1 else f"{metrics[0]} vs. Round"
    ax.set_title(plot_title, fontsize=15, fontweight='bold', pad=10)
    
    if ylim is not None:
        ax.set_ylim(ylim[0] * scale, ylim[1] * scale)
    else:
        all_vals = []
        for metric in metrics:
            if split in {"train", "both"} and f"{metric}_train" in df.columns:
                all_vals.extend((df[f"{metric}_train"] * scale).dropna().tolist())
            if split in {"test", "both"} and f"{metric}_test" in df.columns:
                all_vals.extend((df[f"{metric}_test"] * scale).dropna().tolist())
        if all_vals:
            min_y, max_y = min(all_vals), max(all_vals)
            y_margin = (max_y - min_y) * buffer
            y_margin = max(y_margin, 0.03 * scale)
            ax.set_ylim(max(0, min_y - y_margin), max_y + y_margin)
    
    # Annotate points with values if requested (use true_means_df values if showing true means)
    if show_values:
        annotation_df = true_means_plot_df if show_true_means else plot_df
        if not annotation_df.empty:
            for idx, row in annotation_df.iterrows():
                if show_ci and ci_style == 'errorbar':
                    x_offset = -4
                    y_offset = 10
                    ha = 'right'
                    va = 'top'
                else:
                    x_offset = 0
                    y_offset = 6
                    ha = 'center'
                    va = 'bottom'
                
                val = row['Metric Value']
                val_str = f"{val:.{value_decimals}f}" if value_decimals is not None else f"{int(val)}"
                ax.annotate(val_str, 
                           xy=(row['Round'], row['Metric Value']),
                           xytext=(x_offset, y_offset), textcoords='offset points',
                           ha=ha, va=va, fontsize=10, alpha=0.8)
    
    ax.legend(frameon=False, fontsize=13, 
             loc='upper center', bbox_to_anchor=(0.5, -0.12), 
             ncol=len(ax.get_legend_handles_labels()[0]))
    
    if save_name is not None:
        plt.savefig(f"{save_name}.pdf", bbox_inches='tight')
    
    plt.tight_layout()
    if save_name:
        plt.savefig(f"zfigs/{save_name}.pdf", bbox_inches='tight', dpi=300)    
    plt.show()

In [ ]:
def plot_metric_by_seed_facets(
    dataframes, 
    metric_groups, 
    split="test", 
    buffer=0.5,
    figsize=(14, 5), 
    nrows=1,  # Number of rows in the grid
    legend_labels_list=None, 
    titles=None, 
    ylabels=None, 
    ylims=None,
    suptitle=None, 
    show_values=False, 
    value_decimals=None,  # None for int, 1 for .1f, etc.
    scale=100, 
    lines_only=False,
    average_xy=True,  # If True, average the second metric (XY baseline) across seeds
    show_mean_line=False,  # If True, also show mean across seeds as dashed line
    save_name=None,
):
    """
    Creates faceted plots with each seed as its own line (gradient colors).
    
    This is the faceted version of plot_metric_by_seed.
    
    Args:
        dataframes (pd.DataFrame or list of pd.DataFrame): single df or list of dfs with 'seed' column
        metric_groups (list of lists): each sublist contains metrics for one subplot
        split (str): which curves to plot: "train", "test", or "both"
        buffer (float): margin for y-axis padding
        figsize (tuple): overall figure size (width, height)
        nrows (int): number of rows in the subplot grid (default 1 = single row)
        legend_labels_list (list of lists): custom labels for each subplot's metrics
        titles (list): titles for each subplot
        ylabels (list): y-axis labels for each subplot
        ylims (list): y-axis ranges for each subplot as tuples (ymin, ymax)
        suptitle (str): overall figure title
        show_values (bool): if True, annotate mean values
        scale (float): scale factor to apply to all metric values (default 100)
        lines_only (bool): if True, plot lines without markers (default False)
        average_xy (bool): if True, average the second metric across seeds
        show_mean_line (bool): if True, show mean across seeds as dashed line for first metric
        save_name (str): if provided, save figure to this path
    """
    import matplotlib.colors as mcolors
    
    # Base colors for each metric (will create gradients from these)
    base_colors = [
        "#5fa89d",  # muted teal (first metric)
        "#d9844f",  # muted orange (second metric / XY)
        "#6b7fa3",  # muted blue
    ]
    
    # Validate split arg
    if split not in {"train", "test", "both"}:
        raise ValueError("split must be 'train', 'test', or 'both'")
    
    # Normalize dataframes input - if single df provided, use for all plots
    if isinstance(dataframes, pd.DataFrame):
        dataframes = [dataframes] * len(metric_groups)
    
    n_plots = len(metric_groups)
    ncols = int(np.ceil(n_plots / nrows))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    axes = axes.flatten()  # Flatten to 1D array for easy iteration
    
    # Hide any extra subplot slots
    for i in range(n_plots, len(axes)):
        axes[i].set_visible(False)
    
    def get_color_gradient(base_color, n_shades):
        """Generate n shades from darker to lighter of a base color."""
        rgb = mcolors.to_rgb(base_color)
        hsv = mcolors.rgb_to_hsv(rgb)
        
        shades = []
        for i in range(n_shades):
            factor = 0.6 + 0.5 * (i / max(1, n_shades - 1))
            new_hsv = (hsv[0], hsv[1] * (1.1 - 0.3 * (i / max(1, n_shades - 1))), min(1.0, hsv[2] * factor))
            shades.append(mcolors.hsv_to_rgb(new_hsv))
        return shades
    
    # Track legend handles/labels from first subplot for shared legend
    shared_legend_handles = []
    shared_legend_labels = []
    
    for idx, (ax, df, metrics) in enumerate(zip(axes, dataframes, metric_groups)):
        # Normalize metrics input
        if isinstance(metrics, str):
            metrics = [metrics]
        
        # Get custom labels for this subplot
        metric_label_map = {}
        if legend_labels_list is not None and idx < len(legend_labels_list):
            legend_labels = legend_labels_list[idx]
            for i, metric in enumerate(metrics):
                if i < len(legend_labels):
                    metric_label_map[metric] = legend_labels[i]
        
        seeds = sorted(df['seed'].unique())
        n_seeds = len(seeds)
        rounds = sorted(df['round'].unique())
        
        marker = None if lines_only else 'o'
        legend_handles = []
        legend_labels_final = []
        
        for metric_idx, metric in enumerate(metrics):
            base_color = base_colors[metric_idx % len(base_colors)]
            display_label = metric_label_map.get(metric, metric)
            
            # Check if metric exists directly or needs suffix
            if metric in df.columns:
                col_name = metric
            elif f"{metric}_test" in df.columns:
                col_name = f"{metric}_test" if split in ["test", "both"] else f"{metric}_train"
            else:
                print(f"Warning: metric {metric} not found in dataframe for subplot {idx}")
                continue
            
            # Check if we should average this metric (XY baseline)
            should_average = average_xy and metric_idx > 0
            
            if should_average:
                # Average across seeds for this metric
                avg_df = df.groupby('round')[col_name].mean().reset_index()
                avg_df = avg_df.sort_values('round')
                
                line, = ax.plot(
                    avg_df['round'], 
                    avg_df[col_name] * scale,
                    linestyle='-', 
                    linewidth=2.5,
                    color=base_color,
                    marker=marker,
                    markersize=7,
                    alpha=0.9,
                    zorder=10,
                )
                
                legend_handles.append(line)
                legend_labels_final.append(f"{display_label} (avg)")
                
            else:
                # Plot each seed as its own line with gradient colors
                color_gradient = get_color_gradient(base_color, n_seeds)
                
                for seed_idx, seed in enumerate(seeds):
                    seed_df = df[df['seed'] == seed].sort_values('round')
                    color = color_gradient[seed_idx]
                    
                    line, = ax.plot(
                        seed_df['round'], 
                        seed_df[col_name] * scale,
                        linestyle='-', 
                        linewidth=2.0,
                        color=color,
                        marker=marker,
                        markersize=6,
                        alpha=0.85,
                        zorder=5 + seed_idx,
                    )
                    
                    # Only add to legend for first seed
                    if seed_idx == 0:
                        legend_handles.append(line)
                        legend_labels_final.append(f"{display_label}")
                
                # Optionally show mean line
                if show_mean_line:
                    avg_df = df.groupby('round')[col_name].mean().reset_index()
                    avg_df = avg_df.sort_values('round')
                    
                    mean_line, = ax.plot(
                        avg_df['round'], 
                        avg_df[col_name] * scale,
                        linestyle='--', 
                        linewidth=3.0,
                        color=base_color,
                        marker=None,
                        alpha=0.7,
                        zorder=15,
                    )
                    legend_handles.append(mean_line)
                    legend_labels_final.append(f"{display_label} (mean)")
        
        # Store legend info from first subplot
        if idx == 0:
            shared_legend_handles = legend_handles
            shared_legend_labels = legend_labels_final
        
        # Styling
        ax.spines['top'].set_color('#ffffff')
        ax.spines['right'].set_color('#ffffff')
        ax.spines['left'].set_color('#333333')
        ax.spines['bottom'].set_color('#333333')
        ax.spines['top'].set_linewidth(1.5)
        ax.spines['right'].set_linewidth(1.5)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.set_facecolor(PLOT_BACKGROUND_COLOR)
        
        ax.set_xlabel("Round", fontsize=13, fontweight='bold')
        
        # Y-axis label
        if ylabels is not None and idx < len(ylabels):
            ax.set_ylabel(ylabels[idx], fontsize=13, fontweight='bold')
        else:
            ax.set_ylabel("Metric Value", fontsize=13, fontweight='bold')
        
        ax.set_xticks(rounds)
        ax.set_xticklabels([int(r) for r in rounds])
        ax.tick_params(axis='x', pad=-1)
        
        # Subplot title
        if titles is not None and idx < len(titles):
            ax.set_title(titles[idx], fontsize=13, fontweight='bold', pad=5)
        
        # Y-axis range
        if ylims is not None and idx < len(ylims) and ylims[idx] is not None:
            ax.set_ylim(ylims[idx][0] * scale, ylims[idx][1] * scale)
        else:
            all_vals = []
            for metric in metrics:
                col = metric if metric in df.columns else f"{metric}_test"
                if col in df.columns:
                    all_vals.extend((df[col] * scale).dropna().tolist())
            if all_vals:
                min_y, max_y = min(all_vals), max(all_vals)
                y_margin = (max_y - min_y) * buffer
                y_margin = max(y_margin, 0.03 * scale)
                ax.set_ylim(max(0, min_y - y_margin), max_y + y_margin)
        
        # Annotate mean values if requested
        if show_values:
            for metric in metrics:
                col = metric if metric in df.columns else f"{metric}_test"
                if col not in df.columns:
                    continue
                
                avg_df = df.groupby('round')[col].mean().reset_index()
                for _, row in avg_df.iterrows():
                    val = row[col] * scale
                    val_str = f"{val:.{value_decimals}f}" if value_decimals is not None else f"{int(val)}"
                    ax.annotate(
                        val_str, 
                        xy=(row['round'], row[col] * scale),
                        xytext=(0, 8), 
                        textcoords='offset points',
                        ha='center', 
                        va='bottom', 
                        fontsize=9, 
                        alpha=0.8
                    )
    
    # Create a single shared legend
    fig.legend(
        shared_legend_handles, 
        shared_legend_labels,
        frameon=False, 
        fontsize=12,
        loc='upper center', 
        bbox_to_anchor=(0.51, 0.05),
        ncol=len(shared_legend_handles)
    )
    
    # Overall title
    if suptitle:
        fig.suptitle(suptitle, fontsize=18, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    if save_name:
        plt.savefig(f"zfigs/{save_name}.pdf", dpi=300, bbox_inches="tight", 
                    facecolor='white', edgecolor='none')
    plt.show()

In [ ]:
# ---- Per-Seed Stats Loading and Plotting ----

def load_stats_per_seed(base_exp_name, n_seeds=1):
    """
    Load stats across seeds WITHOUT averaging - keeps each seed as a separate row.
    
    Returns a DataFrame with columns: seed, round, counterfactual_type, + all metric columns
    """
    all_results = []
    for seed in range(n_seeds):
        seed_exp_name = f"{base_exp_name}_sd{seed}"
        stats_df = pd.read_csv(f"results/stats_{seed_exp_name}.csv")
        stats_df['seed'] = seed  # Add seed column
        all_results.append(stats_df)
        print(f"Loaded stats for seed {seed} from {seed_exp_name}")
    
    # Concatenate without averaging
    combined_df = pd.concat(all_results, ignore_index=True)
    return combined_df


def plot_metric_by_seed(
    per_seed_df, 
    metrics, 
    split="test", 
    buffer=0.5,
    figsize=(10, 6), 
    legend_labels=None, 
    title=None, 
    ylabel=None, 
    ylim=None,
    show_values=False, 
    value_decimals=None,  # None for int, 1 for .1f, etc.
    scale=100, 
    lines_only=False,
    average_xy=True,  # If True, average the second metric (XY baseline) across seeds
    show_mean_line=False,  # If True, also show mean across seeds as dashed line
    center_at_round0=False,  # If True, subtract each seed's round 0 value to align starting points
    save_name=None,
):
    """
    Plots metrics with each seed as its own line (gradient colors, same marker).
    
    For comparing XYE vs XY metrics:
    - First metric: shown as individual seed lines with gradient colors
    - Second metric (XY baseline): optionally averaged across seeds (average_xy=True)
    
    Args:
        per_seed_df (pd.DataFrame): DataFrame from load_stats_per_seed with 'seed' column
        metrics (str | list[str]): base metric name(s) (these are direct column names in per_seed_df)
        split (str): which curves to plot: "train", "test", or "both" (ignored if metric names don't have suffix)
        buffer (float): margin for y-axis padding
        figsize (tuple): figure size (width, height)
        legend_labels (list): custom labels for metrics, in same order as metrics list
        title (str): custom title for the plot
        ylabel (str): custom label for y-axis
        ylim (tuple): custom y-axis range as (ymin, ymax)
        show_values (bool): if True, annotate points with their values
        scale (float): scale factor to apply to all metric values (default 100)
        lines_only (bool): if True, plot lines without markers (default False)
        average_xy (bool): if True, average the second metric across seeds
        show_mean_line (bool): if True, show mean across seeds as dashed line for first metric
        center_at_round0 (bool): if True, subtract each seed's round 0 value so all lines start at 0
        save_name (str): if provided, save figure to this path
    """
    import matplotlib.colors as mcolors
    
    # Base colors for each metric (will create gradients from these)
    base_colors = [
        "#5fa89d",  # muted teal (first metric)
        "#d9844f",  # muted orange (second metric / XY)
        "#6b7fa3",  # muted blue
    ]
    
    # Normalize metrics input
    if isinstance(metrics, str):
        metrics = [metrics]
    
    # Validate split arg
    if split not in {"train", "test", "both"}:
        raise ValueError("split must be 'train', 'test', or 'both'")
    
    # Create metric to label mapping
    metric_label_map = {}
    if legend_labels is not None:
        for i, metric in enumerate(metrics):
            if i < len(legend_labels):
                metric_label_map[metric] = legend_labels[i]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    seeds = sorted(per_seed_df['seed'].unique())
    n_seeds = len(seeds)
    rounds = sorted(per_seed_df['round'].unique())
    
    def get_color_gradient(base_color, n_shades):
        """Generate n shades from darker to lighter of a base color."""
        rgb = mcolors.to_rgb(base_color)
        hsv = mcolors.rgb_to_hsv(rgb)
        
        # Create gradient by varying saturation and value
        shades = []
        for i in range(n_shades):
            # Go from darker/more saturated to lighter/less saturated
            factor = 0.6 + 0.5 * (i / max(1, n_shades - 1))  # 0.6 to 1.1
            new_hsv = (hsv[0], hsv[1] * (1.1 - 0.3 * (i / max(1, n_shades - 1))), min(1.0, hsv[2] * factor))
            shades.append(mcolors.hsv_to_rgb(new_hsv))
        return shades
    
    marker = None if lines_only else 'o'
    legend_handles = []
    legend_labels_final = []
    
    # For centering: compute a shared reference from the specified metric's round 0 MEAN
    # All lines (both metrics, all seeds) are shifted by this same value
    # This preserves all relative differences while anchoring at a common reference
    center_reference_mean = 0
    if center_at_round0:
        # Use second metric (XY baseline) as the anchor - it will start at 0
        center_metric = metrics[1] if len(metrics) > 1 else metrics[0]
        if center_metric in per_seed_df.columns:
            center_col = center_metric
        elif f"{center_metric}_test" in per_seed_df.columns:
            center_col = f"{center_metric}_test" if split in ["test", "both"] else f"{center_metric}_train"
        else:
            center_col = center_metric
        
        if center_col in per_seed_df.columns:
            center_reference_mean = per_seed_df[per_seed_df['round'] == 0][center_col].mean()
    
    for metric_idx, metric in enumerate(metrics):
        base_color = base_colors[metric_idx % len(base_colors)]
        display_label = metric_label_map.get(metric, metric)
        
        # Check if metric exists directly or needs suffix
        if metric in per_seed_df.columns:
            col_name = metric
        elif f"{metric}_test" in per_seed_df.columns:
            col_name = f"{metric}_test" if split in ["test", "both"] else f"{metric}_train"
        else:
            print(f"Warning: metric {metric} not found in dataframe")
            continue
        
        # Check if we should average this metric (XY baseline)
        should_average = average_xy and metric_idx > 0  # Average metrics after the first
        
        if should_average:
            # Average across seeds for this metric
            avg_df = per_seed_df.groupby('round')[col_name].mean().reset_index()
            avg_df = avg_df.sort_values('round')
            
            y_vals = avg_df[col_name].values
            if center_at_round0:
                y_vals = y_vals - center_reference_mean  # Center using FIRST metric's round 0 mean
            
            line, = ax.plot(
                avg_df['round'], 
                y_vals * scale,
                linestyle='-', 
                linewidth=2.5,
                color=base_color,
                marker=marker,
                markersize=7,
                alpha=0.9,
                zorder=10,
            )
            
            legend_handles.append(line)
            legend_labels_final.append(f"{display_label} (avg)")
            
        else:
            # Plot each seed as its own line with gradient colors
            color_gradient = get_color_gradient(base_color, n_seeds)
            
            # For centering: also compute this metric's round 0 mean to align seeds
            if center_at_round0:
                this_metric_round0_mean = per_seed_df[per_seed_df['round'] == 0][col_name].mean()
                this_metric_round0_by_seed = per_seed_df[per_seed_df['round'] == 0].set_index('seed')[col_name].to_dict()
            
            for seed_idx, seed in enumerate(seeds):
                seed_df = per_seed_df[per_seed_df['seed'] == seed].sort_values('round')
                color = color_gradient[seed_idx]
                
                y_vals = seed_df[col_name].values.copy()
                if center_at_round0:
                    # First align seeds to their metric's mean, then shift by reference
                    seed_round0 = this_metric_round0_by_seed.get(seed, this_metric_round0_mean)
                    y_vals = y_vals - seed_round0 + this_metric_round0_mean  # Align to metric mean
                    y_vals = y_vals - center_reference_mean  # Shift relative to XY baseline
                
                line, = ax.plot(
                    seed_df['round'], 
                    y_vals * scale,
                    linestyle='-', 
                    linewidth=2.0,
                    color=color,
                    marker=marker,
                    markersize=6,
                    alpha=0.85,
                    zorder=5 + seed_idx,
                )
                
                # Only add to legend for first seed (we'll label the group)
                if seed_idx == 0:
                    legend_handles.append(line)
                    legend_labels_final.append(f"{display_label}")
            
            # Optionally show mean line
            if show_mean_line:
                avg_df = per_seed_df.groupby('round')[col_name].mean().reset_index()
                avg_df = avg_df.sort_values('round')
                
                y_vals_mean = avg_df[col_name].values
                if center_at_round0:
                    y_vals_mean = y_vals_mean - center_reference_mean
                
                mean_line, = ax.plot(
                    avg_df['round'], 
                    y_vals_mean * scale,
                    linestyle='--', 
                    linewidth=3.0,
                    color=base_color,
                    marker=None,
                    alpha=0.7,
                    zorder=15,
                )
                legend_handles.append(mean_line)
                legend_labels_final.append(f"{display_label} (mean)")
    
    # Styling
    ax.spines['top'].set_color('#ffffff')
    ax.spines['right'].set_color('#ffffff')
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.set_facecolor(PLOT_BACKGROUND_COLOR)
    
    ax.set_xlabel("Round", fontsize=13, fontweight='bold')
    # Modify ylabel to indicate centering if applied
    if ylabel:
        final_ylabel = f"Δ {ylabel}" if center_at_round0 else ylabel
    else:
        final_ylabel = "Δ Metric (from Round 0)" if center_at_round0 else "Metric Value"
    ax.set_ylabel(final_ylabel, fontsize=13, fontweight='bold')
    
    ax.set_xticks(rounds)
    ax.set_xticklabels([int(r) for r in rounds])
    ax.tick_params(axis='x', pad=-1)
    
    if title:
        plot_title = title
    else:
        plot_title = f"Metrics by Seed ({split.capitalize()})"
    ax.set_title(plot_title, fontsize=15, fontweight='bold', pad=10)
    
    if ylim is not None:
        ax.set_ylim(ylim[0] * scale, ylim[1] * scale)
    else:
        # Auto-scale based on plotted data (need to recompute if centered)
        # For simplicity, use a reasonable range for centered data
        if center_at_round0:
            # Symmetric range around 0 based on max deviation
            ax.autoscale(enable=True, axis='y')
        else:
            all_vals = []
            for metric in metrics:
                col = metric if metric in per_seed_df.columns else f"{metric}_test"
                if col in per_seed_df.columns:
                    all_vals.extend((per_seed_df[col] * scale).dropna().tolist())
            if all_vals:
                min_y, max_y = min(all_vals), max(all_vals)
                y_margin = (max_y - min_y) * buffer
                y_margin = max(y_margin, 0.03 * scale)
                ax.set_ylim(max(0, min_y - y_margin), max_y + y_margin)
    
    # Add horizontal line at 0 if centered
    if center_at_round0:
        ax.axhline(y=0, color='#888888', linestyle='--', linewidth=1, alpha=0.5, zorder=0)
    
    # Annotate points with values if requested
    if show_values:
        for metric in metrics:
            col = metric if metric in per_seed_df.columns else f"{metric}_test"
            if col not in per_seed_df.columns:
                continue
            
            # Only annotate mean values to avoid clutter
            avg_df = per_seed_df.groupby('round')[col].mean().reset_index()
            for _, row in avg_df.iterrows():
                val = row[col] * scale
                val_str = f"{val:.{value_decimals}f}" if value_decimals is not None else f"{int(val)}"
                ax.annotate(
                    val_str, 
                    xy=(row['round'], row[col] * scale),
                    xytext=(0, 8), 
                    textcoords='offset points',
                    ha='center', 
                    va='bottom', 
                    fontsize=9, 
                    alpha=0.7
                )
    
    ax.legend(
        legend_handles, 
        legend_labels_final,
        frameon=False, 
        fontsize=12, 
        loc='upper center', 
        bbox_to_anchor=(0.5, -0.12), 
        ncol=len(legend_handles)
    )
    
    plt.tight_layout()
    if save_name:
        plt.savefig(f"zfigs/{save_name}.pdf", bbox_inches='tight', dpi=300)
    plt.show()

## main facet plot

In [ ]:
from matplotlib.font_manager import FontProperties

def plot_metric_facets(
    dataframes, metric_groups, split="test", control_condition=None, buffer=0.5,
    figsize=(14, 5), legend_labels_list=None, titles=None, xlabels=None, ylabels=None, ylims=None,
    suptitle=None, show_values=False, value_decimals=None, scale=100, lines_only=False,
    show_ci=False, ci_alpha=0.2, ci_style='ribbon',  # CI parameters
    show_true_means=False, true_means_dfs=None,  # NEW: overlay true means
    show_baseline=False, baseline_metrics=None,  # Baseline metrics to plot as dashed lines
    legend_ncol=None, legend_loc='bottom',  # Legend: ncol=None means auto, loc='bottom' or 'right'
    ncols=None,  # Number of columns for subplot grid (None = all in one row)
    save_name=None,
    # Delta bracket parameters
    show_delta_brackets=False,  # Show bracket annotations between metrics at specific rounds
    delta_bracket_rounds=None,  # List of rounds to show brackets at (e.g., [0, 4])
    bootstrap_samples_list=None,  # List of bootstrap_samples dicts (one per facet) for p-values
    delta_bracket_color='#c97b84',  # Rose-pink color for brackets
    delta_text_positions=None,  # List of positions per facet: 'inbetween', 'topright', 'bottomright'
    delta_text_to_left=True,
    shared_xlabel=None,  # Single centered x-label for whole figure (replaces per-subplot xlabels)
    row_labels=None,  # List of labels for each row (e.g., ["Cue-based Counterfactuals", "Model-based Counterfactuals"])
):
    """
    Creates side-by-side faceted plots for multiple metric groups.

    Args:
        dataframes (pd.DataFrame or list of pd.DataFrame): single dataframe or list of dataframes (one per subplot)
        metric_groups (list of lists): each sublist contains metrics for one subplot
        split (str): which curves to plot: "train", "test", or "both"
        control_condition (str): optional control metric to overlay
        buffer (float): margin for y-axis padding
        figsize (tuple): overall figure size (width, height)
        legend_labels_list (list of lists): custom labels for each subplot's metrics
        titles (list): titles for each subplot
        xlabels (list): x-axis labels for each subplot
        ylabels (list): y-axis labels for each subplot
        ylims (list): y-axis ranges for each subplot as tuples (ymin, ymax)
        suptitle (str): overall figure title
        show_values (bool): if True, annotate points with their values
        scale (float): scale factor to apply to all metric values (default 100)
        lines_only (bool): if True, plot lines without markers (default False)
        show_ci (bool): if True, show confidence intervals (requires _ci_low/_ci_high columns)
        ci_alpha (float): transparency for CI ribbons (only used if ci_style='ribbon')
        ci_style (str): 'ribbon', 'lines', 'ribbon+lines', or 'errorbar'
        show_true_means (bool): if True, overlay true means from true_means_dfs
        true_means_dfs (list of pd.DataFrame): dataframes with true means (if None, uses dataframes)
        show_random_chance (bool): if True, draw horizontal lines for random chance baselines
        random_chance_values (list): list of values (one per facet) for random chance lines (in 0-1 scale)
        show_delta_brackets (bool): if True, draw bracket annotations showing gap between metrics
        delta_bracket_rounds (list): rounds to show brackets at (default: first and last round)
        bootstrap_samples_list (list): list of bootstrap_samples dicts (one per facet) for p-values
        delta_bracket_color (str): color for bracket annotations (default: rose-pink '#c97b84')
    """
    # Custom palette - muted, professional colors
    palette = [
        "#5fa89d",  # muted teal
        "#d9844f",  # muted orange
        "#6b7fa3",  # muted blue
    ]
    
    # Validate split arg
    if split not in {"train", "test", "both"}:
        raise ValueError("split must be 'train', 'test', or 'both'")
    
    # Normalize dataframes input - if single df provided, use for all plots
    if isinstance(dataframes, pd.DataFrame):
        dataframes = [dataframes] * len(metric_groups)
    
    # Normalize true_means_dfs
    if show_true_means:
        if true_means_dfs is None:
            raise ValueError("show_true_means=True requires true_means_dfs to be provided")
        elif isinstance(true_means_dfs, pd.DataFrame):
            true_means_dfs = [true_means_dfs] * len(metric_groups)
    
    n_plots = len(metric_groups)
    
    # Determine grid layout
    if ncols is None:
        ncols_grid = n_plots  # All in one row
    else:
        ncols_grid = ncols
    nrows_grid = int(np.ceil(n_plots / ncols_grid))
    
    fig, axes = plt.subplots(nrows_grid, ncols_grid, figsize=figsize, squeeze=False)
    axes = axes.flatten()[:n_plots]  # Flatten and take only needed axes
    
    for idx, (ax, df, metrics) in enumerate(zip(axes, dataframes, metric_groups)):
        tm_df = true_means_dfs[idx] if show_true_means else None
        
        # Normalize metrics input
        if isinstance(metrics, str):
            metrics = [metrics]
        
        # Get custom labels for this subplot
        metric_label_map = {}
        if legend_labels_list is not None and idx < len(legend_labels_list):
            legend_labels = legend_labels_list[idx]
            for i, metric in enumerate(metrics):
                if i < len(legend_labels):
                    metric_label_map[metric] = legend_labels[i]
        plot_data = []
        # Prepare data for seaborn
        plot_data = []
        
        # --- Case 1: Single metric ---
        if len(metrics) == 1:
            metric = metrics[0]
            train_col, test_col = f"{metric}_train", f"{metric}_test"
            display_metric = metric_label_map.get(metric, metric)

            if split in {"train", "both"} and train_col in df.columns:
                for _, row in df.iterrows():
                    entry = {
                        'Round': row['round'],
                        'Metric Value': row[train_col] * scale,
                        'Split': 'Train',
                        'Metric': display_metric
                    }
                    ci_low_col = f"{metric}_train_ci_low"
                    ci_high_col = f"{metric}_train_ci_high"
                    if show_ci and ci_low_col in df.columns and ci_high_col in df.columns:
                        entry['ci_low'] = row[ci_low_col] * scale
                        entry['ci_high'] = row[ci_high_col] * scale
                    plot_data.append(entry)
            
            if split in {"test", "both"} and test_col in df.columns:
                for _, row in df.iterrows():
                    entry = {
                        'Round': row['round'],
                        'Metric Value': row[test_col] * scale,
                        'Split': 'Test',
                        'Metric': display_metric
                    }
                    ci_low_col = f"{metric}_test_ci_low"
                    ci_high_col = f"{metric}_test_ci_high"
                    if show_ci and ci_low_col in df.columns and ci_high_col in df.columns:
                        entry['ci_low'] = row[ci_low_col] * scale
                        entry['ci_high'] = row[ci_high_col] * scale
                    plot_data.append(entry)

        # --- Case 2: Multiple metrics ---
        else:
            for metric in metrics:
                train_col, test_col = f"{metric}_train", f"{metric}_test"
                display_metric = metric_label_map.get(metric, metric)
                
                if split in {"train", "both"} and train_col in df.columns:
                    for _, row in df.iterrows():
                        entry = {
                            'Round': row['round'],
                            'Metric Value': row[train_col] * scale,
                            'Split': 'Train',
                            'Metric': display_metric
                        }
                        ci_low_col = f"{metric}_train_ci_low"
                        ci_high_col = f"{metric}_train_ci_high"
                        if show_ci and ci_low_col in df.columns and ci_high_col in df.columns:
                            entry['ci_low'] = row[ci_low_col] * scale
                            entry['ci_high'] = row[ci_high_col] * scale
                        plot_data.append(entry)
                
                if split in {"test", "both"} and test_col in df.columns:
                    for _, row in df.iterrows():
                        entry = {
                            'Round': row['round'],
                            'Metric Value': row[test_col] * scale,
                            'Split': 'Test',
                            'Metric': display_metric
                        }
                        ci_low_col = f"{metric}_test_ci_low"
                        ci_high_col = f"{metric}_test_ci_high"
                        if show_ci and ci_low_col in df.columns and ci_high_col in df.columns:
                            entry['ci_low'] = row[ci_low_col] * scale
                            entry['ci_high'] = row[ci_high_col] * scale
                        plot_data.append(entry)
        
        plot_df = pd.DataFrame(plot_data)
        
        # Determine groupby column for CI drawing and get order (matching seaborn's hue order)
        if len(metrics) == 1:
            groupby_col = 'Split'
            group_order = plot_df['Split'].unique().tolist()
        else:
            groupby_col = 'Metric'
            group_order = plot_df['Metric'].unique().tolist()
        
        # --- Draw CIs first (so lines are on top) ---
        if show_ci and 'ci_low' in plot_df.columns:
            for i, group_val in enumerate(group_order):
                group_df = plot_df[plot_df[groupby_col] == group_val].sort_values('Round')
                color = palette[i % len(palette)]
                
                if ci_style == 'ribbon':
                    ax.fill_between(
                        group_df['Round'],
                        group_df['ci_low'],
                        group_df['ci_high'],
                        alpha=ci_alpha,
                        color=color,
                        linewidth=0,
                    )
                elif ci_style == 'lines':
                    ax.plot(
                        group_df['Round'], group_df['ci_low'],
                        linestyle='--', linewidth=1.5, color=color, alpha=0.4
                    )
                    ax.plot(
                        group_df['Round'], group_df['ci_high'],
                        linestyle='--', linewidth=1.5, color=color, alpha=0.4
                    )
                elif ci_style == 'ribbon+lines':
                    ax.fill_between(
                        group_df['Round'],
                        group_df['ci_low'],
                        group_df['ci_high'],
                        alpha=ci_alpha,
                        color=color,
                        linewidth=0,
                    )
                    ax.plot(
                        group_df['Round'], group_df['ci_low'],
                        linestyle='--', linewidth=1.5, color=color, alpha=0.4
                    )
                    ax.plot(
                        group_df['Round'], group_df['ci_high'],
                        linestyle='--', linewidth=1.5, color=color, alpha=0.4
                    )
                elif ci_style == 'errorbar':
                    pass  # Draw after main plot
        
        # --- Draw true means if requested ---
        true_means_plot_df = None
        if show_true_means and tm_df is not None:
            true_means_data = []
            
            if len(metrics) == 1:
                metric = metrics[0]
                display_metric = metric_label_map.get(metric, metric)
                
                if split in {"train", "both"} and f"{metric}_train" in tm_df.columns:
                    for _, row in tm_df.iterrows():
                        true_means_data.append({
                            'Round': row['round'],
                            'Metric Value': row[f"{metric}_train"] * scale,
                            'Split': 'Train',
                            'Metric': display_metric
                        })
                
                if split in {"test", "both"} and f"{metric}_test" in tm_df.columns:
                    for _, row in tm_df.iterrows():
                        true_means_data.append({
                            'Round': row['round'],
                            'Metric Value': row[f"{metric}_test"] * scale,
                            'Split': 'Test',
                            'Metric': display_metric
                        })
            else:
                for metric in metrics:
                    display_metric = metric_label_map.get(metric, metric)
                    
                    if split in {"train", "both"} and f"{metric}_train" in tm_df.columns:
                        for _, row in tm_df.iterrows():
                            true_means_data.append({
                                'Round': row['round'],
                                'Metric Value': row[f"{metric}_train"] * scale,
                                'Split': 'Train',
                                'Metric': display_metric
                            })
                    
                    if split in {"test", "both"} and f"{metric}_test" in tm_df.columns:
                        for _, row in tm_df.iterrows():
                            true_means_data.append({
                                'Round': row['round'],
                                'Metric Value': row[f"{metric}_test"] * scale,
                                'Split': 'Test',
                                'Metric': display_metric
                            })
            
            true_means_plot_df = pd.DataFrame(true_means_data)
            if not true_means_plot_df.empty:
                marker = None if lines_only else 'o'
                if len(metrics) == 1:
                    sns.lineplot(
                        data=true_means_plot_df, x='Round', y='Metric Value', hue='Split',
                        marker=marker, linewidth=3.0, markersize=7,
                        palette=palette[:2], ax=ax
                    )
                else:
                    if split == "both":
                        sns.lineplot(
                            data=true_means_plot_df, x='Round', y='Metric Value', 
                            hue='Metric', style='Split',
                            marker=marker, linewidth=3.0, markersize=7, palette=palette, ax=ax
                        )
                    else:
                        sns.lineplot(
                            data=true_means_plot_df, x='Round', y='Metric Value', 
                            hue='Metric',
                            marker=marker, linewidth=3.0, markersize=7, palette=palette, ax=ax
                        )
        
        # Main plot with seaborn (skip if showing true means - we already plotted the lines)
        if not plot_df.empty and not show_true_means:
            marker = None if lines_only else 'o'
            if len(metrics) == 1:
                sns.lineplot(
                    data=plot_df, x='Round', y='Metric Value', hue='Split',
                    marker=marker, linewidth=3.0, markersize=7,
                    palette=palette[:2], ax=ax
                )
            else:
                if split == "both":
                    sns.lineplot(
                        data=plot_df, x='Round', y='Metric Value', 
                        hue='Metric', style='Split',
                        marker=marker, linewidth=3.0, markersize=7, palette=palette, ax=ax
                    )
                else:
                    sns.lineplot(
                        data=plot_df, x='Round', y='Metric Value', 
                        hue='Metric',
                        marker=marker, linewidth=3.0, markersize=7, palette=palette, ax=ax
                    )
        
        # Draw error bars after main plot (so they're on top)
        if show_ci and 'ci_low' in plot_df.columns and ci_style == 'errorbar':
            for i, group_val in enumerate(group_order):
                group_df = plot_df[plot_df[groupby_col] == group_val].sort_values('Round')
                color = palette[i % len(palette)]
                yerr_low = group_df['Metric Value'] - group_df['ci_low']
                yerr_high = group_df['ci_high'] - group_df['Metric Value']
                ax.errorbar(
                    group_df['Round'], group_df['Metric Value'],
                    yerr=[yerr_low, yerr_high],
                    fmt='none', ecolor=color, elinewidth=1.5, capsize=3, alpha=0.7
                )
        
        # --- Optional control condition ---
        if control_condition is not None:
            assert split in ['train', 'test']
            control_col = f"{control_condition}_{split}"
            if control_col in df.columns:
                ax.plot(
                    df["round"], df[control_col] * scale,
                    label=f"Control ({control_condition})",
                    linestyle="--", linewidth=2, color="gray", alpha=0.6, marker='D'
                )
        
        # --- Draw baseline metric as a line (varies with round) ---
        if show_baseline and baseline_metrics is not None and idx < len(baseline_metrics):
            baseline_metric = baseline_metrics[idx]
            if baseline_metric is not None:
                baseline_col = f"{baseline_metric}_test"
                if baseline_col in df.columns:
                    ax.plot(
                        df['round'].values, df[baseline_col].values * scale,
                        linestyle=(0, (8, 4)), linewidth=2, 
                        color='#888888', alpha=0.7, zorder=1, label='Majority Class'
                    )
        
        # Darker spines, light grid, subtle background
        ax.spines['top'].set_color('#ffffff')
        ax.spines['right'].set_color('#ffffff')
        ax.spines['left'].set_color('#333333')
        ax.spines['bottom'].set_color('#333333')
        ax.spines['top'].set_linewidth(1.5)
        ax.spines['right'].set_linewidth(1.5)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.set_facecolor(PLOT_BACKGROUND_COLOR)
        # X-axis label (skip if using shared_xlabel)
        xlabel_size = 15 if ncols == 2 else 18
        if shared_xlabel is None:
            if xlabels is not None and idx < len(xlabels) and xlabels[idx] is not None:
                ax.set_xlabel(xlabels[idx], fontsize=xlabel_size, fontweight='bold')
            elif ncols_grid > 1:
                # Repeat default xlabel under each facet when ncols > 1
                ax.set_xlabel("Round", fontsize=xlabel_size, fontweight='bold')
            else:
                ax.set_xlabel("", fontsize=xlabel_size, fontweight='bold')
        else:
            ax.set_xlabel("", fontsize=xlabel_size, fontweight='bold')
        
        # Y-axis label
        ylabel_size = 15 if ncols == 2 else 18
        if ylabels is not None and idx < len(ylabels) and ylabels[idx] is not None:
            ax.set_ylabel(ylabels[idx], fontsize=ylabel_size, fontweight='bold', labelpad=2)
        else:
            ax.set_ylabel("", fontsize=ylabel_size, fontweight='bold', labelpad=2)
        
        # Set x-axis to show only whole numbers
        round_vals = df['round'].unique()
        ax.set_xticks(sorted(round_vals))
        ax.set_xticklabels([int(r) for r in sorted(round_vals)])
        ax.tick_params(axis='both', length=3, pad=2)
        # Subplot title (larger font when multiple rows)
        title_fontsize = 15 if nrows_grid > 1 else 14
        title_pad = 12 if nrows_grid > 1 else 10
        if titles is not None and idx < len(titles):
            ax.set_title(titles[idx], fontsize=title_fontsize, fontweight='bold', pad=title_pad)
        else:
            default_title = f"Metrics vs. Round ({split.capitalize()})" if len(metrics) > 1 else f"{metrics[0]} vs. Round"
            ax.set_title(default_title, fontsize=title_fontsize, fontweight='bold', pad=title_pad)
        
        # Y-axis range with margin
        if ylims is not None and idx < len(ylims) and ylims[idx] is not None:
            scaled_ylim = (ylims[idx][0] * scale, ylims[idx][1] * scale)
            ax.set_ylim(scaled_ylim[0], scaled_ylim[1])
        else:
            all_vals = []
            for metric in metrics:
                if split in {"train", "both"} and f"{metric}_train" in df.columns:
                    all_vals.extend((df[f"{metric}_train"] * scale).dropna().tolist())
                if split in {"test", "both"} and f"{metric}_test" in df.columns:
                    all_vals.extend((df[f"{metric}_test"] * scale).dropna().tolist())
            if all_vals:
                min_y, max_y = min(all_vals), max(all_vals)
                y_margin = (max_y - min_y) * buffer
                y_margin = max(y_margin, 0.03 * scale)
                ax.set_ylim(max(0, min_y - y_margin), max_y + y_margin)
        
        # Annotate points with values if requested (use true_means values if showing true means)
        annotation_size = 12
        if show_values:
            annotation_df = true_means_plot_df if (show_true_means and true_means_plot_df is not None) else plot_df
            if annotation_df is not None and not annotation_df.empty:
                for idx_row, row in annotation_df.iterrows():
                    if show_ci and ci_style == 'errorbar':
                        x_offset = -4
                        y_offset = 10
                        ha = 'right'
                        va = 'top'
                    else:
                        x_offset = 0
                        y_offset = 6
                        ha = 'center'
                        va = 'bottom'
                    
                    val = row['Metric Value']
                    val_str = f"{val:.{value_decimals}f}" if value_decimals is not None else f"{int(val)}"
                    ax.annotate(val_str, 
                               xy=(row['Round'], row['Metric Value']),
                               xytext=(x_offset, y_offset), textcoords='offset points',
                               ha=ha, va=va, fontsize=annotation_size, alpha=0.9)
        
        # --- Draw delta brackets between XYE and XY metrics ---
        if show_delta_brackets and len(metrics) == 2:
            # Get the metric columns for the two metrics
            m1, m2 = metrics[0], metrics[1]  # XYE and XY typically
            m1_col = f"{m1}_test" if split == "test" else f"{m1}_train"
            m2_col = f"{m2}_test" if split == "test" else f"{m2}_train"
            
            # Determine which rounds to annotate
            rounds_in_data = sorted(df['round'].unique())
            if delta_bracket_rounds is None:
                bracket_rounds = [rounds_in_data[0], rounds_in_data[-1]] if len(rounds_in_data) > 1 else rounds_in_data
            else:
                bracket_rounds = [r for r in delta_bracket_rounds if r in rounds_in_data]
            
            # Get bootstrap samples for this facet if available
            facet_bootstrap = None
            if bootstrap_samples_list is not None and idx < len(bootstrap_samples_list):
                facet_bootstrap = bootstrap_samples_list[idx]
            
            for bracket_round in bracket_rounds:
                round_df = df[df['round'] == bracket_round]
                if round_df.empty:
                    continue
                
                # Get values for both metrics at this round (use true means if available)
                if show_true_means and tm_df is not None and len(tm_df) > 0:
                    tm_round = tm_df[tm_df['round'] == bracket_round]
                    if tm_round.empty:
                        print(f"DEBUG FACET {idx}: tm_round EMPTY for round={bracket_round}")
                        continue
                    val1 = tm_round[m1_col].values[0] * scale if m1_col in tm_round.columns else None
                    val2 = tm_round[m2_col].values[0] * scale if m2_col in tm_round.columns else None
                else:
                    val1 = round_df[m1_col].values[0] * scale if m1_col in round_df.columns else None
                    val2 = round_df[m2_col].values[0] * scale if m2_col in round_df.columns else None
                if val1 is None or val2 is None:
                    continue
                delta = val1 - val2  # XYE - XY
                
                # Compute p-value if bootstrap samples available
                p_val = None
                if facet_bootstrap is not None:
                    # Check if bracket_round exists, if not try to find closest available round
                    available_rounds = list(facet_bootstrap.keys())
                    use_round = bracket_round if bracket_round in facet_bootstrap else (max(available_rounds) if available_rounds else None)
                    
                    if use_round is not None and use_round in facet_bootstrap:
                        # Try multiple key formats - bootstrap samples may or may not have _test suffix
                        m1_base = m1.replace('_test', '') if m1.endswith('_test') else m1
                        m2_base = m2.replace('_test', '') if m2.endswith('_test') else m2
                        
                        # Try different key combinations
                        key_options = [
                            (m1, m2),  # Raw as provided
                            (m1_base, m2_base),  # Without _test suffix
                            (f"{m1_base}_test", f"{m2_base}_test"),  # With _test suffix
                        ]
                        
                        for m1_key, m2_key in key_options:
                            if m1_key in facet_bootstrap[use_round] and m2_key in facet_bootstrap[use_round]:
                                m1_samples = facet_bootstrap[use_round][m1_key]
                                m2_samples = facet_bootstrap[use_round][m2_key]
                                # Handle different sample sizes by truncating to min length
                                min_len = min(len(m1_samples), len(m2_samples))
                                betas = m1_samples[:min_len] - m2_samples[:min_len]
                                p_val = bootstrap_pvalue_twosided(betas)
                                break
                        
                        # Fallback: if key lookup failed, try using index position
                        # This handles cases where metric names differ but we have 2 metrics per facet
                        if p_val is None:
                            round_keys = list(facet_bootstrap[use_round].keys())
                            if len(round_keys) >= 2:
                                # Use first two keys (assuming they correspond to metric1 and metric2)
                                m1_samples = facet_bootstrap[use_round][round_keys[0]]
                                m2_samples = facet_bootstrap[use_round][round_keys[1]]
                                # Handle different sample sizes by truncating to min length
                                min_len = min(len(m1_samples), len(m2_samples))
                                betas = m1_samples[:min_len] - m2_samples[:min_len]
                                p_val = bootstrap_pvalue_twosided(betas)
                
                # Draw bracket connecting the two points
                x_pos = bracket_round
                y_top = max(val1, val2)
                y_bot = min(val1, val2)
                # Bracket direction: caps point AWAY from text
                # delta_text_to_left=True -> text on left, caps point right (+1)
                # delta_text_to_left=False -> text on right, caps point left (-1)
                cap_direction = 1 if delta_text_to_left else -1
                bracket_offset = - 0.25  # Move bracket away from x=6 in cap direction
                x_bracket = x_pos + cap_direction * bracket_offset
                bracket_width = 0.15  # Width of horizontal parts
                
                # Draw vertical line connecting the two metrics
                ax.plot([x_bracket, x_bracket], [y_bot, y_top], 
                       color=delta_bracket_color, linewidth=1.5, zorder=5)
                
                # Draw horizontal caps pointing in cap_direction
                ax.plot([x_bracket, x_bracket + cap_direction * bracket_width], [y_top, y_top],
                       color=delta_bracket_color, linewidth=1.5, zorder=5)
                ax.plot([x_bracket, x_bracket + cap_direction * bracket_width], [y_bot, y_bot],
                       color=delta_bracket_color, linewidth=1.5, zorder=5)
                # Build annotation text
                delta_str = f"Δ={delta:+.1f}"
                
                # Format p-value string
                if p_val is not None:
                    if p_val < 1e-4:
                        p_str = "p<1e-4"
                    elif p_val < 0.001:
                        p_str = f"p<.001"
                    elif p_val < 0.01:
                        p_str = f"p={p_val:.3f}"
                    else:
                        p_str = f"p={p_val:.2f}"
                    annotation_text = f"{delta_str}\n{p_str}"
                else:
                    annotation_text = delta_str
                
                # Determine text position
                text_pos = 'inbetween'  # default
                x_offset = -0.2 if delta_text_to_left else 0.2
                ha_align = 'right' if delta_text_to_left else 'left'
                if delta_text_positions is not None and idx < len(delta_text_positions):
                    text_pos = delta_text_positions[idx]
                
                if text_pos == 'inbetween':
                    y_mid = (y_top + y_bot) / 2
                    ax.annotate(annotation_text, 
                               xy=(x_bracket + x_offset, y_mid),
                               ha=ha_align, va='center', fontsize=annotation_size, fontweight='bold',
                               color=delta_bracket_color, zorder=6,
                               linespacing=0.9)
                elif text_pos == 'topright':
                    # Get axis limits for positioning
                    ylim = ax.get_ylim()
                    xlim = ax.get_xlim()
                    ax.annotate(annotation_text, 
                               xy=(xlim[1] + x_offset, ylim[1] - (ylim[1]-ylim[0])*0.08),
                               ha=ha_align, va='top', fontsize=annotation_size, fontweight='bold',
                               color=delta_bracket_color, zorder=6,
                               linespacing=0.9)
                elif text_pos == 'bottomright':
                    ylim = ax.get_ylim()
                    xlim = ax.get_xlim()
                    ax.annotate(annotation_text, 
                               xy=(xlim[1] + x_offset, ylim[0] + (ylim[1]-ylim[0])*0.08),
                               ha=ha_align, va='bottom', fontsize=annotation_size, fontweight='bold',
                               color=delta_bracket_color, zorder=6,
                               linespacing=0.9)
        
        # Remove individual legends - we'll create a shared one
        if ax.get_legend():
            ax.get_legend().remove()
    
    # Create a single shared legend with proper colors
    # Get labels from first subplot, then create custom handles with the right colors
    from matplotlib.lines import Line2D
    handles, labels = axes[0].get_legend_handles_labels()
    
    # Add line breaks at spaces for multi-word labels (helps with right-side legends)
    if legend_loc == 'right':
        labels = [label.replace(' ', '\n') for label in labels]
    
    # Create custom handles with proper colors and line styles
    custom_handles = []
    for i, label in enumerate(labels):
        if 'Majority' in label:  # Check with newline-replaced label
            # Use dashed gray line for majority class baseline
            handle = Line2D([0], [0], color='#888888', linestyle=(0, (8, 4)), linewidth=2, alpha=0.7)
        else:
            # Use palette color with solid line for metrics
            color = palette[i % len(palette)]
            handle = Line2D([0], [0], color=color, linestyle='-', linewidth=3, 
                           marker='o' if not lines_only else None, markersize=7)
        custom_handles.append(handle)
    # Determine legend placement and columns
    n_cols = legend_ncol if legend_ncol is not None else len(labels)

    legend_size = 16 if ncols == 2 else 18
    legend_font = FontProperties(weight='bold', size=legend_size)
    if legend_loc == 'right':
        fig.legend(custom_handles, labels, frameon=False,
                  prop=legend_font,
                  loc='center left', bbox_to_anchor=(1, 0.5), ncol=n_cols,
                  columnspacing=1.0, handlelength=1.5, handletextpad=0.5)
    else:  # 'bottom' (default)
        ypos = .02 if ncols == 2 else -0.01
        fig.legend(custom_handles, labels, frameon=False,
                  prop=legend_font,
                  loc='upper center', bbox_to_anchor=(0.51, ypos), ncol=n_cols,
                  columnspacing=1.0, handlelength=1.5, handletextpad=0.5)
    
    # Overall title
    if suptitle:
        suptitle_size = 24 if ncols == 2 else 28
        suptitle_y = 1.02 if ncols == 4 else .97
        fig.suptitle(suptitle, fontsize=suptitle_size, fontweight='bold', y=suptitle_y)

    # Add extra vertical space between rows when nrows > 1
    if nrows_grid > 1:
        plt.tight_layout(h_pad=3.0)  # Add vertical padding between rows
    else:
        plt.tight_layout()
    fig.align_ylabels()  # Align ylabels across all subplots

    # Add row labels (annotations centered above each row)
    if row_labels is not None and nrows_grid > 1:
        for row_idx, row_label in enumerate(row_labels):
            if row_idx >= nrows_grid:
                break
            # Get the first and last axes in this row to determine center position
            first_ax_idx = row_idx * ncols_grid
            last_ax_idx = min((row_idx + 1) * ncols_grid - 1, len(axes) - 1)
            first_ax_in_row = axes[first_ax_idx] if first_ax_idx < len(axes) else None
            last_ax_in_row = axes[last_ax_idx] if last_ax_idx < len(axes) else None
            if first_ax_in_row is not None and last_ax_in_row is not None:
                # Position centered above the row
                bbox_first = first_ax_in_row.get_position()
                bbox_last = last_ax_in_row.get_position()
                center_x = (bbox_first.x0 + bbox_last.x1) / 2
                y_pos = bbox_first.y1 + 0.06
                # Add centered, underlined row label
                txt = fig.text(center_x, y_pos, row_label,
                        ha='center', va='bottom', fontsize=16, fontweight='bold',
                        fontstyle='italic', color='#333333')
                # Add underline by drawing a line below the text
                txt_bbox = txt.get_window_extent(renderer=fig.canvas.get_renderer())
                txt_bbox_fig = txt_bbox.transformed(fig.transFigure.inverted())
                line_y = txt_bbox_fig.y0 - 0.005
                fig.add_artist(plt.Line2D([txt_bbox_fig.x0, txt_bbox_fig.x1], [line_y, line_y],
                              transform=fig.transFigure, color='#333333', linewidth=1.5))

    # Add shared x-label centered at bottom
    if shared_xlabel:
        fig.text(0.5, -0.015, shared_xlabel, ha='center', va='bottom', 
                fontsize=15, fontweight='bold')

    if save_name:
        plt.savefig(f"zfigs/{save_name}.pdf", dpi=300, bbox_inches="tight",
        facecolor='white', edgecolor='none')
    plt.show()


## experiment comparison

In [ ]:
def plot_experiment_comparison(
    experiment_configs: list,
    metric: str | list[str],
    x_axis: str = "round",  # "round" or "elapsed_time"
    run_bootstrap: bool = False,
    n_bootstrap: int = 1000,
    n_seeds: int = 3,
    cf_type_filter = None,
    figsize: tuple = (10, 6),
    ylim: tuple = None,
    title: str = None,
    ylabel: str = None,
    xlabel: str = None,
    show_values: bool = False,
    value_decimals: int = None,  # None for int, 1 for .1f, etc.
    ci_alpha: float = 0.2,
    ci_style: str = 'ribbon',  # 'ribbon', 'lines', 'ribbon+lines', or 'errorbar'
    lines_only: bool = False,
    scale: float = 100,
    buffer: float = 0.5,
    force_rerun: bool = False,
    print_pvalues: bool = False,
    save_name: str = None,
    custom_efficiency_annotation: bool = False,
):
    """
    Compare multiple experiments on a single metric.
    Matches the style of plot_metric exactly.
    
    Args:
        experiment_configs: List of dicts, each with:
            - args_path: path to args JSON
            - label: legend label (optional)
        metric: Single metric name to compare, OR a list of metric names (one per experiment)
                to reconcile different metric names across experiments that measure the same thing.
        x_axis: "round" or "elapsed_time"
        run_bootstrap: If True, compute bootstrap CIs. If False, just plot means (no CIs).
        n_bootstrap: Number of bootstrap iterations (only used if run_bootstrap=True)
        n_seeds: Number of seeds to load
        cf_type_filter: Filter to specific CF types (None = use 'all_ID')
        figsize: Figure size
        ylim: Y-axis limits (in original scale, will be multiplied by scale)
        title: Plot title
        ylabel: Y-axis label
        xlabel: X-axis label (auto-set based on x_axis if None)
        show_values: Show value labels on points
        ci_alpha: CI ribbon transparency
        ci_style: 'ribbon', 'lines', 'ribbon+lines', or 'errorbar'
        lines_only: If True, no markers
        scale: Scale factor for metric values (default 100)
        buffer: Margin for y-axis padding
        save_name: If provided, save figure to zfigs/{save_name}.pdf
    
    Returns:
        dict with loaded data for further analysis
    """
    # Normalize metric to a list (one per experiment)
    if isinstance(metric, str):
        metrics_list = [metric] * len(experiment_configs)
    else:
        metrics_list = metric
        if len(metrics_list) != len(experiment_configs):
            raise ValueError(f"metric list length ({len(metrics_list)}) must match experiment_configs length ({len(experiment_configs)})")
    
    # Custom palette - muted, professional colors (matching plot_metric)
    palette = [
        "#6b7fa3",  # muted blue
        "#a67fb8",  # muted purple
        "#7fb88a",  # muted green
        "#5fa89d",  # muted teal
        "#d9844f",  # muted orange
    ]
    
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor(PLOT_BACKGROUND_COLOR)
    
    all_data = []
    plot_data = []
    all_bootstrap_samples = []  # Collect bootstrap samples for cross-experiment comparison
    all_labels = []
    all_rounds = set()
    
    for i, config in enumerate(experiment_configs):
        args_path = config['args_path']
        label = config.get('label', args_path[:40])
        color = palette[i % len(palette)]
        exp_metric = metrics_list[i]  # Use experiment-specific metric
        
        # Load args
        with open(f"results/{args_path}", "r") as f:
            exp_args = json.load(f)
            exp_args = SimpleNamespace(**exp_args)
        
        if run_bootstrap:
            # Use bootstrap for CIs
            rounds_to_compute = list(range(exp_args.train_rounds + 1))
            metrics_hash = hashlib.md5(exp_metric.encode()).hexdigest()[:8]
            cf_filter_str = "ID" if cf_type_filter == ID_CF_TYPES else "OOD" if cf_type_filter == OOD_CF_TYPES else "all"
            cache_path = f"cache/bootstrap_{utils.get_base_exp_name(exp_args)}_{n_seeds}seeds_{cf_filter_str}_{metrics_hash}_{n_bootstrap}.pkl"
            
            results_long, results_wide, bootstrap_samples = bootstrap_evaluation_stats(
                args=exp_args,
                n_seeds=n_seeds,
                rounds=rounds_to_compute,
                simulator_sweep_configs=simulator_sweep_configs,
                metrics_to_compute=[exp_metric],
                cf_type_filter=cf_type_filter,
                n_bootstrap=n_bootstrap,
                confidence_level=0.95,
                random_state=42,
                block_col='id',
                verbose=True,
                cache_path=cache_path,
                force_rerun=force_rerun,
                print_pvalues=False,
            )
            
            # Get values
            rounds = results_wide['round'].values
            y_vals = results_wide[f'{exp_metric}_test'].values
            ci_low = results_wide[f'{exp_metric}_test_ci_low'].values 
            ci_high = results_wide[f'{exp_metric}_test_ci_high'].values
            
            # If x_axis is elapsed_time, get from raw stats
            if x_axis == "elapsed_time":
                stats_df = load_stats_across_seeds(
                    base_exp_name=utils.get_base_exp_name(exp_args),
                    n_seeds=n_seeds,
                )
                stats_df.loc[stats_df['round'] == 0, 'elapsed_time'] = 0.0
                x_vals = stats_df.groupby('round')['elapsed_time'].mean().reindex(rounds).values
            else:
                x_vals = rounds
            
            # Build plot data
            for j, (x, y) in enumerate(zip(x_vals, y_vals)):
                entry = {
                    'X': x,
                    'Metric Value': y * scale,
                    'Experiment': label,
                }
                if ci_low is not None and ci_high is not None:
                    entry['ci_low'] = ci_low[j] * scale
                    entry['ci_high'] = ci_high[j] * scale
                plot_data.append(entry)
            
            # Store bootstrap samples for cross-experiment comparison
            all_bootstrap_samples.append(bootstrap_samples)
            all_labels.append(label)
            all_rounds.update(rounds)
                
        else:
            # Fast path: just load stats, no bootstrap, NO CIs
            stats_df = load_stats_across_seeds(
                base_exp_name=utils.get_base_exp_name(exp_args),
                n_seeds=n_seeds,
            )
            stats_df = add_gmean_from_gmean2(stats_df)
            
            # Fix round 0 elapsed_time
            stats_df.loc[stats_df['round'] == 0, 'elapsed_time'] = 0.0
            
            # Filter to counterfactual type
            # IMPORTANT: When using pre-computed stats, convert list filters to aggregate strings
            # to use the pre-computed aggregate (e.g., 'all_ID') instead of incorrectly averaging
            if cf_type_filter is not None:
                if isinstance(cf_type_filter, list):
                    # Convert list to aggregate string
                    if cf_type_filter == ID_CF_TYPES:
                        cf_filter_resolved = 'all_ID'
                    elif cf_type_filter == OOD_CF_TYPES:
                        cf_filter_resolved = 'all_OOD'
                    else:
                        # For other lists, warn and use first element or skip
                        print(f"Warning: cf_type_filter is a list but not ID_CF_TYPES or OOD_CF_TYPES. Using 'all_ID'.")
                        cf_filter_resolved = 'all_ID'
                    stats_df = stats_df[stats_df['counterfactual_type'] == cf_filter_resolved]
                else:
                    stats_df = stats_df[stats_df['counterfactual_type'] == cf_type_filter]
            else:
                stats_df = stats_df[stats_df['counterfactual_type'] == 'all_ID']
            
            if len(stats_df) == 0:
                print(f"Warning: No data after filtering for {label}")
                continue
            
            # Get metric column
            metric_col = f'{exp_metric}_test' if f'{exp_metric}_test' in stats_df.columns else exp_metric
            if metric_col not in stats_df.columns:
                print(f"Warning: {metric_col} not found. Available: {[c for c in stats_df.columns if 'sim' in c or 'monitor' in c][:10]}")
                continue
            
            # Group by round - now this is just averaging across seeds (same cf_type), which is correct
            grouped = stats_df.groupby('round').agg({
                metric_col: 'mean',
                'elapsed_time': 'mean',
            }).reset_index()
            
            if x_axis == "elapsed_time":
                x_vals = grouped['elapsed_time'].values
            else:
                x_vals = grouped['round'].values
            
            y_vals = grouped[metric_col].values
            
            # Build plot data (no CIs when not bootstrapping)
            for x, y in zip(x_vals, y_vals):
                plot_data.append({
                    'X': x,
                    'Metric Value': y * scale,
                    'Experiment': label,
                })
        
        all_data.append({
            'config': config,
            'label': label,
            'color': color,
        })
    
    plot_df = pd.DataFrame(plot_data)
    
    if plot_df.empty:
        print("No data to plot!")
        return all_data
    
    # Get experiment order for consistent colors
    exp_order = [config.get('label', config['args_path'][:40]) for config in experiment_configs]
    
    # --- Draw CIs first (so lines are on top) ---
    if run_bootstrap and 'ci_low' in plot_df.columns:
        for i, exp_label in enumerate(exp_order):
            exp_df = plot_df[plot_df['Experiment'] == exp_label].sort_values('X')
            color = palette[i % len(palette)]
            
            if ci_style == 'ribbon':
                ax.fill_between(
                    exp_df['X'], exp_df['ci_low'], exp_df['ci_high'],
                    alpha=ci_alpha, color=color, linewidth=0,
                )
            elif ci_style == 'lines':
                ax.plot(exp_df['X'], exp_df['ci_low'], linestyle='--', linewidth=1.5, color=color, alpha=0.4)
                ax.plot(exp_df['X'], exp_df['ci_high'], linestyle='--', linewidth=1.5, color=color, alpha=0.4)
            elif ci_style == 'ribbon+lines':
                ax.fill_between(
                    exp_df['X'], exp_df['ci_low'], exp_df['ci_high'],
                    alpha=ci_alpha, color=color, linewidth=0,
                )
                ax.plot(exp_df['X'], exp_df['ci_low'], linestyle='--', linewidth=1.5, color=color, alpha=0.4)
                ax.plot(exp_df['X'], exp_df['ci_high'], linestyle='--', linewidth=1.5, color=color, alpha=0.4)
    
    # --- Main plot ---
    marker = None if lines_only else 'o'
    sns.lineplot(
        data=plot_df, x='X', y='Metric Value', hue='Experiment',
        marker=marker, linewidth=3.0, markersize=7,
        palette=palette[:len(exp_order)], ax=ax,
        hue_order=exp_order,
    )
    
    # Draw error bars after main plot
    if run_bootstrap and 'ci_low' in plot_df.columns and ci_style == 'errorbar':
        for i, exp_label in enumerate(exp_order):
            exp_df = plot_df[plot_df['Experiment'] == exp_label].sort_values('X')
            color = palette[i % len(palette)]
            yerr_low = exp_df['Metric Value'] - exp_df['ci_low']
            yerr_high = exp_df['ci_high'] - exp_df['Metric Value']
            ax.errorbar(
                exp_df['X'], exp_df['Metric Value'],
                yerr=[yerr_low, yerr_high],
                fmt='none', ecolor=color, elinewidth=1.5, capsize=3, alpha=0.7
            )
    # Custom efficiency annotation: dotted gray line at y=0.8 from x=2 to x=11
    if custom_efficiency_annotation:
        y_val = 0.8 * scale  # Scale to match the plot units
        x_start, x_end = 1.95, 10.95
        ax.plot([x_start, x_end], [y_val, y_val], 
                color='black', linestyle=(0, (8, 4)), linewidth=2, alpha=0.7)
        # Add text annotation above the midpoint
        x_mid = (x_start + x_end) / 2
        ax.text(x_mid, y_val + 1.5, 'Equal Performance', 
                ha='center', va='bottom', fontsize=10, color='black', fontstyle='italic')
    
    # --- Styling (matching plot_metric exactly) ---
    ax.spines['top'].set_color('#ffffff')
    ax.spines['right'].set_color('#ffffff')
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.tick_params(axis='x', pad=-1)
    if xlabel is None:
        xlabel = "Elapsed Time (hours)" if x_axis == "elapsed_time" else "Round"
    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel if ylabel else "Metric Value", fontsize=13, fontweight='bold')
    
    if x_axis == "round":
        x_vals_all = plot_df['X'].unique()
        ax.set_xticks(sorted(x_vals_all))
        ax.set_xticklabels([int(r) for r in sorted(x_vals_all)])
    ax.tick_params(axis='x', pad=-1)
    
    if title:
        ax.set_title(title, fontsize=15, fontweight='bold', pad=10)
    
    if ylim is not None:
        ax.set_ylim(ylim[0] * scale, ylim[1] * scale)
    else:
        all_vals = plot_df['Metric Value'].dropna().tolist()
        if all_vals:
            min_y, max_y = min(all_vals), max(all_vals)
            y_margin = (max_y - min_y) * buffer
            y_margin = max(y_margin, 0.03 * scale)
            ax.set_ylim(max(0, min_y - y_margin), max_y + y_margin)
    
    # Annotate points with values
    if show_values:
        for _, row in plot_df.iterrows():
            ax.annotate(f"{int(row['Metric Value'])}", 
                       xy=(row['X'], row['Metric Value']),
                       xytext=(0, 6), textcoords='offset points',
                       ha='center', va='bottom', fontsize=11, alpha=0.9)
    
    ax.legend(frameon=False, 
             loc='upper center', bbox_to_anchor=(0.5, -0.13), 
             ncol=len(exp_order), prop={'weight': 'bold', 'size': 12})
    
    plt.tight_layout()
    
    if save_name:
        plt.savefig(f"zfigs/{save_name}.pdf", bbox_inches='tight', dpi=300)
        print(f"Saved to zfigs/{save_name}.pdf")
    
    plt.show()
    
    # Print cross-experiment comparison table if we have exactly 2 experiments with bootstrap
    # Note: p-value comparison may not be meaningful when metrics differ across experiments
    if run_bootstrap and print_pvalues and len(all_bootstrap_samples) == 2:
        # Use first metric for display (they should be comparable)
        display_metric = metrics_list[0] if isinstance(metric, list) else metric
        print_exp1_vs_exp2_table(
            bootstrap_samples_list=all_bootstrap_samples,
            experiment_labels=all_labels,
            metric=display_metric,
            rounds=sorted(all_rounds),
        )
    
    return all_data

In [ ]:
# Convert gmean2 columns to gmean by taking sqrt (if gmean2 exists but gmean doesn't)
def add_gmean_from_gmean2(df):
    """If gmean2 columns exist but gmean columns don't, compute gmean = sqrt(gmean2)."""
    new_cols = {}
    for col in df.columns:
        if 'gmean2' in col:
            gmean_col = col.replace('gmean2', 'gmean')
            if gmean_col not in df.columns:
                new_cols[gmean_col] = np.sqrt(df[col])
    if new_cols:
        print(f"Created {len(new_cols)} gmean columns from gmean2: {list(new_cols.keys())[:3]}...")
        for col_name, col_data in new_cols.items():
            df[col_name] = col_data
    return df

# Facet lead plot

In [ ]:
simulator_sweep_configs = [
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': True},  # cf-sim-xye
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': False},  # cf-sim-xy
]

facet_configs = [
    {
        "args_path": "args_lead_fig_snli-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        # "metrics": ['all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy'],
        "metrics": ['all_bias_monitor_acc_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_acc_k-0_CoT-F_cf-sim-xy'],
        "baseline_metric": 'all_monitor_baseline_never_influenced',  # Baseline for monitor accuracy
        "cf_type_filter": ID_CF_TYPES,  # Use all cf types
        # "cf_type_filter": OOD_CF_TYPES,  # Use all cf types
        # "title": "Monitor G-Mean (Cue-based Counterfactuals)",
        "title": "Monitor Accuracy (Cue-based Counterfactuals)",
        "ylabel": "Accuracy",
        "xlabel": None,
        # "ylim": (0, 1),
        # "ylim": (.8, 1),
        "ylim": (.5, 1),
        "legend_labels": ["Reasoning Monitor", "Outcome-only Monitor"],
    },
    {
        "args_path": "args_main_sweep_snli-2way_model_based_gpt-oss-120b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_6rounds_e20_16samples_deepseek-chat_n1000-2000_sd0.json",
        "metrics": ['all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xye', 'all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xy'],
        "baseline_metric": 'all_sim_baseline_majority_class',  # Baseline for simulator accuracy
        "cf_type_filter": None,  # Use all cf types
        "title": "Simulator Accuracy (Model-based Counterfactuals)",
        "ylabel": "Accuracy",
        "xlabel": "Round",
        # "ylim": (.7, .8),
        # "ylim": (.75, .85),
        "ylim": (.555, .735),
        "legend_labels": ["Reasoning Simulator", "Outcome-only Simulator"],
    },
]

# Run bootstrap for each config
n_seeds = 5
n_bootstrap = 1000
bootstrap_results_list = []

for i, config in enumerate(facet_configs):
    print(f"\n=== Facet {i+1}: {config['title']} ===")
    
    # Load args for this experiment
    with open(f"results/{config['args_path']}", "r") as f:
        facet_args = json.load(f)
        facet_args = SimpleNamespace(**facet_args)
    
    # Compute all rounds
    rounds_to_compute = list(range(facet_args.train_rounds + 1))
    
    # Create a hash of the metrics for cache naming
    all_metrics = config['metrics'] + ([config['baseline_metric']] if config.get('baseline_metric') else [])
    metrics_hash = hashlib.md5('_'.join(sorted(all_metrics)).encode()).hexdigest()[:8]
    cf_filter_str = "ID" if config['cf_type_filter'] == ID_CF_TYPES else "OOD" if config['cf_type_filter'] == OOD_CF_TYPES else "all"
    cache_path = f"cache/bootstrap_{utils.get_base_exp_name(facet_args)}_{n_seeds}seeds_{cf_filter_str}_{metrics_hash}_{n_bootstrap}.pkl"
    
    print(f"Args: {config['args_path'][:50]}...")
    print(f"Metrics: {all_metrics}")
    print(f"Rounds to compute: {rounds_to_compute}")
    print(f"Cache: {cache_path}")
    
    results_long, results_wide, bootstrap_samples, true_means_wide = bootstrap_evaluation_stats(
        args=facet_args,
        n_seeds=n_seeds,
        rounds=rounds_to_compute,
        simulator_sweep_configs=simulator_sweep_configs,
        metrics_to_compute=all_metrics,
        cf_type_filter=config['cf_type_filter'],
        n_bootstrap=n_bootstrap,
        confidence_level=0.95,
        random_state=42,
        block_col='id',
        verbose=True,
        cache_path=cache_path,
        force_rerun=False,
        print_pvalues=True,  # Print p-value table for each facet
    )
    
    bootstrap_results_list.append({
        'results_wide': results_wide,
        'bootstrap_samples': bootstrap_samples,
        'true_means_wide': true_means_wide,
        'config': config,
    })
    print(f"Results shape: {results_wide.shape}")

print(f"\nBootstrap complete for {len(facet_configs)} facets!")

In [ ]:
# Faceted plot with bootstrap CIs - using dynamic configs
max_rounds = 6  # Maximum number of rounds to show in plots (None = show all)

metric_groups = [config['metrics'] for config in facet_configs]

# Filter results to only show up to max_rounds
results_list = []
true_means_list = []
bootstrap_samples_list = []
for r in bootstrap_results_list:
    df = r['results_wide']
    tm_df = r['true_means_wide']
    if max_rounds is not None:
        df = df[df['round'] <= max_rounds]
        tm_df = tm_df[tm_df['round'] <= max_rounds]
    results_list.append(df)
    true_means_list.append(tm_df)
    bootstrap_samples_list.append(r['bootstrap_samples'])

plot_metric_facets(
    results_list,
    metric_groups,
    legend_labels_list=[config['legend_labels'] for config in facet_configs],
    titles=[config['title'] for config in facet_configs],
    xlabels=[config.get('xlabel') for config in facet_configs],
    ylabels=[config['ylabel'] for config in facet_configs],
    ylims=[config['ylim'] for config in facet_configs],
    show_values=True,
    suptitle="CST with gpt-oss-120b",
    # suptitle="CST with $\\mathtt{gpt-oss-120b}$",
    # suptitle="Counterfactual Simulation Training (CST) on $\\mathtt{gpt-oss-120b}$",
    # suptitle="Counterfactual Simulation Training on gpt-oss-120b",
    lines_only=False,
    show_ci=False,
    ci_style='ribbon+lines',  # 'ribbon', 'lines', 'ribbon+lines', or 'errorbar'
    ci_alpha=0.15,
    save_name="lead_fig",
    show_true_means=True,
    true_means_dfs=true_means_list,
    show_baseline=True,
    baseline_metrics=[config.get('baseline_metric') for config in facet_configs],
    value_decimals=1,
    ncols=2,
    # legend_ncol=1,
    # legend_loc='right',
    figsize=(12, 5),
    # Delta bracket annotations showing XYE vs XY gap
    show_delta_brackets=True,
    delta_bracket_rounds=[6],  # Show bracket at final round only
    bootstrap_samples_list=bootstrap_samples_list,
    delta_bracket_color='#c97b84',  # Rose-pink
    delta_text_to_left=False,
)

# Facet Main Plot

In [ ]:
simulator_sweep_configs = [
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': True},  # cf-sim-xye
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': False},  # cf-sim-xy
]

cue_cf_types = ID_CF_TYPES
# cue_cf_types = OOD_CF_TYPES
# 2x4 Facet Plot: Top row = cue-based (6-cf-types), Bottom row = model-based
# Datasets: snli, ethics-commonsense, ethics-justice, mmlu-pro-law
facet_configs = [
    # === TOP ROW: Cue-based (6-cf-types) - Monitor G-Mean ===
    {
        "args_path": "args_main_sweep_snli-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        "metrics": ['all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": cue_cf_types,
        "title": "SNLI",
        "ylabel": "Monitor G-Mean",
        "xlabel": None,
        "ylim": (0, 1),
        "legend_labels": ["Reasoning Monitor", "Outcome-only Monitor"],
    },
    {
        "args_path": "args_main_sweep_ethics-commonsense-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        "metrics": ['all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": cue_cf_types,
        "title": "Ethics-Commonsense",
        "ylabel": None,
        "xlabel": None,
        "ylim": (0, 1),
        "legend_labels": ["Reasoning Monitor", "Outcome-only Monitor"],
    },
    {
        "args_path": "args_main_sweep_ethics-justice-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        "metrics": ['all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": cue_cf_types,
        "title": "Ethics-Justice",
        "ylabel": None,
        "xlabel": None,
        "ylim": (0, 1),
        "legend_labels": ["Reasoning Monitor", "Outcome-only Monitor"],
    },
    {
        "args_path": "args_main_sweep_mmlu-pro-law-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_Qwen3-235B-A22B-tput_n640-320_sd0.json",
        "metrics": ['all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": cue_cf_types,
        "title": "MMLU-Pro-Law",
        "ylabel": None,
        "xlabel": None,
        "ylim": (0, 1),
        "legend_labels": ["Reasoning Monitor", "Outcome-only Monitor"],
    },
    # === BOTTOM ROW: Model-based - Simulator Accuracy ===
    {
        "args_path": "args_main_sweep_snli-2way_model_based_gpt-oss-120b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_6rounds_e20_16samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        "metrics": ['all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xye', 'all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": None,
        "title": "SNLI",
        "ylabel": "Simulator Acc",
        "xlabel": "Round",
        "ylim": (.745, .825),
        "legend_labels": ["Reasoning Simulator", "Outcome-only Simulator"],
    },
    {
        "args_path": "args_main_sweep_ethics-commonsense-2way_model_based_gpt-oss-120b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_6rounds_e20_16samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        "metrics": ['all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xye', 'all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": None,
        "title": "Ethics-Commonsense",
        "ylabel": None,
        "xlabel": "Round",
        "ylim": (.72, .8),
        "legend_labels": ["Reasoning Simulator", "Outcome-only Simulator"],
    },
    {
        "args_path": "args_main_sweep_ethics-justice-2way_model_based_gpt-oss-120b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_6rounds_e20_16samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        "metrics": ['all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xye', 'all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": None,
        "title": "Ethics-Justice",
        "ylabel": None,
        "xlabel": "Round",
        "ylim": (.68, .78),
        "legend_labels": ["Reasoning Simulator", "Outcome-only Simulator"],
    },
    {
        "args_path": "args_main_sweep_mmlu-pro-law-2way_model_based_gpt-oss-120b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_6rounds_e20_16samples_Qwen3-235B-A22B-tput_n640-320_sd0.json",
        "metrics": ['all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xye', 'all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": None,
        "title": "MMLU-Pro-Law",
        "ylabel": None,
        "xlabel": "Round",
        "ylim": (.68, .80),
        "legend_labels": ["Reasoning Simulator", "Outcome-only Simulator"],
    },
]

# Run bootstrap for each config
n_seeds = 5
n_bootstrap = 1000
force_rerun = False
bootstrap_results_list = []

for i, config in enumerate(facet_configs):
    print(f"\n=== Facet {i+1}: {config['title']} ===")
    
    # Load args for this experiment
    with open(f"results/{config['args_path']}", "r") as f:
        facet_args = json.load(f)
        facet_args = SimpleNamespace(**facet_args)
    
    # Compute all rounds
    rounds_to_compute = list(range(facet_args.train_rounds + 1))
    
    # Create a hash of the metrics for cache naming
    all_metrics = config['metrics'] + ([config['baseline_metric']] if config.get('baseline_metric') else [])
    metrics_hash = hashlib.md5('_'.join(sorted(all_metrics)).encode()).hexdigest()[:8]
    cf_filter_str = "ID" if config['cf_type_filter'] == ID_CF_TYPES else "OOD" if config['cf_type_filter'] == OOD_CF_TYPES else "all"
    cache_path = f"cache/bootstrap_{utils.get_base_exp_name(facet_args)}_{n_seeds}seeds_{cf_filter_str}_{metrics_hash}_{n_bootstrap}.pkl"
    
    print(f"Args: {config['args_path'][:50]}...")
    print(f"Metrics: {all_metrics}")
    print(f"Rounds to compute: {rounds_to_compute}")
    print(f"Cache: {cache_path}")
    
    results_long, results_wide, bootstrap_samples, true_means_wide = bootstrap_evaluation_stats(
        args=facet_args,
        n_seeds=n_seeds,
        rounds=rounds_to_compute,
        simulator_sweep_configs=simulator_sweep_configs,
        metrics_to_compute=all_metrics,
        cf_type_filter=config['cf_type_filter'],
        n_bootstrap=n_bootstrap,
        confidence_level=0.95,
        random_state=42,
        block_col='id',
        verbose=True,
        cache_path=cache_path,
        force_rerun=force_rerun,
        print_pvalues=True,  # Print p-value table for each facet
    )
    
    bootstrap_results_list.append({
        'results_wide': results_wide,
        'bootstrap_samples': bootstrap_samples,
        'true_means_wide': true_means_wide,
        'config': config,
    })
    print(f"Results shape: {results_wide.shape}")

print(f"\nBootstrap complete for {len(facet_configs)} facets!")

In [ ]:
# Faceted plot with bootstrap CIs - using dynamic configs
max_rounds = 6  # Maximum number of rounds to show in plots (None = show all)

metric_groups = [config['metrics'] for config in facet_configs]

# Filter results to only show up to max_rounds
results_list = []
true_means_list = []
bootstrap_samples_list = []
for r in bootstrap_results_list:
    df = r['results_wide']
    tm_df = r['true_means_wide']
    if max_rounds is not None:
        df = df[df['round'] <= max_rounds]
        tm_df = tm_df[tm_df['round'] <= max_rounds]
    results_list.append(df)
    true_means_list.append(tm_df)
    bootstrap_samples_list.append(r['bootstrap_samples'])

plot_metric_facets(
    results_list,
    metric_groups,
    legend_labels_list=[config['legend_labels'] for config in facet_configs],
    titles=[config['title'] for config in facet_configs],
    xlabels=[config.get('xlabel') for config in facet_configs],
    ylabels=[config['ylabel'] for config in facet_configs],
    ylims=[config['ylim'] for config in facet_configs],
    show_values=True,
    # suptitle="CST with gpt-oss-120b",
    suptitle="Counterfactual Simulation Training",
    # suptitle="CST with $\\mathtt{gpt-oss-120b}$",
    # suptitle="Counterfactual Simulation Training (CST) on $\\mathtt{gpt-oss-120b}$",
    # suptitle="Counterfactual Simulation Training on gpt-oss-120b",
    lines_only=False,
    show_ci=False,
    ci_style='ribbon+lines',  # 'ribbon', 'lines', 'ribbon+lines', or 'errorbar'
    ci_alpha=0.15,
    save_name=f"main_fig",
    show_true_means=True,
    true_means_dfs=true_means_list,
    show_baseline=True,
    baseline_metrics=[config.get('baseline_metric') for config in facet_configs],
    value_decimals=1,
    ncols=4,  # 4 columns (one per dataset)
    # legend_ncol=1,
    # legend_loc='right',
    figsize=(16, 8),  # Wider for 2x4 layout
    # Delta bracket annotations showing XYE vs XY gap
    show_delta_brackets=True,
    delta_bracket_rounds=[6],  # Show bracket at final round only
    bootstrap_samples_list=bootstrap_samples_list,
    delta_bracket_color='#c97b84',  # Rose-pink
    delta_text_positions=['inbetween']*4 + ['bottomright', 'bottomright', 'topright', 'topright'],
    delta_text_to_left=True,  # Brackets angle to the RIGHT
    shared_xlabel="Round",  # Single centered x-label
    row_labels=["Cue-based Counterfactuals", "Model-based Counterfactuals"],
)

# MMLU facet

In [ ]:
simulator_sweep_configs = [
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': True},  # cf-sim-xye
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': False},  # cf-sim-xy
]

facet_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json",
        "metrics": ['all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy'],
        # "metrics": ['all_bias_monitor_acc_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_acc_k-0_CoT-F_cf-sim-xy'],
        # "baseline_metric": 'all_monitor_baseline_never_influenced',  # Baseline for monitor accuracy
        "cf_type_filter": ID_CF_TYPES,  # Use all cf types
        # "cf_type_filter": OOD_CF_TYPES,  # Use all cf types
        # "title": "Monitor G-Mean (Cue-based Counterfactuals)",
        "title": "Monitor G-mean (Cue-based Counterfactuals)",
        "ylabel": "G-mean",
        "xlabel": None,
        "ylim": (0, 1),
        # "ylim": (.8, 1),
        # "ylim": (.5, 1),
        "legend_labels": ["Reasoning Monitor", "Outcome-only Monitor"],
    },
    {
        "args_path": "args_main_sweep_mmlu-2way_model_based_gpt-oss-120b_unlik_cfa-0.2_faith_k-0_CoT-F_cf-sim-xye_6rounds_e20_16samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        "metrics": ['all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xye', 'all_sim_greedy_cot_acc_k-0_CoT-F_cf-sim-xy'],
        # "baseline_metric": 'all_sim_baseline_majority_class',  # Baseline for simulator accuracy
        "cf_type_filter": None,  # Use all cf types
        "title": "Simulator Accuracy (Model-based Counterfactuals)",
        "ylabel": "Accuracy",
        "xlabel": "Round",
        # "ylim": (.7, .8),
        # "ylim": (.75, .85),
        "ylim": (.855, .935),
        "legend_labels": ["Reasoning Simulator", "Outcome-only Simulator"],
    },
]

# Run bootstrap for each config
n_seeds = 5
n_bootstrap = 10
bootstrap_results_list = []

for i, config in enumerate(facet_configs):
    print(f"\n=== Facet {i+1}: {config['title']} ===")
    
    # Load args for this experiment
    with open(f"results/{config['args_path']}", "r") as f:
        facet_args = json.load(f)
        facet_args = SimpleNamespace(**facet_args)
    
    # Compute all rounds
    rounds_to_compute = list(range(facet_args.train_rounds + 1))
    
    # Create a hash of the metrics for cache naming
    all_metrics = config['metrics'] + ([config['baseline_metric']] if config.get('baseline_metric') else [])
    metrics_hash = hashlib.md5('_'.join(sorted(all_metrics)).encode()).hexdigest()[:8]
    cf_filter_str = "ID" if config['cf_type_filter'] == ID_CF_TYPES else "OOD" if config['cf_type_filter'] == OOD_CF_TYPES else "all"
    cache_path = f"cache/bootstrap_{utils.get_base_exp_name(facet_args)}_{n_seeds}seeds_{cf_filter_str}_{metrics_hash}_{n_bootstrap}.pkl"
    
    print(f"Args: {config['args_path'][:50]}...")
    print(f"Metrics: {all_metrics}")
    print(f"Rounds to compute: {rounds_to_compute}")
    print(f"Cache: {cache_path}")
    
    results_long, results_wide, bootstrap_samples, true_means_wide = bootstrap_evaluation_stats(
        args=facet_args,
        n_seeds=n_seeds,
        rounds=rounds_to_compute,
        simulator_sweep_configs=simulator_sweep_configs,
        metrics_to_compute=all_metrics,
        cf_type_filter=config['cf_type_filter'],
        n_bootstrap=n_bootstrap,
        confidence_level=0.95,
        random_state=42,
        block_col='id',
        verbose=True,
        cache_path=cache_path,
        force_rerun=False,
        print_pvalues=True,  # Print p-value table for each facet
    )
    
    bootstrap_results_list.append({
        'results_wide': results_wide,
        'bootstrap_samples': bootstrap_samples,
        'true_means_wide': true_means_wide,
        'config': config,
    })
    print(f"Results shape: {results_wide.shape}")

print(f"\nBootstrap complete for {len(facet_configs)} facets!")

In [ ]:
# Faceted plot with bootstrap CIs - using dynamic configs
max_rounds = 6  # Maximum number of rounds to show in plots (None = show all)

metric_groups = [config['metrics'] for config in facet_configs]

# Filter results to only show up to max_rounds
results_list = []
true_means_list = []
bootstrap_samples_list = []
for r in bootstrap_results_list:
    df = r['results_wide']
    tm_df = r['true_means_wide']
    if max_rounds is not None:
        df = df[df['round'] <= max_rounds]
        tm_df = tm_df[tm_df['round'] <= max_rounds]
    results_list.append(df)
    true_means_list.append(tm_df)
    bootstrap_samples_list.append(r['bootstrap_samples'])

plot_metric_facets(
    results_list,
    metric_groups,
    legend_labels_list=[config['legend_labels'] for config in facet_configs],
    titles=[config['title'] for config in facet_configs],
    xlabels=[config.get('xlabel') for config in facet_configs],
    ylabels=[config['ylabel'] for config in facet_configs],
    ylims=[config['ylim'] for config in facet_configs],
    show_values=True,
    suptitle="CST on MMLU",
    lines_only=False,
    show_ci=False,
    ci_style='ribbon+lines',  # 'ribbon', 'lines', 'ribbon+lines', or 'errorbar'
    ci_alpha=0.15,
    save_name="mmlu_fig",
    show_true_means=True,
    true_means_dfs=true_means_list,
    show_baseline=True,
    baseline_metrics=[config.get('baseline_metric') for config in facet_configs],
    value_decimals=1,
    ncols=2,
    # legend_ncol=1,
    # legend_loc='right',
    figsize=(12, 5),
    # Delta bracket annotations showing XYE vs XY gap
    show_delta_brackets=False,
    delta_bracket_rounds=[6],  # Show bracket at final round onl
    bootstrap_samples_list=bootstrap_samples_list,
    delta_bracket_color='#c97b84',  # Rose-pink
    delta_text_to_left=False,
)

# Prompting Baselines

In [ ]:
def load_prompting_baselines(
    baseline_configs,
    metrics,
    cf_type_filter,
    n_seeds=5,
    n_bootstrap=1000,
    force_rerun=False,
    verbose=True,
):
    """
    Load prompting baseline experiments and run bootstrap to get true means.
    
    Args:
        baseline_configs: List of dicts with 'args_path' and 'label'
        metrics: List of metric names to compute
        cf_type_filter: ID_CF_TYPES, OOD_CF_TYPES, or None for model_based
        n_seeds: Number of seeds
        n_bootstrap: Number of bootstrap iterations
        force_rerun: Whether to force recomputation
        verbose: Print progress
    
    Returns:
        List of dicts with 'label' and 'value' for each baseline
    """
    baseline_results = []
    
    for config in baseline_configs:
        args_path = config['args_path']
        label = config['label']
        
        if verbose:
            print(f"\nLoading baseline: {label}")
            print(f"  Args: {args_path[:60]}...")
        
        # Load args
        with open(f"results/{args_path}", "r") as f:
            baseline_args = json.load(f)
            baseline_args = SimpleNamespace(**baseline_args)
        
        # Run bootstrap for round 0 only (prompting baselines have 0 training rounds)
        metrics_hash = hashlib.md5('_'.join(sorted(metrics)).encode()).hexdigest()[:8]
        
        # Determine cf filter string for cache
        if cf_type_filter == ID_CF_TYPES:
            cf_filter_str = "ID"
        elif cf_type_filter == OOD_CF_TYPES:
            cf_filter_str = "OOD"
        elif cf_type_filter is None or cf_type_filter == ["model_based"]:
            cf_filter_str = "model_based"
        else:
            cf_filter_str = "all"
        
        cache_path = f"cache/bootstrap_baseline_{utils.get_base_exp_name(baseline_args)}_{n_seeds}seeds_{cf_filter_str}_{metrics_hash}_{n_bootstrap}.pkl"
        
        results_long, results_wide, bootstrap_samples, true_means_wide = bootstrap_evaluation_stats(
            args=baseline_args,
            n_seeds=n_seeds,
            rounds=[0],  # Prompting baselines only have round 0
            simulator_sweep_configs=simulator_sweep_configs,
            metrics_to_compute=metrics,
            cf_type_filter=cf_type_filter,
            n_bootstrap=n_bootstrap,
            confidence_level=0.95,
            random_state=42,
            block_col='id',
            verbose=verbose,
            cache_path=cache_path,
            force_rerun=force_rerun,
            print_pvalues=False,
        )
        
        # Extract the metric value from true means (round 0)
        # Use the first metric in the list for the baseline value
        metric_col = f"{metrics[0]}_test"
        
        if not true_means_wide.empty and metric_col in true_means_wide.columns:
            round0_row = true_means_wide[true_means_wide['round'] == 0]
            if len(round0_row) > 0:
                value = round0_row[metric_col].values[0]
                baseline_results.append({
                    'label': label,
                    'value': value,
                    'bootstrap_samples': bootstrap_samples,
                    'true_means_wide': true_means_wide,
                    'results_wide': results_wide,  # Contains CI columns
                })
                if verbose:
                    print(f"  {label}: {value:.4f} ({metric_col})")
            else:
                print(f"  WARNING: No round 0 data for {label}")
        else:
            print(f"  WARNING: Metric {metric_col} not found for {label}")
    
    return baseline_results

In [ ]:
# Facet plot with prompting baselines
# This compares CST training curves against prompting-only baselines
# Uses control variables DATASET, TASK_MODEL, MONITOR_METRIC, CF_TYPES_STR, TRAINING_ARGS from the cell above

# Define facet configs for training experiments (dynamically built from control variables)
# Both facets use the same TRAINING_ARGS, just with different cf_type_filter
cf_filter = ID_CF_TYPES
# cf_filter = OOD_CF_TYPES

facet_configs_with_baselines = [
    {
        "args_path": "args_scaling_model_mmlu-2way_6-cf-types_Qwen3-235B-A22B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "metrics": ['all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": cf_filter,
        "title": f"Qwen3-235B",
        "ylabel": "Monitor G-Mean",
        "xlabel": "Round",
        # "ylim": (.5, 1),
        "ylim": (0, 1),
        "legend_labels": ["Reasoning Monitor", "Outcome-only Monitor"],
        "baseline_experiments": [
            {
                "args_path": "args_prompting_qwen_principles_mmlu-2way_6-cf-types_qwen3-235b-a22b-principles_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_0rounds_e0_1samples_deepseek-chat_n0-2000_sd0.json",
                "label": "General Principles",
            },
            {
                "args_path": "args_prompting_qwen_exhaustive_mmlu-2way_6-cf-types_qwen3-235b-a22b-exhaustive_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_0rounds_e0_1samples_deepseek-chat_n0-2000_sd0.json",
                "label": "Exhaustive Reasoning",
            },
            {
                "args_path": "args_prompting_qwen_faithful_def_mmlu-2way_6-cf-types_qwen3-235b-a22b-faithful_def_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_0rounds_e0_1samples_deepseek-chat_n0-2000_sd0.json",
                "label": "Faithfulness Definition",
            },
            {
                "args_path": "args_prompting_qwen_test_description_mmlu-2way_6-cf-types_qwen3-235b-a22b-test_description_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_0rounds_e0_1samples_deepseek-chat_n0-2000_sd0.json",
                "label": "Test Description",
            },
        ],
    },
    # Second facet with cue-based CFs (using 6-cf-types) - separate training run
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "metrics": ['all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye', 'all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xy'],
        "cf_type_filter": cf_filter,
        "title": "gpt-oss-120b",
        "ylabel": "G-Mean",
        "xlabel": "Round",
        # "ylim": (.5, 1),
        "ylim": (0, 1),
        "legend_labels": ["Reasoning Monitor", "Outcome-only Monitor"],
        "baseline_experiments": [
            {
                "args_path": "args_prompting_gpt-oss_low_mmlu-2way_6-cf-types_gpt-oss-120b_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_0rounds_e0_1samples_deepseek-chat_n0-2000_sd0.json",
                "label": "Low Reasoning Effort",
            },
            {
                "args_path": "args_prompting_gpt-oss_medium_mmlu-2way_6-cf-types_gpt-oss-120b_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_0rounds_e0_1samples_deepseek-chat_n0-2000_sd0.json",
                "label": "Medium Reasoning Effort",
            },
            {
                "args_path": "args_prompting_gpt-oss_high_mmlu-2way_6-cf-types_gpt-oss-120b_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_0rounds_e0_1samples_deepseek-chat_n0-2000_sd0.json",
                "label": "High Reasoning Effort",
            },
        ],
    },
]

# --- Load training experiment data with bootstrap ---
n_seeds = 5
n_bootstrap = 100  # Quick check (use 1000 for final results)
n_cols = 2  # Number of columns in facet plot
force_rerun = False

training_results_list = []
training_true_means_list = []
training_bootstrap_samples_list = []  # Store bootstrap samples for delta CIs
baseline_true_means_per_facet = []

for i, config in enumerate(facet_configs_with_baselines):
    print(f"\n{'='*60}")
    print(f"Facet {i+1}: {config['title']}")
    print(f"{'='*60}")
    
    # Load args for training experiment
    with open(f"results/{config['args_path']}", "r") as f:
        facet_args = json.load(f)
        facet_args = SimpleNamespace(**facet_args)
    
    # Compute all rounds
    rounds_to_compute = list(range(facet_args.train_rounds + 1))
    
    # Create cache path
    all_metrics = config['metrics']
    metrics_hash = hashlib.md5('_'.join(sorted(all_metrics)).encode()).hexdigest()[:8]
    
    # Determine cf filter string
    cf_filter = config['cf_type_filter']
    if cf_filter == ID_CF_TYPES:
        cf_filter_str = "ID"
    elif cf_filter == OOD_CF_TYPES:
        cf_filter_str = "OOD"
    elif cf_filter == ["model_based"] or cf_filter is None:
        cf_filter_str = "model_based"
    else:
        cf_filter_str = "all"
    
    cache_path = f"cache/bootstrap_promptbaseline_{utils.get_base_exp_name(facet_args)}_{n_seeds}seeds_{cf_filter_str}_{metrics_hash}_{n_bootstrap}.pkl"
    
    print(f"Training experiment: {config['args_path'][:60]}...")
    print(f"Metrics: {all_metrics}")
    
    results_long, results_wide, bootstrap_samples, true_means_wide = bootstrap_evaluation_stats(
        args=facet_args,
        n_seeds=n_seeds,
        rounds=rounds_to_compute,
        simulator_sweep_configs=simulator_sweep_configs,
        metrics_to_compute=all_metrics,
        cf_type_filter=cf_filter,
        n_bootstrap=n_bootstrap,
        confidence_level=0.95,
        random_state=42,
        block_col='id',
        verbose=True,
        cache_path=cache_path,
        force_rerun=force_rerun,
        print_pvalues=True,
    )
    
    training_results_list.append(results_wide)
    training_true_means_list.append(true_means_wide)
    training_bootstrap_samples_list.append(bootstrap_samples)  # Store for delta CIs
    
    # --- Load prompting baselines for this facet ---
    print(f"\nLoading prompting baselines for facet {i+1}...")
    baseline_configs = config.get('baseline_experiments', [])
    
    if baseline_configs:
        baselines = load_prompting_baselines(
            baseline_configs=baseline_configs,
            metrics=all_metrics,
            cf_type_filter=cf_filter,
            n_seeds=n_seeds,
            n_bootstrap=n_bootstrap,
            force_rerun=force_rerun,
            verbose=True,
        )
        baseline_true_means_per_facet.append(baselines)
    else:
        baseline_true_means_per_facet.append([])

print(f"\n{'='*60}")
print(f"Data loading complete for {len(facet_configs_with_baselines)} facets!")
print(f"{'='*60}")

In [ ]:
def prepare_xye_xy_comparison_data(
    training_true_means_list,
    baseline_true_means_per_facet,
    facet_configs,
    facet_idx=0,
    training_results_list=None,  # Contains CIs from bootstrap
    training_bootstrap_samples_list=None,  # Bootstrap samples for delta CIs
    confidence_level=0.95,
):
    """
    Prepare conditions data for plot_xye_xy_comparison_bars from loaded experiment data.
    
    Args:
        training_true_means_list: List of true means DataFrames for training experiments
        baseline_true_means_per_facet: List of lists with baseline results
        facet_configs: The facet configuration list
        facet_idx: Which facet to extract data for
        training_results_list: List of results_wide DataFrames with CI columns (optional)
        training_bootstrap_samples_list: List of bootstrap_samples dicts for computing delta CIs
        confidence_level: Confidence level for delta CIs (default 0.95)
    
    Returns:
        List of condition dicts ready for plot_xye_xy_comparison_bars
    """
    config = facet_configs[facet_idx]
    metrics = config['metrics']
    
    # Assume first metric is XYE and second is XY
    metric_xye = metrics[0]
    metric_xy = metrics[1]
    
    metric_xye_col = f"{metric_xye}_test"
    metric_xy_col = f"{metric_xy}_test"
    metric_xye_ci_low = f"{metric_xye}_test_ci_low"
    metric_xye_ci_high = f"{metric_xye}_test_ci_high"
    metric_xy_ci_low = f"{metric_xy}_test_ci_low"
    metric_xy_ci_high = f"{metric_xy}_test_ci_high"
    
    alpha = 1 - confidence_level
    
    conditions_data = []
    
    # --- CST: Before (round 0) and After (last round) ---
    tm_df = training_true_means_list[facet_idx]
    results_df = training_results_list[facet_idx] if training_results_list else None
    bootstrap_samples = training_bootstrap_samples_list[facet_idx] if training_bootstrap_samples_list else None
    
    # Helper to extract CI if available
    def get_ci(df, row_filter, ci_low_col, ci_high_col):
        if df is None or ci_low_col not in df.columns:
            return None
        row = df[row_filter]
        if len(row) > 0:
            return (row[ci_low_col].values[0], row[ci_high_col].values[0])
        return None
    
    # Helper to compute delta CI from bootstrap samples
    def get_delta_ci(bootstrap_samples, round_num, metric_xye, metric_xy):
        if bootstrap_samples is None:
            return None
        if round_num not in bootstrap_samples:
            return None
        if metric_xye not in bootstrap_samples[round_num] or metric_xy not in bootstrap_samples[round_num]:
            return None
        xye_samples = bootstrap_samples[round_num][metric_xye]
        xy_samples = bootstrap_samples[round_num][metric_xy]
        delta_samples = xye_samples - xy_samples
        ci_low = np.percentile(delta_samples, 100 * alpha / 2)
        ci_high = np.percentile(delta_samples, 100 * (1 - alpha / 2))
        return (ci_low, ci_high)
    
    # Round 0 = Before CST
    r0_row = tm_df[tm_df['round'] == 0]
    before_cst_data = None
    if len(r0_row) > 0:
        r0_filter = results_df['round'] == 0 if results_df is not None else None
        before_cst_data = {
            'label': 'Before CST',
            'xye_value': r0_row[metric_xye_col].values[0],
            'xy_value': r0_row[metric_xy_col].values[0],
            'xye_ci': get_ci(results_df, r0_filter, metric_xye_ci_low, metric_xye_ci_high),
            'xy_ci': get_ci(results_df, r0_filter, metric_xy_ci_low, metric_xy_ci_high),
            'delta_ci': get_delta_ci(bootstrap_samples, 0, metric_xye, metric_xy),
        }
    
    # Last round = After CST
    # Use last available round in bootstrap_samples for delta CI if last_round is missing
    last_round = tm_df['round'].max()
    last_row = tm_df[tm_df['round'] == last_round]
    after_cst_data = None
    if len(last_row) > 0:
        last_filter = results_df['round'] == last_round if results_df is not None else None
        
        # Try to get delta CI from last round, fall back to previous rounds if missing
        delta_ci = None
        if bootstrap_samples is not None:
            for try_round in range(int(last_round), -1, -1):
                delta_ci = get_delta_ci(bootstrap_samples, try_round, metric_xye, metric_xy)
                if delta_ci is not None:
                    break
        
        after_cst_data = {
            'label': 'After CST',
            'xye_value': last_row[metric_xye_col].values[0],
            'xy_value': last_row[metric_xy_col].values[0],
            'xye_ci': get_ci(results_df, last_filter, metric_xye_ci_low, metric_xye_ci_high),
            'xy_ci': get_ci(results_df, last_filter, metric_xy_ci_low, metric_xy_ci_high),
            'delta_ci': delta_ci,
        }
    
    # Build conditions_data in order: Before CST, Prompting Baselines, After CST
    if before_cst_data:
        conditions_data.append(before_cst_data)
    
    # --- Prompting baselines ---
    baselines = baseline_true_means_per_facet[facet_idx]
    
    for baseline in baselines:
        label = baseline['label']
        baseline_tm = baseline['true_means_wide']
        baseline_results = baseline.get('results_wide')  # Contains CIs
        baseline_bootstrap = baseline.get('bootstrap_samples')  # For delta CIs
        
        # Get round 0 values (prompting only has round 0)
        r0_row = baseline_tm[baseline_tm['round'] == 0]
        if len(r0_row) > 0 and metric_xye_col in r0_row.columns and metric_xy_col in r0_row.columns:
            r0_filter = baseline_results['round'] == 0 if baseline_results is not None else None
            conditions_data.append({
                'label': label,
                'xye_value': r0_row[metric_xye_col].values[0],
                'xy_value': r0_row[metric_xy_col].values[0],
                'xye_ci': get_ci(baseline_results, r0_filter, metric_xye_ci_low, metric_xye_ci_high),
                'xy_ci': get_ci(baseline_results, r0_filter, metric_xy_ci_low, metric_xy_ci_high),
                'delta_ci': get_delta_ci(baseline_bootstrap, 0, metric_xye, metric_xy),
            })
    
    # Add After CST at the end
    if after_cst_data:
        conditions_data.append(after_cst_data)
    
    return conditions_data, metric_xye, metric_xy

In [ ]:
# Prepare and plot XYE vs XY comparison bar chart as a FACETED PLOT (side by side)
# Uses data already loaded from the facets-with-baselines section above

# Collect data for all facets
all_conditions_data = []
for facet_idx in range(len(facet_configs_with_baselines)):
    config = facet_configs_with_baselines[facet_idx]
    
    conditions_data, _, _ = prepare_xye_xy_comparison_data(
        training_true_means_list=training_true_means_list,
        baseline_true_means_per_facet=baseline_true_means_per_facet,
        facet_configs=facet_configs_with_baselines,
        facet_idx=facet_idx,
        training_results_list=training_results_list,
        training_bootstrap_samples_list=training_bootstrap_samples_list,
    )
    all_conditions_data.append(conditions_data)
    
    print(f"\n=== Facet {facet_idx + 1}: {config['title']} ===")
    for cond in conditions_data:
        xye_ci_str = f" CI=[{cond['xye_ci'][0]:.3f}, {cond['xye_ci'][1]:.3f}]" if cond.get('xye_ci') else ""
        xy_ci_str = f" CI=[{cond['xy_ci'][0]:.3f}, {cond['xy_ci'][1]:.3f}]" if cond.get('xy_ci') else ""
        delta_ci_str = f" ΔCI=[{cond['delta_ci'][0]:.3f}, {cond['delta_ci'][1]:.3f}]" if cond.get('delta_ci') else ""
        print(f"  {cond['label']}: XYE={cond['xye_value']:.3f}{xye_ci_str}, XY={cond['xy_value']:.3f}{xy_ci_str}, Δ={(cond['xye_value'] - cond['xy_value']):.3f}{delta_ci_str}")

# === Create faceted barplot ===
scale = 100
colors = {
    'xye': "#5fa89d",  # muted teal (Reasoning)
    'xy': "#d9844f",   # muted orange (Outcome-only)
}
ncols = 2

# Layout configuration
n_facets = len(facet_configs_with_baselines)
nrows = int(np.ceil(n_facets / ncols))

# Adjust figsize based on layout
if ncols == 1:
    figsize = (5.5, 5 * nrows)
else:
    figsize = (8.5 * ncols, 5 * nrows)

fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharey=False, squeeze=False)
axes = axes.flatten()[:n_facets]  # Flatten and take only needed axes

for facet_idx, (ax, conditions_data) in enumerate(zip(axes, all_conditions_data)):
    config = facet_configs_with_baselines[facet_idx]
    
    n_conditions = len(conditions_data)
    bar_width = 0.40
    
    # Extract values
    labels = [d['label'] for d in conditions_data]
    xye_vals = np.array([d['xye_value'] for d in conditions_data]) * scale
    xy_vals = np.array([d['xy_value'] for d in conditions_data]) * scale
    deltas = xye_vals - xy_vals
    
    # Extract CIs
    def get_yerr_arrays(conditions_data, ci_key, vals, scale):
        low_errs = []
        high_errs = []
        for i, d in enumerate(conditions_data):
            ci = d.get(ci_key)
            if ci is not None:
                ci_low = ci[0] * scale
                ci_high = ci[1] * scale
                low_errs.append(max(0, vals[i] - ci_low))
                high_errs.append(max(0, ci_high - vals[i]))
            else:
                low_errs.append(0)
                high_errs.append(0)
        return [np.array(low_errs), np.array(high_errs)]
    
    any_has_ci = any(d.get('xye_ci') is not None for d in conditions_data)
    if any_has_ci:
        xye_yerr = get_yerr_arrays(conditions_data, 'xye_ci', xye_vals, scale)
        xy_yerr = get_yerr_arrays(conditions_data, 'xy_ci', xy_vals, scale)
    else:
        xye_yerr = None
        xy_yerr = None
    
    # === Define section groupings with spacing ===
    # Sections: "Before CST" (first), "Prompting Baselines" (middle), "After CST" (last)
    # labels are: ['Before CST', '0-shot CoT', '5-shot', '5-shot CoT', 'After CST']
    section_definitions = [
        ('Before CST', [0]),  # Before CST
        ('Prompting Baselines', list(range(1, n_conditions - 1))),  # Middle conditions
        ('After CST', [n_conditions - 1]),  # After CST
    ]
    
    # Build x positions with gaps between sections
    gap = 0.6  # Gap size between sections
    x_pos = []
    section_boundaries = []  # x positions for vertical dividers
    section_ranges = []  # (start_x, end_x, label) for group labels
    
    current_x = 0
    for sec_idx, (sec_name, indices) in enumerate(section_definitions):
        if len(indices) == 0:
            continue
        
        sec_start_x = current_x
        for i, orig_idx in enumerate(indices):
            x_pos.append((orig_idx, current_x))  # Map original index to new x position
            current_x += 1
        sec_end_x = current_x - 1
        
        section_ranges.append((sec_start_x, sec_end_x, sec_name))
        
        # Add boundary and gap after this section (except last)
        if sec_idx < len(section_definitions) - 1:
            section_boundaries.append(current_x - 0.5 + gap / 2)
            current_x += gap
    
    # Convert x_pos to array maintaining original order
    x_pos_dict = {orig_idx: new_x for orig_idx, new_x in x_pos}
    x_pos = np.array([x_pos_dict[i] for i in range(n_conditions)])
    
    # Plot bars with error bars
    bars_xy = ax.bar(x_pos - bar_width/2, xy_vals, bar_width, color=colors['xy'], 
                     label='Outcome-only', alpha=0.9, edgecolor='#333333', linewidth=1,
                     yerr=xy_yerr, capsize=4 if any_has_ci else 0,
                     error_kw={'linewidth': 1.5, 'alpha': 0.7} if any_has_ci else {})
    bars_xye = ax.bar(x_pos + bar_width/2, xye_vals, bar_width, color=colors['xye'], 
                      label='Reasoning', alpha=0.9, edgecolor='#333333', linewidth=1,
                      yerr=xye_yerr, capsize=4 if any_has_ci else 0,
                      error_kw={'linewidth': 1.5, 'alpha': 0.7} if any_has_ci else {})
    
    # Add vertical dividers between sections (gray lines like plot_cue_influence_clustered)
    for boundary in section_boundaries:
        ax.axvline(x=boundary, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)
    
    # Add value annotations on bars (tight to CIs)
    for i, (bar_xy, bar_xye) in enumerate(zip(bars_xy, bars_xye)):
        xy_top = xy_vals[i] + (xy_yerr[1][i] if xy_yerr else 0)
        xye_top = xye_vals[i] + (xye_yerr[1][i] if xye_yerr else 0)
        
        ax.annotate(f'{xy_vals[i]:.1f}', 
                    xy=(bar_xy.get_x() + bar_xy.get_width() / 2, xy_top),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, 
                    # fontweight='bold', 
                    alpha=0.8)
        ax.annotate(f'{xye_vals[i]:.1f}', 
                    xy=(bar_xye.get_x() + bar_xye.get_width() / 2, xye_top),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, 
                    # fontweight='bold', 
                    alpha=0.8)
    
    # Delta annotations (tight to value annotations)
    for i in range(n_conditions):
        delta = deltas[i]
        delta_sign = '+' if delta > 0 else ''
        
        # Position above the higher bar + CI + value annotation
        xy_top = xy_vals[i] + (xy_yerr[1][i] if xy_yerr else 0)
        xye_top = xye_vals[i] + (xye_yerr[1][i] if xye_yerr else 0)
        max_height = max(xy_top, xye_top)
        
        delta_color = '#2e7d32' if delta > 0 else '#c62828'
        
        # Build annotation text (no CI)
        annotation_text = f'Δ={delta_sign}{delta:.1f}'
        
        ax.annotate(
            annotation_text,
            xy=(x_pos[i], max_height),
            xytext=(0, 18),
            textcoords="offset points",
            ha='center', va='bottom',
            fontsize=11, fontweight='bold',
            color=delta_color,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=delta_color, alpha=0.9),
            linespacing=0.9,
        )
    
    # Styling
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=12, fontweight='bold', rotation=25, ha='right')
    ax.set_title(config['title'], fontsize=21, fontweight='bold', pad=10)
    
    # Add section name labels above y=100 line
    y_label_pos = 107  # Position above the 100% line
    for start_x, end_x, sec_name in section_ranges:
        center_x = (start_x + end_x) / 2
        ax.text(center_x, y_label_pos, sec_name, ha='center', va='bottom', 
                fontsize=12, fontweight='bold', fontstyle='italic',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gray'))
    
    # Y-axis label only for first column of each row
    if facet_idx % ncols == 0:
        ylabel = config.get('ylabel', 'Accuracy')
        ax.set_ylabel(f"{ylabel} (%)", fontsize=18, fontweight='bold', labelpad=2)
    
    # Y-limits - extend to 115 to make room for delta annotations
    ylim = config.get('ylim', (0.4, 1.0))
    ax.set_ylim(ylim[0] * scale, 115)
    
    # Add horizontal line at 100% to mark the boundary (same style as vertical dividers)
    ax.axhline(y=100, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)
    
    # Background and spines
    ax.set_facecolor(PLOT_BACKGROUND_COLOR)
    ax.spines['top'].set_color('#ffffff')
    ax.spines['right'].set_color('#ffffff')
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.tick_params(axis='both', length=3, pad=2)

# Shared legend below
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, ("Outcome-only Monitor", "Reasoning Monitor"),
    frameon=False, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.13),
    ncol=2,
    prop={"size": 17, "weight": 'bold'},
)

fig.suptitle("CST Outperforms Prompting Baselines", fontsize=26, fontweight='bold', y=1.06)
fig.align_ylabels()

# Save figure
plt.savefig(f'zfigs/prompting_baselines_barplot_facet_{cf_filter_str}.pdf', dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved to zfigs/prompting_baselines_barplot_facet_{cf_filter_str}.pdf")

# Plot SL vs RL

In [ ]:
cf_types = ID_CF_TYPES 
# cf_types = OOD_CF_TYPES

# Example: Compare SL vs RL experiments
# qwen CoT-F
experiment_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json",
        "label": "SL",
    },
    {
        "args_path": "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_8samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        "label": "RL",  
    },
]
metric = "all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye"

# Quick plot without bootstrap (fast iteration, no CIs)
data = plot_experiment_comparison(
    experiment_configs=experiment_configs,
    metric=metric,
    x_axis="elapsed_time",  # or "round"
    figsize=(6, 5),
    ylim=(0.1, 0.9),
    n_seeds=3,
    title="Qwen CoT-F Monitor G-Mean over Time",
    ylabel="G-Mean",
    show_values=True,
    print_pvalues=True,
    # cf_type_filter='all_ID',
    ci_style='ribbon+lines',
    cf_type_filter=cf_types,
    run_bootstrap=False,    # Set True for final plots with proper CIs
    # save_name="sl_vs_rl_comparison",
)

# Example: Compare SL vs RL experiments
# deepseek CoT-F
experiment_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "SL",
    },
    {
        "args_path": "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_8samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "RL",  
    },
]
metric = "all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye"

# Quick plot without bootstrap (fast iteration, no CIs)
data = plot_experiment_comparison(
    experiment_configs=experiment_configs,
    metric=metric,
    x_axis="elapsed_time",  # or "round"
    figsize=(6, 5),
    ylim=(0.1, 0.9),
    n_seeds=3,
    title="Deepseek CoT-F Monitor G-Mean over Time",
    ylabel="G-Mean",
    show_values=True,
    print_pvalues=True,
    # cf_type_filter='all_ID',
    ci_style='ribbon+lines',
    cf_type_filter=cf_types,
    run_bootstrap=False,    # Set True for final plots with proper CIs
    # save_name="sl_vs_rl_comparison",
)

# Example: Compare SL vs RL experiments
# deepseek CoT-T
experiment_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "SL",
    },
    {
        "args_path": "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_8samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "RL",  
    },
]
metric = "all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye"

data = plot_experiment_comparison(
    experiment_configs=experiment_configs,
    metric=metric,
    x_axis="elapsed_time",  # or "round"
    figsize=(6, 5),
    ylim=(0.1, 1),
    n_seeds=5,
    title="CoT Rewriting is 5x More Efficient Than RL",
    ylabel="G-Mean",
    show_values=True,
    print_pvalues=True,
    # cf_type_filter='all_ID',
    ci_style='ribbon+lines',
    cf_type_filter=cf_types,
    run_bootstrap=False,    # Set True for final plots with proper CIs
    custom_efficiency_annotation=True,
    # save_name="sl_vs_rl_comparison",
)

In [ ]:
# FACET WRAP EXPERIMENT COMPARISON

# Example: Compare SL vs RL experiments
# deepseek CoT-T
experiment_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "CoT Rewriting",
    },
    {
        "args_path": "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_8samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "RL",  
    },
]
metric = "all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye"

data = plot_experiment_comparison(
    experiment_configs=experiment_configs,
    metric=metric,
    x_axis="elapsed_time",  # or "round"
    figsize=(6, 5),
    ylim=(0.1, 1),
    n_seeds=5,
    title="CoT Rewriting is 5x More Efficient vs. RL",
    ylabel="G-Mean",
    show_values=True,
    print_pvalues=True,
    # cf_type_filter='all_ID',
    ci_style='ribbon+lines',
    cf_type_filter=ID_CF_TYPES,
    run_bootstrap=False,    # Set True for final plots with proper CIs
    custom_efficiency_annotation=True,
    # save_name="sl_vs_rl_comparison",
)

data = plot_experiment_comparison(
    experiment_configs=experiment_configs,
    metric=metric,
    x_axis="elapsed_time",  # or "round"
    figsize=(6, 5),
    ylim=(0.1, 1),
    n_seeds=5,
    title="CoT Rewriting Generalizes Better OOD vs. RL",
    ylabel="G-Mean",
    show_values=True,
    print_pvalues=True,
    # cf_type_filter='all_ID',
    ci_style='ribbon+lines',
    cf_type_filter=OOD_CF_TYPES,
    run_bootstrap=False,    # Set True for final plots with proper CIs
    custom_efficiency_annotation=False,
    # save_name="sl_vs_rl_comparison",
)

In [ ]:
def plot_experiment_comparison_facets(
    facet_configs: list,
    experiment_configs: list,
    metric: str | list[str],
    x_axis: str = "round",
    run_bootstrap: bool = False,
    n_bootstrap: int = 1000,
    n_seeds: int = 3,
    figsize: tuple = None,
    ylim: tuple = None,
    ylabel: str = None,
    xlabel: str = None,
    show_values: bool = False,
    value_decimals: int = None,
    ci_alpha: float = 0.2,
    ci_style: str = 'ribbon',
    lines_only: bool = False,
    scale: float = 100,
    buffer: float = 0.5,
    force_rerun: bool = False,
    ncols: int = 2,
    suptitle: str = None,
    save_name: str = None,
):
    """
    Create a faceted grid of experiment comparison plots.
    
    Args:
        facet_configs: List of dicts, each with:
            - cf_type_filter: The cf type filter for this facet
            - title: Title for this facet
            - metric: str or list[str] (optional, overrides global metric for this facet)
            - custom_efficiency_annotation: bool (optional)
            - ylim: tuple (optional, overrides global ylim)
        experiment_configs: List of experiment configs (shared across all facets)
        metric: Metric name(s) (default for all facets, can be overridden per-facet)
        ncols: Number of columns in the grid
        suptitle: Overall figure title
        ... other args passed to each facet
    
    Returns:
        Figure and axes
    """
    n_facets = len(facet_configs)
    nrows = (n_facets + ncols - 1) // ncols
    
    if figsize is None:
        figsize = (6 * ncols, 5 * nrows)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    axes = axes.flatten()
    
    # Normalize metric to a list
    if isinstance(metric, str):
        metrics_list = [metric] * len(experiment_configs)
    else:
        metrics_list = metric
    
    # Custom palette
    palette = [
        "#6b7fa3",  # muted blue
        "#a67fb8",  # muted purple
        "#7fb88a",  # muted green
        "#5fa89d",  # muted teal
        "#d9844f",  # muted orange
    ]
    
    for facet_idx, facet_config in enumerate(facet_configs):
        ax = axes[facet_idx]
        ax.set_facecolor(PLOT_BACKGROUND_COLOR)
        
        cf_type_filter = facet_config.get('cf_type_filter')
        facet_title = facet_config.get('title', '')
        custom_efficiency_annotation = facet_config.get('custom_efficiency_annotation', False)
        facet_ylim = facet_config.get('ylim', ylim)
        facet_ylabel = facet_config.get('ylabel', ylabel)
        facet_scale = facet_config.get('scale', scale)  # Per-facet scale (default to global)
        facet_x_axis = facet_config.get('x_axis', x_axis)  # Per-facet x_axis (default to global)
        
        # Get facet-specific metric (or use global)
        facet_metric = facet_config.get('metric', metric)
        if isinstance(facet_metric, str):
            facet_metrics_list = [facet_metric] * len(experiment_configs)
        else:
            facet_metrics_list = facet_metric
        
        plot_data = []
        
        for i, config in enumerate(experiment_configs):
            args_path = config['args_path']
            label = config.get('label', args_path[:40])
            exp_metric = facet_metrics_list[i]
            
            # Load args
            with open(f"results/{args_path}", "r") as f:
                exp_args = json.load(f)
                exp_args = SimpleNamespace(**exp_args)
            
            if run_bootstrap:
                rounds_to_compute = list(range(exp_args.train_rounds + 1))
                metrics_hash = hashlib.md5(exp_metric.encode()).hexdigest()[:8]
                cf_filter_str = "ID" if cf_type_filter == ID_CF_TYPES else "OOD" if cf_type_filter == OOD_CF_TYPES else "all"
                cache_path = f"cache/bootstrap_{utils.get_base_exp_name(exp_args)}_{n_seeds}seeds_{cf_filter_str}_{metrics_hash}.pkl"
                
                results_long, results_wide, bootstrap_samples = bootstrap_evaluation_stats(
                    args=exp_args,
                    n_seeds=n_seeds,
                    rounds=rounds_to_compute,
                    simulator_sweep_configs=simulator_sweep_configs,
                    metrics_to_compute=[exp_metric],
                    cf_type_filter=cf_type_filter,
                    n_bootstrap=n_bootstrap,
                    confidence_level=0.95,
                    random_state=42,
                    block_col='id',
                    verbose=False,
                    cache_path=cache_path,
                    force_rerun=force_rerun,
                    print_pvalues=False,
                )
                
                rounds = results_wide['round'].values
                y_vals = results_wide[f'{exp_metric}_test'].values
                ci_low = results_wide[f'{exp_metric}_test_ci_low'].values 
                ci_high = results_wide[f'{exp_metric}_test_ci_high'].values
                if facet_x_axis == "elapsed_time":
                    stats_df = load_stats_across_seeds(
                        base_exp_name=utils.get_base_exp_name(exp_args),
                        n_seeds=n_seeds,
                    )
                    stats_df.loc[stats_df['round'] == 0, 'elapsed_time'] = 0.0
                    x_vals = stats_df.groupby('round')['elapsed_time'].mean().reindex(rounds).values
                else:
                    x_vals = rounds
                
                for j, (x, y) in enumerate(zip(x_vals, y_vals)):
                    entry = {
                        'X': x,
                        'Metric Value': y * facet_scale,
                        'Experiment': label,
                    }
                    if ci_low is not None and ci_high is not None:
                        entry['ci_low'] = ci_low[j] * facet_scale
                        entry['ci_high'] = ci_high[j] * facet_scale
                    plot_data.append(entry)
            else:
                stats_df = load_stats_across_seeds(
                    base_exp_name=utils.get_base_exp_name(exp_args),
                    n_seeds=n_seeds,
                )
                stats_df = add_gmean_from_gmean2(stats_df)
                stats_df.loc[stats_df['round'] == 0, 'elapsed_time'] = 0.0
                
                if cf_type_filter is not None:
                    if isinstance(cf_type_filter, list):
                        if cf_type_filter == ID_CF_TYPES:
                            cf_filter_resolved = 'all_ID'
                        elif cf_type_filter == OOD_CF_TYPES:
                            cf_filter_resolved = 'all_OOD'
                        else:
                            cf_filter_resolved = 'all_ID'
                        stats_df = stats_df[stats_df['counterfactual_type'] == cf_filter_resolved]
                    else:
                        stats_df = stats_df[stats_df['counterfactual_type'] == cf_type_filter]
                else:
                    stats_df = stats_df[stats_df['counterfactual_type'] == 'all_ID']
                if len(stats_df) == 0:
                    continue
                
                metric_col = f'{exp_metric}_test' if f'{exp_metric}_test' in stats_df.columns else exp_metric
                if metric_col not in stats_df.columns:
                    continue
                
                grouped = stats_df.groupby('round').agg({
                    metric_col: 'mean',
                    'elapsed_time': 'mean',
                }).reset_index()
                if facet_x_axis == "elapsed_time":
                    x_vals = grouped['elapsed_time'].values
                else:
                    x_vals = grouped['round'].values
                
                y_vals = grouped[metric_col].values
                
                for x, y in zip(x_vals, y_vals):
                    plot_data.append({
                        'X': x,
                        'Metric Value': y * facet_scale,
                        'Experiment': label,
                    })
        exp_order = [config.get('label', config['args_path'][:40]) for config in experiment_configs]
        plot_df = pd.DataFrame(plot_data)
        
        if plot_df.empty:
            continue
        
        exp_order = [config.get('label', config['args_path'][:40]) for config in experiment_configs]
        
        # Draw CIs
        if run_bootstrap and 'ci_low' in plot_df.columns:
            for i, exp_label in enumerate(exp_order):
                exp_df = plot_df[plot_df['Experiment'] == exp_label].sort_values('X')
                color = palette[i % len(palette)]
                
                if ci_style in ['ribbon', 'ribbon+lines']:
                    ax.fill_between(
                        exp_df['X'], exp_df['ci_low'], exp_df['ci_high'],
                        alpha=ci_alpha, color=color, linewidth=0,
                    )
                if ci_style in ['lines', 'ribbon+lines']:
                    ax.plot(exp_df['X'], exp_df['ci_low'], linestyle='--', linewidth=1.5, color=color, alpha=0.4)
                    ax.plot(exp_df['X'], exp_df['ci_high'], linestyle='--', linewidth=1.5, color=color, alpha=0.4)
        
        # Main plot
        marker = None if lines_only else 'o'
        sns.lineplot(
            data=plot_df, x='X', y='Metric Value', hue='Experiment',
            marker=marker, linewidth=3.0, markersize=7,
            palette=palette[:len(exp_order)], ax=ax,
            hue_order=exp_order,
        )
        
        # Custom efficiency annotation
        if custom_efficiency_annotation:
            y_val = 0.788 * scale
            x_start, x_end = 1.95, 10.65
            ax.plot([x_start, x_end], [y_val, y_val], 
                    color='black', linestyle=(0, (8, 4)), linewidth=2, alpha=0.7)
            x_mid = (x_start + x_end) / 2
            ax.text(x_mid, y_val + 1.5, 'Equal Performance', 
                    ha='center', va='bottom', fontsize=10, color='black', fontstyle='italic')
        
        # Styling
        ax.spines['top'].set_color('#ffffff')
        ax.spines['right'].set_color('#ffffff')
        ax.spines['left'].set_color('#333333')
        ax.spines['bottom'].set_color('#333333')
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        facet_xlabel = xlabel if xlabel else ("Elapsed Time (hours)" if facet_x_axis == "elapsed_time" else "Round")
        ax.set_xlabel(facet_xlabel, fontsize=19, fontweight='bold')
        ax.set_ylabel(facet_ylabel if facet_ylabel else "Metric Value", fontsize=19, fontweight='bold')
        if facet_x_axis == "round":
            x_vals_all = plot_df['X'].unique()
            ax.set_xticks(sorted(x_vals_all))
            ax.set_xticklabels([int(r) for r in sorted(x_vals_all)])
        
        # Increase tick label sizes
        ax.tick_params(axis='x', labelsize=16)
        ax.tick_params(axis='y', labelsize=15)
        
        if facet_title:
            ax.set_title(facet_title, fontsize=22, fontweight='bold', pad=10)
        
        if facet_ylim is not None:
            ax.set_ylim(facet_ylim[0] * facet_scale, facet_ylim[1] * facet_scale)
        # Show values
        if show_values:
            for _, row in plot_df.iterrows():
                val_str = f"{row['Metric Value']:.{value_decimals}f}" if value_decimals else f"{int(row['Metric Value'])}"
                ax.annotate(val_str, 
                           xy=(row['X'], row['Metric Value']),
                           xytext=(0, 6), textcoords='offset points',
                           ha='center', va='bottom', fontsize=13, alpha=0.7)
        
        # Remove legend from individual plots
        if ax.get_legend():
            ax.get_legend().remove()
    
    # Hide extra axes
    for idx in range(n_facets, len(axes)):
        axes[idx].set_visible(False)
    
    # Create shared legend at bottom
    # Center legend relative to visible plots, not full figure
    from matplotlib.lines import Line2D
    exp_order = [config.get('label', config['args_path'][:40]) for config in experiment_configs]
    custom_handles = [
        Line2D([0], [0], color=palette[i % len(palette)], linestyle='-', linewidth=3, 
               marker='o' if not lines_only else None, markersize=7)
        for i in range(len(exp_order))
    ]
    # Compute center x position based on visible plots
    n_cols_used = min(n_facets, ncols)
    legend_x = n_cols_used / (2 * ncols)  # Center of visible columns
    legend_prop = {'size': 20, 'weight': 'bold'}
    fig.legend(custom_handles, exp_order, frameon=False, prop=legend_prop,
              loc='upper center', bbox_to_anchor=(legend_x, 0.02), ncol=len(exp_order))
    
    if suptitle:
        fig.suptitle(suptitle, fontsize=30, fontweight='bold', x=legend_x, y=0.99)
    
    plt.tight_layout()
    plt.subplots_adjust(wspace=0.3)  # Add horizontal space between subplots
    
    if save_name:
        plt.savefig(f"zfigs/{save_name}.pdf", bbox_inches='tight', dpi=300)
        print(f"Saved to zfigs/{save_name}.pdf")
    
    plt.show()
    return fig, axes

# Example usage:
experiment_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "CoT Rewriting",
    },
    {
        "args_path": "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_8samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "RL",  
    },
]

facet_configs = [
    {
        "cf_type_filter": ID_CF_TYPES,
        "title": "5x More Efficient",
        "custom_efficiency_annotation": True,
        "ylabel": "G-Mean (In-Distribution)",
        "xlabel": "Elapsed Time (hours)",
        "metric": "all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye",
        "x_axis": "elapsed_time",
        "ylim": (0, 1),
    },
    {
        "cf_type_filter": OOD_CF_TYPES,
        "title": "Better OOD Generalization",
        "ylabel": "G-Mean (OOD)",
        "xlabel": "Elapsed Time (hours)",
        "custom_efficiency_annotation": False,
        "metric": "all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye",
        "x_axis": "elapsed_time",
        "ylim": (0, 1),
    },
    {
        "cf_type_filter": ID_CF_TYPES,
        "title": "RL Increases CoT Length",
        "ylabel": "CoT Length (Words)",
        "metric": "avg_original_greedy_cot_words",
        "ylim": (100, 200),  # Auto scale
        "scale": 1,  # Don't multiply by 100
    },
    # {
    #     "cf_type_filter": ID_CF_TYPES,
    #     "title": "Bias Rate Is Constant",
    #     "ylabel": "Bias Rate (In-Distribution)",
    #     "xlabel": "Round",
    #     "custom_efficiency_annotation": False,
    #     "metric":  "all_greedy_bias_rate",
    #     "x_axis": "round",
    #     "ylim": (0, 1),
    # },
    # {
    #     "cf_type_filter": OOD_CF_TYPES,
    #     "title": "OOD Bias Rate",
    #     "ylabel": "OOD Bias Rate",
    #     "custom_efficiency_annotation": False,
    #     "metric": "all_greedy_bias_rate",
    #     "ylim": (0, 1),
    # },
]

plot_experiment_comparison_facets(
    facet_configs=facet_configs,
    experiment_configs=experiment_configs,
    # metric="all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye",
    metric="all_greedy_bias_rate",
    # x_axis="elapsed_time",
    ylim=(0.1, 1),
    n_seeds=5,
    # ylabel="G-Mean",
    show_values=True,
    ncols=4,
    run_bootstrap=False,
    save_name="sl_vs_rl_faceted",
    suptitle="Advantages of Process Supervision over RL",
)

# simulator ablation

In [ ]:
experiment_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "CoT Rewriting",
    },
    {
        "args_path": "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_8samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "RL",  
    },
]

facet_configs = [
    {
        "cf_type_filter": ID_CF_TYPES,
        "title": "CoT Rewriting is 5x More Efficient vs. RL",
        "custom_efficiency_annotation": True,
        "ylabel": "G-Mean (In-Distribution)",
        "xlabel": "Elapsed Time (hours)",
        "metric": "all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye",
        "x_axis": "elapsed_time",
        "ylim": (0, 1),
    },
    {
        "cf_type_filter": OOD_CF_TYPES,
        "title": "CoT Rewriting Generalizes Better OOD vs. RL",
        "ylabel": "G-Mean (OOD)",
        "xlabel": "Elapsed Time (hours)",
        "custom_efficiency_annotation": False,
        "metric": "all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye",
        "x_axis": "elapsed_time",
        "ylim": (0, 1),
    },
    {
        "cf_type_filter": ID_CF_TYPES,
        "title": "RL Alone Increases Answer Length",
        "ylabel": "CoT Length (Words)",
        "metric": "avg_original_greedy_cot_words",
        "ylim": (100, 200),  # Auto scale
        "scale": 1,  # Don't multiply by 100
    },
]

plot_experiment_comparison_facets(
    facet_configs=facet_configs,
    experiment_configs=experiment_configs,
    # metric="all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye",
    metric="all_greedy_bias_rate",
    # x_axis="elapsed_time",
    ylim=(0.1, 1),
    n_seeds=3,
    # ylabel="G-Mean",
    show_values=True,
    ncols=4,
    run_bootstrap=False,
    save_name="sl_vs_rl_faceted",
)

# Plot CST vs VFT

In [ ]:
experiment_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json",
        "label": "CST",
    },
    {
        "args_path": "args_VFT_mmlu-2way_6-cf-types_gpt-oss-120b_PFT_cfm-0.1_faith-FNs_k-0_CoT-T_yesno-xye_1rounds_e20_1samples_qwen3-235b-a22b_n1000-2000_sd0.json",
        "label": "VFT",
    },
]

# Choose your metric
metrics_to_reconcile = [
    "all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye",
    "all_bias_monitor_gmean_k-0_CoT-T_yesno-xye",
]

# Quick plot without bootstrap (fast iteration, no CIs)
data = plot_experiment_comparison(
    experiment_configs=experiment_configs,
    metric=metrics_to_reconcile,
    x_axis="round",
    figsize=(9, 6),
    ylim=(0.1, 0.9),
    n_seeds=5,
    title="Comparison: Monitor G-Mean over Time",
    ylabel="G-Mean",
    show_values=True,
    print_pvalues=True,
    # cf_type_filter='all_ID',
    ci_style='ribbon+lines',
    # cf_type_filter=ID_CF_TYPES,
    cf_type_filter=OOD_CF_TYPES,
    run_bootstrap=False,    # Set True for final plots with proper CIs
    # save_name="sl_vs_rl_comparison",
)

In [ ]:
def plot_experiment_comparison_bars(
    experiment_configs: list[dict],
    cf_type_filter=None,
    metrics_to_plot: list[str] = ['recall', 'FPR', 'gmean', 'bias_rate'],
    n_bootstrap: int = 10,
    n_seeds: int = 5,
    confidence_level: float = 0.95,
    force_rerun: bool = False,
    title: str = None,
    figsize: tuple = (10, 6),
    save_path: str = None,
    show_annotations: bool = True,
    ylim: tuple = (0, 105),
    bold_after_gmean: bool = False,
):
    """
    Compare multiple experiments on monitor metrics (Recall, FPR, G-Mean).
    
    Each experiment_config should be a dict with:
        - 'args_path': str, path to args JSON file (relative to results/)
        - 'name': str, display name for the experiment (e.g., 'CST', 'VFT')
        - 'metrics_prefix': str, prefix for metrics (e.g., 'all_bias_monitor')
        - 'metrics_suffix': str, suffix for metrics (e.g., 'k-0_CoT-F_cf-sim-xye')
        - 'simulator_config': dict, simulator config for bootstrap
        
    Example:
        experiment_configs = [
            {
                'args_path': 'args_cue_pivot_mmlu-2way_...json',
                'name': 'CST',
                'metrics_prefix': 'all_bias_monitor',
                'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
                'simulator_config': {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
                                     'condition_on_original_answers': True, 'condition_on_explanations': True},
            },
            ...
        ]
    """
    palette = {
        'recall': "#6a9fb5",  # soft blue
        'FPR': "#c97b84",     # dusty rose
        'gmean': "#708090",   # slate gray
        'bias_rate': "#d9a066", # warm gold
    }
    
    metric_labels = {
        'recall': 'Recall (↑)',
        'FPR': 'FPR (↓)',
        'gmean': 'G-Mean (↑)',
        'bias_rate': 'Cue Influence Rate',
    }
    
    # Determine cf_type cache string
    cache_id = "ID" if cf_type_filter == ID_CF_TYPES else "OOD" if cf_type_filter == OOD_CF_TYPES else "all"
    
    # Process each experiment
    all_results = []
    
    for exp_config in experiment_configs:
        args_path = exp_config['args_path']
        exp_name = exp_config['name']
        metrics_prefix = exp_config['metrics_prefix']
        metrics_suffix = exp_config['metrics_suffix']
        simulator_config = exp_config['simulator_config']
        
        # Load args
        with open(f"results/{args_path}", "r") as f:
            args = json.load(f)
            args = SimpleNamespace(**args)
        
        last_round = int(args.train_rounds)
        print(f"{exp_name} rounds: 0 -> {last_round}")
        
        # Build metric names - handle bias_rate specially (uses different naming pattern)
        metrics_to_compute = []
        for m in metrics_to_plot:
            if m == 'bias_rate':
                metrics_to_compute.append('all_greedy_bias_rate')
            else:
                metrics_to_compute.append(f"{metrics_prefix}_{m}_{metrics_suffix}")
        
        # Run bootstrap
        cache_path = f"cache/bootstrap_barplot_{exp_name.lower()}_{utils.get_base_exp_name(args)}_{n_seeds}seeds_{cache_id}.pkl"
        results_long, results_wide, bootstrap_samples, true_means = bootstrap_evaluation_stats(
            args=args,
            n_seeds=n_seeds,
            rounds=[0, last_round],
            simulator_sweep_configs=[simulator_config],
            metrics_to_compute=metrics_to_compute,
            cf_type_filter=cf_type_filter,
            n_bootstrap=n_bootstrap,
            confidence_level=confidence_level,
            random_state=42,
            block_col='id',
            verbose=True,
            cache_path=cache_path,
            force_rerun=force_rerun,
            print_pvalues=True,
        )
        
        # Get available rounds from the results
        available_rounds = results_wide['round'].unique().tolist()
        rounds_to_extract = [r for r in [0, last_round] if r in available_rounds]
        
        if len(rounds_to_extract) == 0:
            print(f"Warning: No data available for {exp_name}, skipping")
            continue
        
        # Extract values for available rounds
        for round_idx, round_num in enumerate(rounds_to_extract):
            row = results_wide[results_wide['round'] == round_num].iloc[0]
            round_label = "Before" if round_num == 0 else "After"
            
            result_entry = {
                'experiment': exp_name,
                'round': round_num,
                'group_label': f"{round_label} {exp_name}",
            }
            
            for metric in metrics_to_plot:
                if metric == 'bias_rate':
                    col = 'all_greedy_bias_rate_test'
                else:
                    col = f"{metrics_prefix}_{metric}_{metrics_suffix}_test"
                # Skip if column doesn't exist in results
                if col not in row.index:
                    print(f"Warning: Column {col} not found in results, skipping metric {metric}")
                    continue
                result_entry[metric] = row[col]
                result_entry[f'{metric}_ci_low'] = row[col] - row[f"{col}_ci_low"]
                result_entry[f'{metric}_ci_high'] = row[f"{col}_ci_high"] - row[col]
            
            all_results.append(result_entry)
    
    # Create plot
    scale = 100
    fig, ax = plt.subplots(figsize=figsize)
    
    # Calculate x positions with gaps between experiment groups
    # Group results by experiment name
    experiments_seen = []
    for r in all_results:
        if r['experiment'] not in experiments_seen:
            experiments_seen.append(r['experiment'])
    
    # Create x positions with gaps between groups
    x = []
    gap = 0.4  # Gap between experiment groups
    current_x = 0
    prev_experiment = None
    group_boundaries = []  # Track where to put dividers
    experiment_centers = {}  # Track center position for each experiment's bars
    
    for r in all_results:
        if prev_experiment is not None and r['experiment'] != prev_experiment:
            group_boundaries.append(current_x - 0.5 + gap / 2)  # Midpoint of gap
            current_x += gap  # Add gap when switching to new experiment
        
        # Track positions for each experiment
        if r['experiment'] not in experiment_centers:
            experiment_centers[r['experiment']] = []
        experiment_centers[r['experiment']].append(current_x)
        
        x.append(current_x)
        current_x += 1
        prev_experiment = r['experiment']
    x = np.array(x)
    
    # Dynamically adjust bar width and spacing based on number of metrics
    n_metrics = len(metrics_to_plot)
    if n_metrics <= 3:
        width = 0.25
        within_group_spacing = 1.0
    else:
        # For 4+ metrics, use narrower bars and more spacing
        width = 0.20
        within_group_spacing = 1.3  # More space between Before/After
    
    # Recompute x positions with dynamic spacing
    x = []
    current_x = 0
    prev_experiment = None
    group_boundaries = []
    experiment_centers = {}
    
    for r in all_results:
        if prev_experiment is not None and r['experiment'] != prev_experiment:
            group_boundaries.append(current_x - within_group_spacing/2 + gap / 2)
            current_x += gap
        
        if r['experiment'] not in experiment_centers:
            experiment_centers[r['experiment']] = []
        experiment_centers[r['experiment']].append(current_x)
        
        x.append(current_x)
        current_x += within_group_spacing
        prev_experiment = r['experiment']
    x = np.array(x)
    
    # Position bars symmetrically around x, with extra gap before bias_rate
    offsets = np.linspace(-(n_metrics-1)/2 * width, (n_metrics-1)/2 * width, n_metrics)
    
    # Add extra offset for bias_rate (if present) to create visual separation
    bias_rate_extra_gap = 0.08  # Extra gap before bias_rate
    if 'bias_rate' in metrics_to_plot:
        bias_rate_idx = metrics_to_plot.index('bias_rate')
        for i in range(bias_rate_idx, len(offsets)):
            offsets[i] += bias_rate_extra_gap
    
    bars_list = []
    for i, metric in enumerate(metrics_to_plot):
        values = [r[metric] * scale for r in all_results]
        cis = np.array([[r[f'{metric}_ci_low'] * scale, r[f'{metric}_ci_high'] * scale] for r in all_results]).T
        
        # Use hatching for bias_rate to visually distinguish it
        hatch = '///' if metric == 'bias_rate' else None
        edge_lw = 1.2 if metric == 'bias_rate' else 1.0
        edge_color = '#555555' if metric == 'bias_rate' else 'black'  # Gray hatch for bias_rate
        
        bars = ax.bar(x + offsets[i], values, width, label=metric_labels[metric],
                      color=palette[metric], edgecolor=edge_color, linewidth=edge_lw, hatch=hatch)
        bars_list.append(bars)
        
        # Error bars
        graycolor = "#565756"
        ax.errorbar(x + offsets[i], values, yerr=cis, fmt='none',
                    color=graycolor, capsize=4, capthick=1.5, elinewidth=1.5)
        
        # Annotations - position above CI upper bounds
        if show_annotations:
            blackcolor = "#2b2b2b"
            ci_uppers = cis[1]  # Upper CI values
            for j, bar in enumerate(bars):
                height = bar.get_height()
                ci_top = height + ci_uppers[j]
                # Check if this is G-Mean + After bar and bold flag is set
                is_after = all_results[j]['round'] != 0
                should_bold = bold_after_gmean and metric == 'gmean' and is_after
                fontweight = 'bold' if should_bold else 'normal'
                alpha = 1.0 if should_bold else 0.7
                ax.annotate(f'{int(round(height))}',
                            xy=(bar.get_x() + bar.get_width() / 2, ci_top),
                            xytext=(0, 3),
                            color=blackcolor,
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=10, fontweight=fontweight, alpha=alpha)
    
    # Add vertical dividers between experiment groups
    for boundary in group_boundaries:
        ax.axvline(x=boundary, color='gray', linestyle='-', linewidth=1.5, alpha=0.5)
    
    # Two-level x-axis labels: "Before CST"/"After CST" on top, model name below
    before_after_labels = []
    for r in all_results:
        label = "Before CST" if r['round'] == 0 else "After CST"
        before_after_labels.append(label)
    
    ax.set_xticks(x)
    ax.set_xticklabels(before_after_labels, fontsize=11, fontweight='bold')
    
    # Add model names as secondary labels below (using ax.text)
    for exp_name, positions in experiment_centers.items():
        center_x = np.mean(positions)
        ax.text(center_x, -0.12, exp_name, transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=12, fontweight='bold', color='#333333')
    
    # Styling
    ax.set_ylabel('Metric Value', fontsize=13, fontweight='bold')
    if title:
        ax.set_title(title, fontsize=15, fontweight='bold', pad=10)
    ax.set_ylim(ylim)
    
    if PLOT_BACKGROUND_COLOR:
        ax.set_facecolor(PLOT_BACKGROUND_COLOR)
    
    ax.legend(frameon=False, 
                loc='upper center', bbox_to_anchor=(0.5, -0.20),
                ncol=n_metrics,
                prop={'weight': 'bold', 'size': 12}
              )
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.24)  # Make room for two-level labels and legend
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    
    plt.show()
    
    return all_results

In [ ]:
def plot_metric_panels_bars(
    experiment_configs: list[dict],
    cf_type_filter=None,
    metrics_to_plot: list[str] = ['recall', 'FPR', 'gmean', 'bias_rate'],
    n_bootstrap: int = 100,
    n_seeds: int = 5,
    confidence_level: float = 0.95,
    force_rerun: bool = False,
    title: str = None,
    figsize: tuple = (16, 4),
    save_path: str = None,
    show_annotations: bool = True,
    show_delta_annotations: bool = True,
    ylim: tuple = (0, 100),
):
    """
    Plot metrics with one panel per metric (1x4 strip layout).
    
    Within each panel:
    - X-axis = experiments (model sizes)
    - Paired Before/After bars at each position (touching, with hatching for before)
    - Colors by experiment (model), hatching distinguishes before/after
    - Delta annotations with CIs shown in boxed annotations above bar pairs
    
    Args:
        experiment_configs: List of dicts with keys:
            - 'args_path': str, path to args JSON file (relative to results/)
            - 'name': str, display name for the experiment (e.g., 'Qwen3-4B')
            - 'metrics_prefix': str, prefix for metrics (e.g., 'all_bias_monitor')
            - 'metrics_suffix': str, suffix for metrics
            - 'simulator_config': dict, simulator config for bootstrap
            - 'cf_type_filter': (optional) per-experiment cf_type filter (overrides function-level)
        cf_type_filter: Default list of counterfactual types to include (can be overridden per-experiment)
        metrics_to_plot: List of metric names
        n_bootstrap: Number of bootstrap iterations
        n_seeds: Number of random seeds
        confidence_level: CI level (default 0.95)
        force_rerun: Force recompute even if cached
        title: Overall figure title
        figsize: Figure size (width, height)
        save_path: If provided, save figure to this path
        show_annotations: Show value annotations on bars
        show_delta_annotations: Show delta (Δ) annotations with CIs
        ylim: Y-axis limits (default 0-105 for percentage display)
    
    Returns:
        all_results: List of result dicts with metric values and CIs
        all_bootstrap_samples: Dict of {exp_name: {round: {metric: samples}}} for further analysis
    """
    
    # Metric colors - same color for all models within a panel
    metric_colors = {
        'recall': '#6a9fb5',    # soft blue
        'FPR': '#c97b84',       # dusty rose
        'gmean': '#708090',     # slate gray
        'bias_rate': '#d9a066', # warm gold
    }
    
    metric_labels = {
        'recall': 'Recall (↑)',
        'FPR': 'FPR (↓)',
        'gmean': 'G-Mean (↑)',
        'bias_rate': 'Cue Influence Rate',
    }
    
    # Helper to simplify model names (drop "Qwen3-" prefix only if ALL models share it)
    all_exp_names = [cfg['name'] for cfg in experiment_configs]
    all_have_qwen_prefix = all(n.startswith('Qwen3-') for n in all_exp_names)
    
    def simplify_name(name):
        if all_have_qwen_prefix and name.startswith('Qwen3-'):
            return name[6:]  # Remove "Qwen3-"
        return name
    
    # Process each experiment and collect bootstrap samples for delta CIs
    all_results = []
    all_bootstrap_samples = {}  # {exp_name: {round: {metric: samples}}}
    
    for exp_idx, exp_config in enumerate(experiment_configs):
        args_path = exp_config['args_path']
        exp_name = exp_config['name']
        metrics_prefix = exp_config['metrics_prefix']
        metrics_suffix = exp_config['metrics_suffix']
        simulator_config = exp_config['simulator_config']
        # Use per-experiment cf_type_filter if provided, otherwise use function-level
        exp_cf_type_filter = exp_config.get('cf_type_filter', cf_type_filter)
        
        # Determine cf_type cache string for this experiment
        cache_id = "ID" if exp_cf_type_filter == ID_CF_TYPES else "OOD" if exp_cf_type_filter == OOD_CF_TYPES else "all"
        
        # Load args
        with open(f"results/{args_path}", "r") as f:
            args = json.load(f)
            args = SimpleNamespace(**args)
        
        last_round = int(args.train_rounds)
        print(f"{exp_name} rounds: 0 -> {last_round}")
        
        # Build metric names
        metrics_to_compute = []
        for m in metrics_to_plot:
            if m == 'bias_rate':
                metrics_to_compute.append('all_greedy_bias_rate')
            else:
                metrics_to_compute.append(f"{metrics_prefix}_{m}_{metrics_suffix}")
        
        # Run bootstrap
        cache_path = f"cache/bootstrap_panels_{exp_name.lower()}_{utils.get_base_exp_name(args)}_{n_seeds}seeds_{cache_id}.pkl"
        results_long, results_wide, bootstrap_samples, true_means = bootstrap_evaluation_stats(
            args=args,
            n_seeds=n_seeds,
            rounds=[0, last_round],
            simulator_sweep_configs=[simulator_config],
            metrics_to_compute=metrics_to_compute,
            cf_type_filter=exp_cf_type_filter,
            n_bootstrap=n_bootstrap,
            confidence_level=confidence_level,
            random_state=42,
            block_col='id',
            verbose=True,
            cache_path=cache_path,
            force_rerun=force_rerun,
            print_pvalues=False,
        )
        
        # Store bootstrap samples for delta computation
        all_bootstrap_samples[exp_name] = bootstrap_samples
        
        available_rounds = results_wide['round'].unique().tolist()
        rounds_to_extract = [r for r in [0, last_round] if r in available_rounds]
        
        if len(rounds_to_extract) == 0:
            print(f"Warning: No data available for {exp_name}, skipping")
            continue
        
        # Extract values for available rounds
        for round_num in rounds_to_extract:
            row = results_wide[results_wide['round'] == round_num].iloc[0]
            round_label = "Before" if round_num == 0 else "After"
            
            result_entry = {
                'experiment': exp_name,
                'exp_idx': exp_idx,
                'round': round_num,
                'round_label': round_label,
                'last_round': last_round,
            }
            
            for metric in metrics_to_plot:
                if metric == 'bias_rate':
                    col = 'all_greedy_bias_rate_test'
                else:
                    col = f"{metrics_prefix}_{metric}_{metrics_suffix}_test"
                
                if col not in row.index:
                    print(f"Warning: Column {col} not found, skipping metric {metric}")
                    continue
                    
                result_entry[metric] = row[col]
                result_entry[f'{metric}_ci_low'] = row[col] - row[f"{col}_ci_low"]
                result_entry[f'{metric}_ci_high'] = row[f"{col}_ci_high"] - row[col]
            
            all_results.append(result_entry)
    
    # Compute deltas with CIs from bootstrap samples
    delta_results = {}  # {(exp_name, metric): {'delta': float, 'ci_low': float, 'ci_high': float}}
    
    for exp_config in experiment_configs:
        exp_name = exp_config['name']
        metrics_prefix = exp_config['metrics_prefix']
        metrics_suffix = exp_config['metrics_suffix']
        
        if exp_name not in all_bootstrap_samples:
            continue
            
        bootstrap_samples = all_bootstrap_samples[exp_name]
        
        # Find the last round for this experiment
        exp_results = [r for r in all_results if r['experiment'] == exp_name]
        if len(exp_results) < 2:
            continue
        last_round = exp_results[0]['last_round']
        
        if 0 not in bootstrap_samples or last_round not in bootstrap_samples:
            continue
        
        for metric in metrics_to_plot:
            if metric == 'bias_rate':
                metric_key = 'all_greedy_bias_rate'
            else:
                metric_key = f"{metrics_prefix}_{metric}_{metrics_suffix}"
            
            if metric_key not in bootstrap_samples[0] or metric_key not in bootstrap_samples[last_round]:
                continue
            
            samples_0 = bootstrap_samples[0][metric_key]
            samples_n = bootstrap_samples[last_round][metric_key]
            
            # Paired differences
            min_len = min(len(samples_0), len(samples_n))
            deltas = samples_n[:min_len] - samples_0[:min_len]
            
            alpha = 1 - confidence_level
            delta_results[(exp_name, metric)] = {
                'delta': np.mean(deltas),
                'ci_low': np.percentile(deltas, 100 * alpha / 2),
                'ci_high': np.percentile(deltas, 100 * (1 - alpha / 2)),
            }
    
    # Create figure with 1xN panels
    n_metrics = len(metrics_to_plot)
    fig, axes = plt.subplots(1, n_metrics, figsize=figsize, sharey=True)
    if n_metrics == 1:
        axes = [axes]
    
    scale = 100
    n_experiments = len(experiment_configs)
    bar_width = 0.35
    gap_between_models = 0.3  # Space between model groups
    
    for panel_idx, metric in enumerate(metrics_to_plot):
        ax = axes[panel_idx]
        
        if PLOT_BACKGROUND_COLOR:
            ax.set_facecolor(PLOT_BACKGROUND_COLOR)
        
        # Get experiments in order
        exp_names = [cfg['name'] for cfg in experiment_configs]
        
        # X positions: bars within model touch, gaps between models
        # Each model takes 2*bar_width, plus gap_between_models after each
        x_positions = np.arange(n_experiments) * (2 * bar_width + gap_between_models)
        
        for exp_idx, exp_name in enumerate(exp_names):
            # Get before/after results for this experiment
            before_result = next((r for r in all_results 
                                   if r['experiment'] == exp_name and r['round'] == 0), None)
            after_result = next((r for r in all_results 
                                  if r['experiment'] == exp_name and r['round'] != 0), None)
            
            if before_result is None or after_result is None:
                continue
            
            # Color for this metric (same for all models, hatching distinguishes before/after)
            color = metric_colors.get(metric, '#888888')
            
            # Bar positions: touching within model group
            x_before = x_positions[exp_idx]
            x_after = x_positions[exp_idx] + bar_width
            
            # Get metric values
            val_before = before_result.get(metric, 0) * scale
            val_after = after_result.get(metric, 0) * scale
            ci_before = [before_result.get(f'{metric}_ci_low', 0) * scale, 
                         before_result.get(f'{metric}_ci_high', 0) * scale]
            ci_after = [after_result.get(f'{metric}_ci_low', 0) * scale,
                        after_result.get(f'{metric}_ci_high', 0) * scale]
            
            # Plot before bar (with hatching)
            ax.bar(x_before, val_before, bar_width, color=color, 
                   edgecolor='black', linewidth=0.8, hatch='///', alpha=0.7,
                   label='Before' if (exp_idx == 0 and panel_idx == 0) else None)
            ax.errorbar(x_before, val_before, yerr=[[ci_before[0]], [ci_before[1]]], 
                        fmt='none', color='#565756', capsize=3, capthick=1.2, elinewidth=1.2)
            
            # Plot after bar (solid, no hatching)
            ax.bar(x_after, val_after, bar_width, color=color,
                   edgecolor='black', linewidth=0.8,
                   label='After' if (exp_idx == 0 and panel_idx == 0) else None)
            ax.errorbar(x_after, val_after, yerr=[[ci_after[0]], [ci_after[1]]],
                        fmt='none', color='#565756', capsize=3, capthick=1.2, elinewidth=1.2)
            
            # Value annotations
            if show_annotations:
                ax.annotate(f'{int(round(val_before))}', 
                            xy=(x_before, val_before + ci_before[1]),
                            xytext=(0, 3), textcoords='offset points',
                            ha='center', va='bottom', fontsize=13, alpha=0.7)
                ax.annotate(f'{int(round(val_after))}',
                            xy=(x_after, val_after + ci_after[1]),
                            xytext=(0, 3), textcoords='offset points',
                            ha='center', va='bottom', fontsize=13, fontweight='bold')
            
            # Delta annotation with CI (boxed)
            if show_delta_annotations and (exp_name, metric) in delta_results:
                delta_info = delta_results[(exp_name, metric)]
                delta_val = delta_info['delta'] * scale
                delta_ci_low = delta_info['ci_low'] * scale
                delta_ci_high = delta_info['ci_high'] * scale
                
                # Position delta annotation above the higher bar
                max_bar_top = max(val_before + ci_before[1], val_after + ci_after[1])
                delta_y = max_bar_top + 12
                
                # Format delta string with CI
                sign = '+' if delta_val >= 0 else ''
                delta_text = f'Δ = {sign}{delta_val:.0f}' #\n[{delta_ci_low:.0f}, {delta_ci_high:.0f}]'
                
                # Color: gray for bias_rate (neutral), otherwise green/red based on direction
                if metric == 'bias_rate':
                    delta_color = '#666666'  # Gray for neutral
                    box_edge = '#888888'
                elif metric == 'FPR':
                    delta_color = '#2e7d32' if delta_val < 0 else '#c62828'
                    box_edge = delta_color
                else:
                    delta_color = '#2e7d32' if delta_val > 0 else '#c62828'
                    box_edge = delta_color
                
                # Center of the bar pair
                x_center = x_positions[exp_idx] + bar_width / 2
                
                ax.annotate(delta_text,
                            xy=(x_center, delta_y),
                            ha='center', va='bottom', fontsize=12,
                            fontweight='bold', color=delta_color,
                            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', 
                                      edgecolor=box_edge, alpha=0.9, linewidth=1.5))
        
        # Panel formatting
        ax.set_title(metric_labels.get(metric, metric), fontsize=19, fontweight='bold', pad=10)
        # Set x-ticks at center of each bar pair
        tick_positions = x_positions + bar_width / 2
        ax.set_xticks(tick_positions)
        # Simplify model names (drop "Qwen3-")
        simplified_names = [simplify_name(n) for n in exp_names]
        ax.set_xticklabels(simplified_names, fontsize=17, fontweight='bold')
        ax.set_ylim(ylim)
        ax.tick_params(axis='y', labelsize=14)
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)
        
        # No ylabel to save horizontal space
    
    # Add legend below plot - use hatching to show before/after
    # Use generic "Training" labels if any config contains VFT
    has_vft = any('VFT' in cfg.get('name', '') for cfg in experiment_configs)
    before_label = 'Before Training' if has_vft else 'Before CST'
    after_label = 'After Training' if has_vft else 'After CST'
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='white', edgecolor='black', hatch='///', label=before_label),
        Patch(facecolor='white', edgecolor='black', label=after_label),
    ]
    fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.14),
               ncol=2, frameon=False, prop={'weight': 'bold', 'size': 19})
    
    # Overall title - bigger and tighter to plot
    if title:
        fig.suptitle(title, fontsize=25, fontweight='bold', y=0.91)
    plt.tight_layout(rect=[0, 0.12, 1, 0.94])  # Leave room for suptitle and legend
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

    
    return all_results, delta_results

In [ ]:
# CST vs VFT Comparison - Loop over all configs and cf_type_filters

# Define all three experiment config sets
all_experiment_configs = {
    # 'gpt-oss_qwen-235b': [
    #     {
    #         'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
    #         'name': 'CST',
    #         'metrics_prefix': 'all_bias_monitor',
    #         'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
    #         'simulator_config': {
    #             'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
    #             'condition_on_original_answers': True, 'condition_on_explanations': True
    #         },
    #     },
    #     {
    #         'args_path': 'args_VFT_mmlu-2way_6-cf-types_gpt-oss-120b_PFT_cfm-0.1_faith-FNs_k-0_CoT-T_yesno-xye_1rounds_e20_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
    #         'name': 'VFT',
    #         'metrics_prefix': 'all_bias_monitor',
    #         'metrics_suffix': 'k-0_CoT-T_yesno-xye',
    #         'simulator_config': {
    #             'k_shots': 0, 'use_reasoning': True, 'prompt_type': 'yesno',
    #             'condition_on_original_answers': True, 'condition_on_explanations': True
    #         },
    #     },
    # ],
    'gpt-oss_deepseek-chat': [
        {
            'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
            'name': 'CST',
            'metrics_prefix': 'all_bias_monitor',
            'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
            'simulator_config': {
                'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
                'condition_on_original_answers': True, 'condition_on_explanations': True
            },
        },
        {
            'args_path': 'args_VFT_v2_mmlu-2way_6-cf-types_gpt-oss-120b_PFT_cfa-0.1_faith-FNs_k-0_CoT-T_yesno-xye_1rounds_e20_1samples_deepseek-chat_n1000-2000_sd0.json',
            'name': 'VFT',
            'metrics_prefix': 'all_bias_monitor',
            'metrics_suffix': 'k-0_CoT-T_yesno-xye',
            'simulator_config': {
                'k_shots': 0, 'use_reasoning': True, 'prompt_type': 'yesno',
                'condition_on_original_answers': True, 'condition_on_explanations': True
            },
        },
    ],
    # 'qwen-235b_deepseek-chat': [
    #     {
    #         'args_path': 'args_scaling_model_mmlu-2way_6-cf-types_Qwen3-235B-A22B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
    #         'name': 'CST',
    #         'metrics_prefix': 'all_bias_monitor',
    #         'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
    #         'simulator_config': {
    #             'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
    #             'condition_on_original_answers': True, 'condition_on_explanations': True
    #         },
    #     },
    #     {
    #         'args_path': 'args_VFT_mmlu-2way_6-cf-types_Qwen3-235B-A22B_PFT_cfm-0.1_faith-FNs_k-0_CoT-T_yesno-xye_1rounds_e20_1samples_deepseek-chat_n1000-2000_sd0.json',
    #         'name': 'VFT',
    #         'metrics_prefix': 'all_bias_monitor',
    #         'metrics_suffix': 'k-0_CoT-T_yesno-xye',
    #         'simulator_config': {
    #             'k_shots': 0, 'use_reasoning': True, 'prompt_type': 'yesno',
    #             'condition_on_original_answers': True, 'condition_on_explanations': True
    #         },
    #     },
    # ],
}

# Define cf_type_filters
cf_type_filters = {
    'ID': ID_CF_TYPES,
    # 'OOD': OOD_CF_TYPES,
}

# Loop over all combinations
all_results = {}
for config_name, experiment_configs in all_experiment_configs.items():
    for cf_name, cf_filter in cf_type_filters.items():
        print(f"\n{'='*60}")
        print(f"Config: {config_name}, CF Type: {cf_name}")
        print(f"{'='*60}")
        
        results = plot_experiment_comparison_bars(
            experiment_configs=experiment_configs,
            cf_type_filter=cf_filter,
            metrics_to_plot=['recall', 'FPR', 'gmean'],
            n_bootstrap=100,
            n_seeds=5,
            title=f'CST vs VFT',
            save_path=f'zfigs/cst_vs_vft_{config_name}_{cf_name}.pdf',
            force_rerun=False,
        )
        all_results[(config_name, cf_name)] = results

In [ ]:
# CST vs VFT Comparison using plot_metric_panels_bars

experiment_configs = [
        {
            'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
            'name': 'CST',
            'metrics_prefix': 'all_bias_monitor',
            'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
            'simulator_config': {
                'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
                'condition_on_original_answers': True, 'condition_on_explanations': True
            },
        },
        {
            'args_path': 'args_VFT_v2_mmlu-2way_6-cf-types_gpt-oss-120b_PFT_cfa-0.1_faith-FNs_k-0_CoT-T_yesno-xye_1rounds_e20_1samples_deepseek-chat_n1000-2000_sd0.json',
            'name': 'VFT',
            'metrics_prefix': 'all_bias_monitor',
            'metrics_suffix': 'k-0_CoT-T_yesno-xye',
            'simulator_config': {
                'k_shots': 0, 'use_reasoning': True, 'prompt_type': 'yesno',
                'condition_on_original_answers': True, 'condition_on_explanations': True
            },
        },
]

# experiment_configs = [
#     {
#         'args_path': 'args_scaling_model_mmlu-2way_6-cf-types_Qwen3-235B-A22B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
#         'name': 'CST',
#         'metrics_prefix': 'all_bias_monitor',
#         'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
#         'simulator_config': {
#             'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
#             'condition_on_original_answers': True, 'condition_on_explanations': True
#         },
#     },
#     {
#         'args_path': 'args_VFT_v2_mmlu-2way_6-cf-types_Qwen3-235B-A22B_PFT_cfa-0.1_faith-FNs_k-0_CoT-T_yesno-xye_1rounds_e20_1samples_deepseek-chat_n1000-2000_sd0.json',
#         'name': 'VFT',
#         'metrics_prefix': 'all_bias_monitor',
#         'metrics_suffix': 'k-0_CoT-T_yesno-xye',
#         'simulator_config': {
#             'k_shots': 0, 'use_reasoning': True, 'prompt_type': 'yesno',
#             'condition_on_original_answers': True, 'condition_on_explanations': True
#         },
#     },
# ]

results, deltas = plot_metric_panels_bars(
    experiment_configs=experiment_configs,
    cf_type_filter=ID_CF_TYPES,
    # cf_type_filter=OOD_CF_TYPES,
    metrics_to_plot=['gmean', 'recall', 'FPR', 'bias_rate'],
    n_bootstrap=100,
    figsize=(16, 6),
    ylim=(0, 106),
    n_seeds=5,
    title='CST vs VFT on MMLU (In-distribution Cues)',
    # title='CST vs VFT on MMLU (OOD Cues)',
    save_path='zfigs/cst_vs_vft_panels.pdf',
    force_rerun=False,
)

# Online vs offline

In [ ]:
# CST vs VFT Comparison using the modular function

# gpt-oss + qwen-235b
experiment_configs = [
    {
        'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'Online',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_offline_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_1rounds_e20_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'Offline',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

results = plot_experiment_comparison_bars(
    experiment_configs=experiment_configs,
    cf_type_filter=ID_CF_TYPES,
    # cf_type_filter=OOD_CF_TYPES,
    metrics_to_plot=['recall', 'FPR', 'gmean'],
    n_bootstrap=100,
    n_seeds=5,
    title='Online vs Offline Training',
    save_path='zfigs/online_vs_offline_barplot.pdf',
    force_rerun=False,
)

In [ ]:
# Online vs Offline Training - Metric Panels Visualization

experiment_configs = [
    {
        'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'Online',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_offline_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-1.0_faith_k-0_CoT-F_cf-sim-xye_1rounds_e20_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'Offline',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

results = plot_metric_panels_bars(
    experiment_configs=experiment_configs,
    cf_type_filter=ID_CF_TYPES,
    metrics_to_plot=['gmean', 'recall', 'FPR'],
    n_bootstrap=100,
    n_seeds=5,
    figsize=(14, 7),
    title='Online vs Offline Training',
    save_path='zfigs/online_vs_offline_metric_panels.pdf',
    force_rerun=False,
)

# ID vs OOD

In [ ]:
experiment_configs = [
    {
        'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'ID',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'cf_type_filter': ID_CF_TYPES,  # <-- per-experiment filter
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'OOD',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'cf_type_filter': OOD_CF_TYPES,  # <-- per-experiment filter
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

results, deltas = plot_metric_panels_bars(
    experiment_configs=experiment_configs,
    # cf_type_filter parameter is now optional - per-experiment overrides take precedence
    metrics_to_plot=['gmean', 'recall', 'FPR'],
    n_bootstrap=100,
    figsize=(14, 6),
    n_seeds=5,
    title='ID vs OOD Cue Performance',
    save_path='zfigs/id_vs_ood_barplot.pdf',
    force_rerun=False,
)

# Rxye vs PFT

In [ ]:
# CST vs VFT Comparison using the modular function

# gpt-oss + qwen-235b
experiment_configs = [
    {
        'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'CST',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_Rxye_mmlu-2way_6-cf-types_gpt-oss-120b_PFT_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'PFT',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

results = plot_metric_panels_bars(
    experiment_configs=experiment_configs,
    cf_type_filter=ID_CF_TYPES,
    metrics_to_plot=['gmean', 'recall', 'FPR'],
    n_bootstrap=100,
    n_seeds=5,
    figsize=(12, 7.5),
    title='Reward Structure Ablation',
    save_path='zfigs/reward_structure_ablation_barplot.pdf',
    force_rerun=False,
)

# Rewriter model

In [ ]:
# gpt-oss + qwen-235b
experiment_configs = [
    {
        'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json',
        'name': 'gpt-oss-120b',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_rewriter_ritm-False_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd2.json',
        'name': 'Qwen3-235B',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

# # gpt-oss + deepseek
# experiment_configs = [
#     {
#         'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
#         'name': 'gpt-oss-120b',
#         'metrics_prefix': 'all_bias_monitor',
#         'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
#         'simulator_config': {
#             'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
#             'condition_on_original_answers': True, 'condition_on_explanations': True
#         },
#     },
#     {
#         'args_path': 'args_rewriter_ritm-False_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
#         'name': 'deepseek-V3-0324',
#         'metrics_prefix': 'all_bias_monitor',
#         'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
#         'simulator_config': {
#             'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
#             'condition_on_original_answers': True, 'condition_on_explanations': True
#         },
#     },
# ]

results = plot_metric_panels_bars(
    experiment_configs=experiment_configs,
    # cf_type_filter=ID_CF_TYPES,
    cf_type_filter=OOD_CF_TYPES,
    metrics_to_plot=['gmean', 'recall', 'FPR'],
    n_bootstrap=100,
    n_seeds=5,
    ylim=(0, 105),
    figsize=(14, 7.5),
    title='Rewriter Model Ablation',
    save_path='zfigs/rewriter_ablation_barplot.pdf',
    force_rerun=False,
)

# Task Model Ablation

In [ ]:
# task model ablation

experiment_configs = [
    {
        'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
        'name': 'gpt-oss-120b',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_scaling_model_mmlu-2way_6-cf-types_Qwen3-235B-A22B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
        'name': 'Qwen3-235B',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

results = plot_experiment_comparison_bars(
    experiment_configs=experiment_configs,
    # cf_type_filter=ID_CF_TYPES,
    cf_type_filter=OOD_CF_TYPES,
    metrics_to_plot=['recall', 'FPR', 'gmean', 'bias_rate'],
    n_bootstrap=100,
    n_seeds=5,
    title='Task Model Ablation',
    save_path='zfigs/task_model_ablation.pdf',
    force_rerun=False,
)

In [ ]:
# Task Model Ablation - Metric Panels Visualization

experiment_configs = [
    {
        'args_path': 'args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
        'name': 'gpt-oss-120b',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_scaling_model_mmlu-2way_6-cf-types_Qwen3-235B-A22B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
        'name': 'Qwen3-235B',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

results = plot_metric_panels_bars(
    experiment_configs=experiment_configs,
    cf_type_filter=ID_CF_TYPES,
    # cf_type_filter=OOD_CF_TYPES,
    metrics_to_plot=['gmean', 'recall', 'FPR', 'bias_rate'],
    n_bootstrap=100,
    figsize=(18, 6),
    ylim=(0, 110),
    n_seeds=5,
    title='Task Model Ablation',
    save_path='zfigs/task_model_ablation_metric_panels.pdf',
    force_rerun=False,
)

# Model Scaling Analysis

In [ ]:
# Model Scaling: Compare across model sizes (4B, 30B, 235B)

experiment_configs = [
    {
        'args_path': 'args_scaling_model_mmlu-2way_6-cf-types_Qwen3-4B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e10_1samples_deepseek-chat_n1000-2000_sd0.json',
        'name': 'Qwen3-4B',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_scaling_model_mmlu-2way_6-cf-types_Qwen3-30B-A3B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
        'name': 'Qwen3-30B',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': 'args_scaling_model_mmlu-2way_6-cf-types_Qwen3-235B-A22B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json',
        'name': 'Qwen3-235B',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

results = plot_experiment_comparison_bars(
    experiment_configs=experiment_configs,
    metrics_to_plot=['recall', 'FPR', 'gmean', 'bias_rate'],
    n_bootstrap=1000,
    n_seeds=5,
    # cf_type_filter=ID_CF_TYPES,
    # title='Model Scaling: Monitor Performance on In-distribution Cues',
    # save_path='zfigs/model_scaling_barplot_ID.pdf',
    cf_type_filter=OOD_CF_TYPES,
    title='Model Scaling: Monitor Performance on OOD Cues',
    save_path='zfigs/model_scaling_barplot_OOD.pdf',
    figsize=(12, 6),
    bold_after_gmean=False,
    force_rerun=False,
)

# More Metric panels

In [ ]:
experiment_configs = [
    {
        'args_path': "args_main_sweep_snli-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        'name': 'SNLI',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': "args_main_sweep_ethics-commonsense-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        'name': 'ETHICS-cm',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': "args_main_sweep_ethics-justice-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json",
        'name': 'ETHICS-justice',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
    {
        'args_path': "args_main_sweep_mmlu-pro-law-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_6rounds_e5_1samples_Qwen3-235B-A22B-tput_n640-320_sd0.json",
        'name': 'MMLU-Pro-Law',
        'metrics_prefix': 'all_bias_monitor',
        'metrics_suffix': 'k-0_CoT-F_cf-sim-xye',
        'simulator_config': {
            'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim',
            'condition_on_original_answers': True, 'condition_on_explanations': True
        },
    },
]

results, deltas = plot_metric_panels_bars(
    experiment_configs=experiment_configs,
    metrics_to_plot=['gmean', 'recall', 'FPR'],
    n_bootstrap=100,
    n_seeds=5,
    cf_type_filter=ID_CF_TYPES,
    save_path='zfigs/metric_panels_ID.pdf',
    title='Cue Influence Monitorability (ID Cues)',
    # cf_type_filter=OOD_CF_TYPES,
    # save_path='zfigs/metric_panels_OOD.pdf',
    # title='Cue Influence Monitorability (OOD Cues)',
    figsize=(16, 5),
    show_annotations=True,
    show_delta_annotations=True,
    force_rerun=False,
    ylim=(0, 104),
)

# Plot backfire cues

In [ ]:
# Example: Compare SL vs RL experiments
# Update these paths to your actual SL and RL experiment args files

experiment_configs = [
    {
        "args_path": "args_scaling_model_mmlu-2way_6-cf-types_Qwen3-235B-A22B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "Persuading Cues",
    },
    {
        "args_path": "args_backfire_cues_mmlu-2way_4-cf-types_Qwen3-235B-A22B_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e10_1samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "Dissuading Cues",
    },
]
metrics_to_reconcile = [
    "all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye",
    "all_backfire_monitor_gmean_k-0_CoT-F_cf-sim-xye",
]

# Quick plot without bootstrap (fast iteration, no CIs)
data = plot_experiment_comparison(
    experiment_configs=experiment_configs,
    metric=metrics_to_reconcile,
    x_axis="round",  # or "round"
    figsize=(5.2, 4.2),
    ylim=(0.1, 0.9),
    n_seeds=5,
    title="Persuading vs. Dissuading Cues",
    ylabel="G-Mean",
    show_values=True,
    print_pvalues=True,
    ci_style='ribbon+lines',
    cf_type_filter=ID_CF_TYPES,
    run_bootstrap=False,    # Set True for final plots with proper CIs
    save_name="backfire_cues",
)

# Monitorability/Bias/correctness/CoT length

In [ ]:
# Compare SL vs RL on: G-mean, Bias rate, CF accuracy, CoT length
experiment_configs = [
    {
        "args_path": "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "CoT Rewriting",
    },
    {
        "args_path": "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_8samples_deepseek-chat_n1000-2000_sd0.json",
        "label": "RL",  
    },
]

facet_configs = [
    {
        "cf_type_filter": ID_CF_TYPES,
        "title": "G-Mean (In-Distribution)",
        "ylabel": "G-Mean",
        "metric": "all_bias_monitor_gmean_k-0_CoT-T_cf-sim-xye",
        "ylim": (0, 1),
    },
    {
        "cf_type_filter": ID_CF_TYPES,
        "title": "Bias Rate (In-Distribution)",
        "ylabel": "Bias Rate",
        "metric": "all_greedy_bias_rate",
        "ylim": (0, 0.6),
    },
    {
        "cf_type_filter": ID_CF_TYPES,
        "title": "Model Accuracy on CF Questions",
        "ylabel": "Accuracy",
        "metric": "cf_from_greedy_data_greedy_acc",
        "ylim": (.9, 1),
    },
    {
        "cf_type_filter": ID_CF_TYPES,
        "title": "CoT Length",
        "ylabel": "Words",
        "metric": "avg_original_greedy_cot_words",
        "ylim": None,  # Auto scale
        "scale": 1,  # Don't multiply by 100
        "ylim": (100, 200),
    },
]

plot_experiment_comparison_facets(
    facet_configs=facet_configs,
    experiment_configs=experiment_configs,
    metric="all_greedy_bias_rate",  # Default fallback
    x_axis="round",
    n_seeds=3,
    show_values=True,
    ncols=4,
    run_bootstrap=False,
    save_name="sl_vs_rl_4metrics",
    value_decimals=1,
)

# Scaling Data Plot

In [ ]:
# Scaling Data: Plot last round G-Mean vs n_train - ID and OOD on same plot

# === CONFIGURATION ===
run_bootstrap = True
n_bootstrap = 1000
n_seeds = 5
confidence_level = 0.95
force_rerun = False
show_error_bar = True

exp_labels_and_names = [
    (50, "args_scaling_data_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e11_1samples_qwen3-235b-a22b_n50-2000_sd0.json"),
    (125, "args_scaling_data_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n125-2000_sd0.json"),
    (500, "args_scaling_data_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n500-2000_sd0.json"),
    (1000, "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json"),
]

metrics_to_compute = ["all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye"]
gmean_col = "all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye_test"

simulator_configs = [
    {'k_shots': 0, 'use_reasoning': False, 'prompt_type': 'cf-sim', 
     'condition_on_original_answers': True, 'condition_on_explanations': True},
]

# Run bootstrap for both ID and OOD
cf_type_configs = [
    ("ID", ID_CF_TYPES),
]

all_plot_data = {}  # {cf_label: list of dicts with n_train, mean, ci_low, ci_high}

for cf_label, cf_type_filter in cf_type_configs:
    print(f"\n{'='*60}")
    print(f"Processing {cf_label} counterfactual types")
    print(f"{'='*60}")
    
    plot_data = []
    
    for n_train, args_path in exp_labels_and_names:
        print(f"\n--- n_train = {n_train} ---")
        
        # Load args
        with open(f"results/{args_path}", "r") as f:
            args = json.load(f)
            args = SimpleNamespace(**args)
        
        last_round = int(args.train_rounds)
        all_rounds = list(range(last_round + 1))
        
        # --- Print trend by loading stats across seeds ---
        results = load_stats_across_seeds(
            base_exp_name=utils.get_base_exp_name(args),
            n_seeds=n_seeds,
        )
        # Filter to ID or OOD cf types
        if cf_type_filter == ID_CF_TYPES:
            results = results[results['counterfactual_type'] == 'all_ID']
        elif cf_type_filter == OOD_CF_TYPES:
            results = results[results['counterfactual_type'] == 'all_OOD']
        
        # Get the gmean column for the last round
        gmean_col_stats = "all_bias_monitor_gmean_k-0_CoT-F_cf-sim-xye_test"
        gmean_vals = results[gmean_col_stats]
        print(f"n_train={n_train} ({cf_label}): {gmean_vals.round(3).tolist()}")
        
        # --- Bootstrap only last round for CIs ---
        cache_id = "ID" if cf_type_filter == ID_CF_TYPES else "OOD" if cf_type_filter == OOD_CF_TYPES else "all"
        cache_path = f"cache/bootstrap_scaling_lastround_{utils.get_base_exp_name(args)}_{n_seeds}seeds_{cache_id}_{n_bootstrap}.pkl"
        
        results_long, results_wide, bootstrap_samples, true_means_wide = bootstrap_evaluation_stats(
            args=args,
            n_seeds=n_seeds,
            rounds=[last_round],  # Only last round for CIs
            simulator_sweep_configs=simulator_configs,
            metrics_to_compute=metrics_to_compute,
            cf_type_filter=cf_type_filter,
            n_bootstrap=n_bootstrap,
            confidence_level=confidence_level,
            random_state=42,
            block_col='id',
            verbose=True,
            cache_path=cache_path,
            force_rerun=force_rerun,
            print_pvalues=False,
        )
        
        # Extract last round stats for the plot
        # Use TRUE MEANS for point estimate (bootstrap means are biased for nonlinear metrics like G-Mean)
        # Use bootstrap only for CI bounds
        last_row_true = true_means_wide[true_means_wide['round'] == last_round].iloc[0]
        last_row_bootstrap = results_wide[results_wide['round'] == last_round].iloc[0]
        
        mean = last_row_true[gmean_col]  # True sample mean (unbiased)
        ci_low = last_row_bootstrap[f"{gmean_col}_ci_low"]  # Bootstrap CI
        ci_high = last_row_bootstrap[f"{gmean_col}_ci_high"]  # Bootstrap CI
        
        plot_data.append({
            'n_train': n_train,
            'mean': mean,
            'ci_low': ci_low,
            'ci_high': ci_high,
        })
        
        bootstrap_mean = last_row_bootstrap[gmean_col]
        print(f"Last round ({last_round}) True G-Mean: {mean:.3f}, Bootstrap Mean: {bootstrap_mean:.3f} (bias: {bootstrap_mean - mean:+.3f})")
        print(f"  CI: [{ci_low:.3f}, {ci_high:.3f}]")
    
    all_plot_data[cf_label] = plot_data

# === CREATE LINE PLOT ===
print(f"\n{'='*60}")
print("Creating scaling plot...")
print(f"{'='*60}")

fig, ax = plt.subplots(figsize=(7, 5))

# Custom palette - muted, professional colors (matching plot_metric)
palette = [
    "#5fa89d",  # muted teal
    "#d9844f",  # muted orange
    "#6b7fa3",  # muted blue
    "#a67fb8",  # muted purple
    "#7fb88a",  # muted green
]

# Colors for ID and OOD (using palette colors from notebook)
colors = {
    'ID': "#7fb88a",   # muted green
    'OOD': "#6b7fa3",  # muted blue 
}
markers = {'ID': 'o', 'OOD': 's'}

# Scale factor for y-axis (convert 0-1 to percentage)
scale = 100

for cf_label, plot_data in all_plot_data.items():
    n_train_values = [d['n_train'] for d in plot_data]
    means = [d['mean'] * scale for d in plot_data]
    ci_lows = [d['ci_low'] * scale for d in plot_data]
    ci_highs = [d['ci_high'] * scale for d in plot_data]
    
    # Compute error bars (distance from mean to CI bounds)
    yerr_low = [m - cl for m, cl in zip(means, ci_lows)]
    yerr_high = [ch - m for m, ch in zip(means, ci_highs)]
    yerr = [yerr_low, yerr_high]
    
    if show_error_bar:
        ax.errorbar(
            n_train_values, means, yerr=yerr,
            label=cf_label,
            color=colors[cf_label],
            marker=markers[cf_label],
            markersize=8,
            capsize=4,
            capthick=1.5,
            elinewidth=1.5,
            linewidth=2,
        )
    else:
        ax.plot(
            n_train_values, means,
            label=cf_label,
            color=colors[cf_label],
            marker=markers[cf_label],
            markersize=8,
            linewidth=2,
        )
    
    # Add text annotations above CIs
    for i, (x, m, ch) in enumerate(zip(n_train_values, means, ci_highs)):
        y_pos = ch if show_error_bar else m
        ax.annotate(
            f"{m:.0f}",
            xy=(x, y_pos),
            xytext=(0, 12),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=10,
            alpha=0.7,
        )

# Log scale x-axis with nominal tick labels
ax.set_xscale('log')
n_train_values = [d['n_train'] for d in all_plot_data['ID']]
ax.set_xticks(n_train_values)
ax.set_xticklabels([str(n) for n in n_train_values])
ax.xaxis.set_minor_locator(plt.NullLocator())  # Remove minor ticks

# Labels and styling
ax.set_xlabel('Training Examples', fontsize=13, fontweight='bold')
ax.set_ylabel('G-Mean', fontsize=13, fontweight='bold')
ax.set_title('Data Scaling: G-Mean vs Training Data Size', fontsize=15, fontweight='bold', pad=10)

if PLOT_BACKGROUND_COLOR:
    ax.set_facecolor(PLOT_BACKGROUND_COLOR)

# Dark x/y axes styling (matching other plots)
ax.spines['top'].set_color('#ffffff')
ax.spines['right'].set_color('#ffffff')
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')
ax.spines['top'].set_linewidth(1.5)
ax.spines['right'].set_linewidth(1.5)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

# Legend below figure
if len(cf_type_configs) > 1:
    ax.legend(
        frameon=False, 
        fontsize=12,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.12),
        ncol=len(all_plot_data)
    )

ax.set_ylim(40, 100)  # Scaled y-axis (percentage)

plt.tight_layout()
plt.savefig('zfigs/scaling_data_gmean.pdf', bbox_inches='tight', dpi=300)
plt.show()

# Simulator Ablation: SL vs RL Faceted by Simulator Type

In [ ]:
# Simulator Ablation: G-Mean vs Rounds, faceted by SL/RL, lines by simulator type
# Each experiment has its own CoT setting (CoT-F or CoT-T), so metric name changes accordingly

n_seeds = 3

# Define experiments: (facet, simulator_label, args_path, cot_setting)
experiments = [
    ("SL", "Qwen3-235B", "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_qwen3-235b-a22b_n1000-2000_sd0.json", "CoT-F"),
    ("RL", "Qwen3-235B", "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_8samples_Qwen3-235B-A22B-tput_n1000-2000_sd0.json", "CoT-F"),
    ("SL", "DeepSeek-V3", "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json", "CoT-F"),
    ("SL", "DeepSeek-V3 + CoT", "args_cue_pivot_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_1samples_deepseek-chat_n1000-2000_sd0.json", "CoT-T"),
    ("RL", "DeepSeek-V3", "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-F_cf-sim-xye_5rounds_e5_8samples_deepseek-chat_n1000-2000_sd0.json", "CoT-F"),
    ("RL", "DeepSeek-V3 + CoT", "args_RL_mmlu-2way_6-cf-types_gpt-oss-120b_unlik_cfa-0.8_faith_k-0_CoT-T_cf-sim-xye_5rounds_e5_8samples_deepseek-chat_n1000-2000_sd0.json", "CoT-T"),
]

# Collect data: {facet: {simulator: [{round, mean}, ...]}}
plot_data = {"SL": {}, "RL": {}}

for facet, sim_label, args_path, cot_setting in experiments:
    print(f"Loading {facet} - {sim_label}...")
    
    # Load args
    with open(f"results/{args_path}", "r") as f:
        args = json.load(f)
        args = SimpleNamespace(**args)
    
    # Load stats across seeds
    results = load_stats_across_seeds(
        base_exp_name=utils.get_base_exp_name(args),
        n_seeds=n_seeds,
    )
    
    # Filter to ID counterfactual types
    results = results[results['counterfactual_type'] == 'all_ID']
    
    # Get the correct gmean column based on CoT setting
    gmean_col = f"all_bias_monitor_gmean_k-0_{cot_setting}_cf-sim-xye_test"
    
    # Get mean across seeds for each round
    round_data = []
    for rnd in range(int(args.train_rounds) + 1):
        rnd_results = results[results['round'] == rnd]
        if len(rnd_results) > 0 and gmean_col in rnd_results.columns:
            mean_val = rnd_results[gmean_col].mean()
            round_data.append({'round': rnd, 'mean': mean_val})
    
    plot_data[facet][sim_label] = round_data
    list_means = [round(d['mean'],3) for d in round_data]
    print(f"  Rounds: {[d['round'] for d in round_data]}, Means: {list_means}")

# === CREATE FACETED PLOT ===
print(f"\n{'='*60}")
print("Creating simulator ablation faceted plot...")
print(f"{'='*60}")

# Custom palette
palette = [
    "#a67fb8",  # muted purple (Qwen)
    "#6b7fa3",  # muted blue (DeepSeek-V3)
    "#5fa89d",  # muted teal (DeepSeek-V3 + CoT)
]
sim_colors = {"Qwen3-235B": palette[0], "DeepSeek-V3": palette[1], "DeepSeek-V3 + CoT": palette[2]}

scale = 100  # Convert to percentage

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
facet_order = ["SL", "RL"]
facet_titles = {"SL": "CoT Rewriting with LLM", "RL": "Reinforcement Learning (RL)"}

for ax_idx, facet in enumerate(facet_order):
    ax = axes[ax_idx]
    
    # Build DataFrame for seaborn
    facet_plot_data = []
    for sim_label, round_data in plot_data[facet].items():
        if not round_data:
            continue
        for d in round_data:
            facet_plot_data.append({
                'Round': d['round'],
                'G-Mean': d['mean'] * scale,
                'Simulator': sim_label
            })
    
    facet_df = pd.DataFrame(facet_plot_data)
    
    if not facet_df.empty:
        # Use seaborn lineplot for consistent styling
        sns.lineplot(
            data=facet_df, x='Round', y='G-Mean', hue='Simulator',
            marker='o', linewidth=3.0, markersize=7,
            palette=[sim_colors[s] for s in facet_df['Simulator'].unique()],
            ax=ax
        )
    
    ax.set_xlabel('Round', fontsize=13, fontweight='bold')
    if ax_idx == 0:
        ax.set_ylabel('G-Mean', fontsize=13, fontweight='bold')
    if ax_idx == 1:
        ax.set_ylabel('')
    ax.set_title(facet_titles[facet], fontsize=14, fontweight='bold', pad=10)
    
    if PLOT_BACKGROUND_COLOR:
        ax.set_facecolor(PLOT_BACKGROUND_COLOR)
    
    # Dark x/y axes styling
    ax.spines['top'].set_color('#ffffff')
    ax.spines['right'].set_color('#ffffff')
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.spines['top'].set_linewidth(1.5)
    ax.spines['right'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    
    ax.set_xticks(range(6))
    ax.set_ylim(20, 100)
    
    # Remove per-axis legend (we'll create a shared one)
    if ax.get_legend():
        ax.get_legend().remove()

# Shared legend below figure
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    frameon=False,
    loc='upper center',
    prop = {'size': 16, 'weight': 'bold'},
    bbox_to_anchor=(0.5, .08),
    ncol=len(handles)
)

fig.suptitle("Simulator Model Ablation", fontsize=24, fontweight='bold', y=1.0)
plt.tight_layout()
plt.subplots_adjust(bottom=0.18, top=0.85)  # Make room for legend and suptitle
plt.savefig('zfigs/simulator_ablation_sl_vs_rl.pdf', bbox_inches='tight', dpi=300)
plt.show()

# Plot Cue Influence

In [ ]:
from statsmodels.stats.proportion import proportion_confint
use_dir = "results"

load_args_paths = [
    "args_cue_influence_cr0.5_mmlu-2way_20-cf-types_gpt-oss-120b_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xy_0rounds_e5_1samples_qwen3-235b-a22b_n0-2000_sd0.json",
    "args_cue_influence_cr0.5_mmlu-2way_20-cf-types_qwen3-235b-a22b_PFT_cfa-1.0_faith_k-0_CoT-F_cf-sim-xy_0rounds_e5_1samples_qwen3-235b-a22b_n0-2000_sd0.json"
]
dfs = []

for args_path in load_args_paths:
    with open(f"{use_dir}/{args_path}", "r") as f:
        args = json.load(f)
        args = SimpleNamespace(**args)
    exp_name = utils.get_exp_name(args)
    results = pd.read_csv(f"{use_dir}/stats_{exp_name}.csv")
    roundspread_df = pd.read_csv(f"{use_dir}/stats_roundspread_{exp_name}.csv")
    dfs.append((args_path, roundspread_df))

In [ ]:
def plot_cue_influence_summary(roundspread_df, round_num=0, figsize=(18, 6), print_table=False, title=None):
    """
    Simple summary plot: 3 bars per cue type showing total rates:
    - Bias Rate (blue)
    - Backfire Rate (orange)
    - Stochastic Flip Rate (gray)
    
    Ordered by total bias rate (descending, left to right).
    
    Args:
        roundspread_df: DataFrame with metrics in 'metric' column and values in 'test_round{N}' columns
        round_num: Which round to plot (default 0)
        figsize: Figure size tuple
        title: Optional custom title
    """
    # Define metrics to extract
    metrics_needed = [
        "all_greedy_bias_rate",
        "all_greedy_backfire_rate",
        "all_stochastic_flip_rate",
        "all_greedy_bias_rate_eligible_n",
        "all_greedy_backfire_rate_eligible_n",
        "all_n",
    ]
    
    val_col = f'test_round{round_num}'
    filtered_df = roundspread_df[roundspread_df['metric'].isin(metrics_needed)].copy()
    
    # Pivot to get metrics as columns per cue type
    pivot_df = filtered_df.pivot(
        index='counterfactual_type', 
        columns='metric', 
        values=val_col
    ).reset_index()
    
    # Sort by bias rate descending (left to right)
    pivot_df = pivot_df.sort_values('all_greedy_bias_rate', ascending=False)
    
    cue_types = pivot_df['counterfactual_type'].values
    x_pos = np.arange(len(cue_types))
    
    # Extract values and compute CIs
    def get_vals_and_cis(rate_col, n_col):
        # Handle missing columns gracefully
        if rate_col not in pivot_df.columns:
            vals = np.zeros(len(pivot_df))
            return vals, vals.copy(), vals.copy()
        
        vals = pivot_df[rate_col].fillna(0).values
        ns = pivot_df[n_col].fillna(1).values if n_col in pivot_df.columns else np.ones(len(vals))
        
        ci_lows, ci_highs = [], []
        for v, n in zip(vals, ns):
            if n > 0 and not np.isnan(v):
                successes = int(np.round(v * n))
                successes = min(successes, int(n))
                low, high = proportion_confint(successes, n, alpha=0.05, method='wilson')
                ci_lows.append(low)
                ci_highs.append(high)
            else:
                ci_lows.append(v)
                ci_highs.append(v)
        return vals, np.array(ci_lows), np.array(ci_highs)
    
    # Get all values
    bias_rate, bias_ci_low, bias_ci_high = get_vals_and_cis(
        'all_greedy_bias_rate', 'all_greedy_bias_rate_eligible_n')
    backfire_rate, backfire_ci_low, backfire_ci_high = get_vals_and_cis(
        'all_greedy_backfire_rate', 'all_greedy_backfire_rate_eligible_n')
    flip_rate, flip_ci_low, flip_ci_high = get_vals_and_cis(
        'all_stochastic_flip_rate', 'all_n')
    
    # Color palette
    colors = {
        'bias': '#5CACDB',        # Blue
        'backfire': '#E69557',    # Orange
        'stochastic': '#969696',  # Gray
    }
    
    # Helper to compute non-negative error bars
    def safe_yerr(vals, ci_low, ci_high):
        return [np.maximum(0, vals - ci_low), np.maximum(0, ci_high - vals)]
    
    # Create single figure
    fig, ax = plt.subplots(figsize=figsize)
    if PLOT_BACKGROUND_COLOR:
        ax.set_facecolor(PLOT_BACKGROUND_COLOR)
        # fig.patch.set_facecolor(PLOT_BACKGROUND_COLOR)
    
    bar_width = 0.35
    offsets = [-bar_width/2, bar_width/2]
    
    # Plot two bar groups
    ax.bar(x_pos + offsets[0], bias_rate, bar_width, 
           label='Bias Rate', color=colors['bias'], 
           yerr=safe_yerr(bias_rate, bias_ci_low, bias_ci_high), 
           capsize=2, error_kw={'linewidth': 0.8, 'alpha': 0.7})
    ax.bar(x_pos + offsets[1], backfire_rate, bar_width,
           label='Backfire Rate', color=colors['backfire'],
           yerr=safe_yerr(backfire_rate, backfire_ci_low, backfire_ci_high),
           capsize=2, error_kw={'linewidth': 0.8, 'alpha': 0.7})
    
    # Add horizontal dotted line for stochastic flip rate (same across all cues)
    stochastic_flip_val = flip_rate[0]  # Same for all cues
    ax.axhline(y=stochastic_flip_val, color=colors['stochastic'], linestyle=':', linewidth=2,
               label=f'Stochastic Flip Rate ({stochastic_flip_val:.2f})')
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(cue_types, rotation=45, ha='right', fontsize=10)
    ax.set_xlabel('Cue Type', fontsize=12, labelpad=10)
    ax.set_ylabel('Rate', fontsize=12, labelpad=10)
    
    # Set y-axis
    ax.set_ylim(0, 1)
    ax.legend(loc='upper center', fontsize=10, frameon=True, fancybox=True, ncol=3)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    
    # Add text labels above bars
    all_vals = [bias_rate, backfire_rate]
    for i in range(len(cue_types)):
        for j, vals in enumerate(all_vals):
            ax.text(x_pos[i] + offsets[j], vals[i] + 0.02, f'{vals[i]:.2f}', 
                    ha='center', va='bottom', fontsize=7, rotation=0)
    
    # Title
    if title is None:
        title = f'Cue Influence Summary (Round {round_num})'
    ax.set_title(title, fontsize=14, weight='bold', pad=15)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary table with sample sizes and CIs
    bias_n = pivot_df['all_greedy_bias_rate_eligible_n'].fillna(0).values
    backfire_n = pivot_df['all_greedy_backfire_rate_eligible_n'].fillna(0).values
    flip_n = pivot_df['all_n'].fillna(0).values
    
    if print_table:
        print(f"\n{'Cue Type':<25} | {'Bias Rate':<20} | {'Backfire Rate':<20} | {'Stochastic Flip':<20}")
        print(f"{'':<25} | {'(n, rate, 95% CI)':<20} | {'(n, rate, 95% CI)':<20} | {'(n, rate, 95% CI)':<20}")
        print("-" * 95)
        for i, cue in enumerate(cue_types):
            bias_str = f"n={int(bias_n[i])}, {bias_rate[i]:.3f} [{bias_ci_low[i]:.3f}, {bias_ci_high[i]:.3f}]"
            backfire_str = f"n={int(backfire_n[i])}, {backfire_rate[i]:.3f} [{backfire_ci_low[i]:.3f}, {backfire_ci_high[i]:.3f}]"
            flip_str = f"n={int(flip_n[i])}, {flip_rate[i]:.3f} [{flip_ci_low[i]:.3f}, {flip_ci_high[i]:.3f}]"
            print(f"{cue:<25} | {bias_str:<20} | {backfire_str:<20} | {flip_str:<20}")
        print()
    
    return pivot_df


# Run the summary plot
for args_path, roundspread_df in dfs:
    print(f"\n{'='*60}")
    print(f"Summary plot: {args_path}")
    result_df = plot_cue_influence_summary(roundspread_df, round_num=0, figsize=(18, 6), title=args_path)
    print(f"{'='*60}")

In [ ]:
def plot_cue_influence_clustered(roundspread_df, round_num=0, figsize=(22, 7), print_table=False, title=None, save_path=None):
    """
    Summary plot grouped by cue clusters with vertical delimiters and cluster averages.
    
    Clusters:
    1. gpt_oss_bias_cues_train (training bias cues)
    2. gpt_oss_bias_cues_test (test bias cues)
    3. Novel cues (shut_down, llm_peer_pressure, backfire_older_brother, backfire_bully)
    4. gpt_oss_backfire_cues (backfire versions of training cues)
    
    Args:
        roundspread_df: DataFrame with metrics in 'metric' column and values in 'test_round{N}' columns
        round_num: Which round to plot (default 0)
        figsize: Figure size tuple
        title: Optional custom title
        save_path: Optional path to save figure as PDF
    """
    from globals import gpt_oss_bias_cues_train, gpt_oss_bias_cues_test, gpt_oss_backfire_cues, cue_renaming_dict
    
    # Define clusters
    cluster_novel = ['shut_down', 'llm_peer_pressure', 'backfire_older_brother', 'backfire_bully']
    
    clusters = [
        ('Train Bias Cues', gpt_oss_bias_cues_train),
        ('Test Bias Cues', gpt_oss_bias_cues_test),
        ('Extra Unused Cues', cluster_novel),
        ('Dissuading Cues', gpt_oss_backfire_cues),
    ]
    
    # Define metrics to extract
    metrics_needed = [
        "all_greedy_bias_rate",
        "all_greedy_backfire_rate",
        "all_stochastic_flip_rate",
        "all_greedy_bias_rate_eligible_n",
        "all_greedy_backfire_rate_eligible_n",
        "all_n",
    ]
    
    val_col = f'test_round{round_num}'
    filtered_df = roundspread_df[roundspread_df['metric'].isin(metrics_needed)].copy()
    
    # Pivot to get metrics as columns per cue type
    pivot_df = filtered_df.pivot(
        index='counterfactual_type', 
        columns='metric', 
        values=val_col
    ).reset_index()
    
    # Drop 'all_ID' if present
    pivot_df = pivot_df[pivot_df['counterfactual_type'] != 'all_ID']
    
    # Extract values and compute CIs
    def get_vals_and_cis_for_cues(cue_list, rate_col, n_col):
        vals, ci_lows, ci_highs, ns = [], [], [], []
        for cue in cue_list:
            row = pivot_df[pivot_df['counterfactual_type'] == cue]
            if len(row) == 0:
                vals.append(np.nan)
                ci_lows.append(np.nan)
                ci_highs.append(np.nan)
                ns.append(0)
                continue
            v = row[rate_col].fillna(0).values[0]
            n = row[n_col].fillna(1).values[0] if n_col in row.columns else 1
            vals.append(v)
            ns.append(n)
            if n > 0 and not np.isnan(v):
                successes = int(np.round(v * n))
                successes = min(successes, int(n))
                low, high = proportion_confint(successes, n, alpha=0.05, method='wilson')
                ci_lows.append(low)
                ci_highs.append(high)
            else:
                ci_lows.append(v)
                ci_highs.append(v)
        return np.array(vals), np.array(ci_lows), np.array(ci_highs), np.array(ns)
    
    # Build ordered list of cues with sorting within each cluster
    ordered_cues = []
    cluster_boundaries = []  # x positions where vertical lines should go
    cluster_avg_positions = []  # x positions for cluster averages
    cluster_names_for_avg = []
    cluster_ranges = []  # (start_pos, end_pos, cluster_name) for text labels
    
    current_pos = 0
    for cluster_name, cluster_cues in clusters:
        # Get cues that exist in data, sorted by bias rate within cluster
        cluster_data = pivot_df[pivot_df['counterfactual_type'].isin(cluster_cues)].copy()
        cluster_data = cluster_data.sort_values('all_greedy_bias_rate', ascending=False)
        sorted_cues = cluster_data['counterfactual_type'].tolist()
        
        if len(sorted_cues) > 0:
            cluster_start = current_pos
            ordered_cues.extend(sorted_cues)
            # Position for cluster average (after all cues in cluster)
            cluster_avg_positions.append(current_pos + len(sorted_cues))
            cluster_names_for_avg.append(cluster_name)
            current_pos += len(sorted_cues) + 1  # +1 for avg bar
            cluster_boundaries.append(current_pos - 0.5)  # boundary after avg
            cluster_ranges.append((cluster_start, current_pos - 1, cluster_name))  # -1 because current_pos is now past avg
    
    # Remove last boundary (no line after last cluster)
    if cluster_boundaries:
        cluster_boundaries = cluster_boundaries[:-1]
    
    # Now compute values for all ordered cues + averages
    all_labels = []
    bias_vals, bias_ci_low, bias_ci_high = [], [], []
    backfire_vals, backfire_ci_low, backfire_ci_high = [], [], []
    flip_vals, flip_ci_low, flip_ci_high = [], [], []
    bias_ns, backfire_ns, flip_ns = [], [], []
    is_avg = []  # track which bars are averages
    
    cue_idx = 0
    for cluster_name, cluster_cues in clusters:
        cluster_data = pivot_df[pivot_df['counterfactual_type'].isin(cluster_cues)].copy()
        cluster_data = cluster_data.sort_values('all_greedy_bias_rate', ascending=False)
        sorted_cues = cluster_data['counterfactual_type'].tolist()
        
        if len(sorted_cues) == 0:
            continue
            
        # Add individual cues
        for cue in sorted_cues:
            all_labels.append(cue)
            is_avg.append(False)
            row = pivot_df[pivot_df['counterfactual_type'] == cue].iloc[0]
            
            # Bias
            v = row['all_greedy_bias_rate'] if not pd.isna(row['all_greedy_bias_rate']) else 0
            n = row['all_greedy_bias_rate_eligible_n'] if not pd.isna(row['all_greedy_bias_rate_eligible_n']) else 1
            bias_vals.append(v)
            bias_ns.append(n)
            if n > 0:
                succ = min(int(np.round(v * n)), int(n))
                low, high = proportion_confint(succ, n, alpha=0.05, method='wilson')
                bias_ci_low.append(low)
                bias_ci_high.append(high)
            else:
                bias_ci_low.append(v)
                bias_ci_high.append(v)
            
            # Backfire
            v = row['all_greedy_backfire_rate'] if not pd.isna(row['all_greedy_backfire_rate']) else 0
            n = row['all_greedy_backfire_rate_eligible_n'] if not pd.isna(row['all_greedy_backfire_rate_eligible_n']) else 1
            backfire_vals.append(v)
            backfire_ns.append(n)
            if n > 0:
                succ = min(int(np.round(v * n)), int(n))
                low, high = proportion_confint(succ, n, alpha=0.05, method='wilson')
                backfire_ci_low.append(low)
                backfire_ci_high.append(high)
            else:
                backfire_ci_low.append(v)
                backfire_ci_high.append(v)
            
            # Flip
            v = row['all_stochastic_flip_rate'] if not pd.isna(row['all_stochastic_flip_rate']) else 0
            n = row['all_n'] if not pd.isna(row['all_n']) else 1
            flip_vals.append(v)
            flip_ns.append(n)
            if n > 0:
                succ = min(int(np.round(v * n)), int(n))
                low, high = proportion_confint(succ, n, alpha=0.05, method='wilson')
                flip_ci_low.append(low)
                flip_ci_high.append(high)
            else:
                flip_ci_low.append(v)
                flip_ci_high.append(v)
        
        # Add cluster average
        all_labels.append(f"AVG")
        is_avg.append(True)
        
        # Compute weighted averages for cluster
        cluster_bias = cluster_data['all_greedy_bias_rate'].fillna(0).values
        cluster_bias_n = cluster_data['all_greedy_bias_rate_eligible_n'].fillna(0).values
        cluster_backfire = cluster_data['all_greedy_backfire_rate'].fillna(0).values
        cluster_backfire_n = cluster_data['all_greedy_backfire_rate_eligible_n'].fillna(0).values
        cluster_flip = cluster_data['all_stochastic_flip_rate'].fillna(0).values
        cluster_flip_n = cluster_data['all_n'].fillna(0).values
        
        # Weighted average for bias
        total_bias_n = cluster_bias_n.sum()
        avg_bias = (cluster_bias * cluster_bias_n).sum() / total_bias_n if total_bias_n > 0 else 0
        bias_vals.append(avg_bias)
        bias_ns.append(total_bias_n)
        if total_bias_n > 0:
            succ = min(int(np.round(avg_bias * total_bias_n)), int(total_bias_n))
            low, high = proportion_confint(succ, total_bias_n, alpha=0.05, method='wilson')
            bias_ci_low.append(low)
            bias_ci_high.append(high)
        else:
            bias_ci_low.append(avg_bias)
            bias_ci_high.append(avg_bias)
        
        # Weighted average for backfire
        total_backfire_n = cluster_backfire_n.sum()
        avg_backfire = (cluster_backfire * cluster_backfire_n).sum() / total_backfire_n if total_backfire_n > 0 else 0
        backfire_vals.append(avg_backfire)
        backfire_ns.append(total_backfire_n)
        if total_backfire_n > 0:
            succ = min(int(np.round(avg_backfire * total_backfire_n)), int(total_backfire_n))
            low, high = proportion_confint(succ, total_backfire_n, alpha=0.05, method='wilson')
            backfire_ci_low.append(low)
            backfire_ci_high.append(high)
        else:
            backfire_ci_low.append(avg_backfire)
            backfire_ci_high.append(avg_backfire)
        
        # Weighted average for flip
        total_flip_n = cluster_flip_n.sum()
        avg_flip = (cluster_flip * cluster_flip_n).sum() / total_flip_n if total_flip_n > 0 else 0
        flip_vals.append(avg_flip)
        flip_ns.append(total_flip_n)
        if total_flip_n > 0:
            succ = min(int(np.round(avg_flip * total_flip_n)), int(total_flip_n))
            low, high = proportion_confint(succ, total_flip_n, alpha=0.05, method='wilson')
            flip_ci_low.append(low)
            flip_ci_high.append(high)
        else:
            flip_ci_low.append(avg_flip)
            flip_ci_high.append(avg_flip)
    
    # Convert to arrays
    bias_vals, bias_ci_low, bias_ci_high = np.array(bias_vals), np.array(bias_ci_low), np.array(bias_ci_high)
    backfire_vals, backfire_ci_low, backfire_ci_high = np.array(backfire_vals), np.array(backfire_ci_low), np.array(backfire_ci_high)
    flip_vals, flip_ci_low, flip_ci_high = np.array(flip_vals), np.array(flip_ci_low), np.array(flip_ci_high)
    is_avg = np.array(is_avg)
    
    x_pos = np.arange(len(all_labels))
    
    # Color palette
    colors = {
        'bias': '#5CACDB',        # Blue
        'backfire': '#E69557',    # Orange
        'stochastic': '#969696',  # Gray
    }
    
    # Helper to compute non-negative error bars (scaled to percentage)
    def safe_yerr(vals, ci_low, ci_high):
        return [np.maximum(0, (vals - ci_low) * 100), np.maximum(0, (ci_high - vals) * 100)]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    if PLOT_BACKGROUND_COLOR:
        ax.set_facecolor(PLOT_BACKGROUND_COLOR)
        # fig.patch.set_facecolor(PLOT_BACKGROUND_COLOR)
    
    bar_width = 0.35
    offsets = [-bar_width/2, bar_width/2]
    
    # Plot bars (use hatching for averages) - only bias and backfire, scaled to %
    for i, (vals, ci_low, ci_high, color, label) in enumerate([
        (bias_vals, bias_ci_low, bias_ci_high, colors['bias'], 'Persuasion Rate'),
        (backfire_vals, backfire_ci_low, backfire_ci_high, colors['backfire'], 'Dissuasion Rate'),
    ]):
        # Regular bars with light black outline
        mask_regular = ~is_avg
        ax.bar(x_pos[mask_regular] + offsets[i], vals[mask_regular] * 100, bar_width, 
               label=label, color=color, edgecolor='black', linewidth=0.5,
               yerr=safe_yerr(vals[mask_regular], ci_low[mask_regular], ci_high[mask_regular]), 
               capsize=2, error_kw={'linewidth': 0.8, 'alpha': 0.7})
        # Average bars with hatching
        mask_avg = is_avg
        ax.bar(x_pos[mask_avg] + offsets[i], vals[mask_avg] * 100, bar_width, 
               color=color, hatch='///', edgecolor='black', linewidth=0.5, alpha=0.8,
               yerr=safe_yerr(vals[mask_avg], ci_low[mask_avg], ci_high[mask_avg]), 
               capsize=2, error_kw={'linewidth': 0.8, 'alpha': 0.7})
    
    # Add horizontal dotted line for stochastic flip rate (same across all cues), scaled to %
    # Use first non-avg value
    stochastic_flip_val = flip_vals[~is_avg][0] * 100 if any(~is_avg) else flip_vals[0] * 100
    ax.axhline(y=stochastic_flip_val, color=colors['stochastic'], linestyle=':', linewidth=2,
               label=f'Stochastic Flip Rate ({stochastic_flip_val:.1f}%)')
    
    # Add vertical delimiters (solid gray lines)
    for boundary in cluster_boundaries:
        ax.axvline(x=boundary, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)
    
    ax.set_xticks(x_pos)
    # Apply cue renaming dict for display
    display_labels = [cue_renaming_dict.get(lbl, lbl) for lbl in all_labels]
    ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=10, fontweight='bold')
    ax.set_xlabel('Cue Type', fontsize=18, fontweight='bold', labelpad=10)
    ax.set_ylabel('Rate (%)', fontsize=18, fontweight='bold', labelpad=10)
    
    # Fix bar centering - set xlim to remove extra space on sides
    ax.set_xlim(-0.5, len(all_labels) - 0.5)
    
    # Set y-axis (scaled to percentage)
    ax.set_ylim(0, 100)
    
    # Add cluster name labels at top center of each cluster
    y_label_pos = 97  # Position above the 100% line
    for start_pos, end_pos, cluster_name in cluster_ranges:
        center_x = (start_pos + end_pos) / 2
        ax.text(center_x, y_label_pos, cluster_name, ha='center', va='top', 
                fontsize=14, fontweight='bold', fontstyle='italic',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gray'))
    
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    
    # Add text labels above bars (only bias and backfire)
    # Position above CI upper bound (scaled to percentage)
    all_vals_list = [bias_vals, backfire_vals]
    all_ci_highs = [bias_ci_high, backfire_ci_high]
    for i in range(len(all_labels)):
        for j, (vals, ci_high) in enumerate(zip(all_vals_list, all_ci_highs)):
            if not np.isnan(vals[i]):
                # Position above CI upper bound (scaled to %)
                y_pos = ci_high[i] * 100 + 2
                ax.text(x_pos[i] + offsets[j], y_pos, f'{vals[i]*100:.1f}', 
                        ha='center', va='bottom', fontsize=11, rotation=0,
                        fontweight='bold' if is_avg[i] else 'normal')
    
    # Title
    if title is None:
        title = f'Cue Influence (Round {round_num})'
    ax.set_title(title, fontsize=24, fontweight='bold', pad=22)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.28)  # Make room for rotated labels + legend
    fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.14), ncol=3,
               frameon=False, prop={'weight': 'bold', 'size': 17})
    
    if save_path:
        fig.savefig(save_path, format='pdf', bbox_inches='tight', dpi=300)
        print(f"Saved figure to {save_path}")
    plt.show()
    
    # Print summary table
    if print_table:
        print(f"\n{'Cue Type':<30} | {'Bias Rate':<35} | {'Backfire Rate':<35} | {'Stochastic Flip':<35}")
        print("-" * 145)
        for i, cue in enumerate(all_labels):
            prefix = ">>> " if is_avg[i] else ""
            bias_str = f"n={int(bias_ns[i])}, {bias_vals[i]:.3f} [{bias_ci_low[i]:.3f}, {bias_ci_high[i]:.3f}]"
            backfire_str = f"n={int(backfire_ns[i])}, {backfire_vals[i]:.3f} [{backfire_ci_low[i]:.3f}, {backfire_ci_high[i]:.3f}]"
            flip_str = f"n={int(flip_ns[i])}, {flip_vals[i]:.3f} [{flip_ci_low[i]:.3f}, {flip_ci_high[i]:.3f}]"
            print(f"{prefix}{cue:<26} | {bias_str:<35} | {backfire_str:<35} | {flip_str:<35}")
            if is_avg[i] and i < len(all_labels) - 1:
                print("-" * 145)
        print()
        
    return pivot_df


# Run the clustered plot
for args_path, roundspread_df in dfs:
    print(f"\n{'='*60}")
    print(f"Clustered plot: {args_path}")
    # Extract model name from args_path for title
    if 'gpt-oss-120b' in args_path:
        model_name = 'gpt-oss-120b'
    elif 'qwen3-235b-a22b' in args_path or 'qwen-235' in args_path:
        model_name = 'Qwen3-235B-A22B'
    else:
        model_name = args_path.split('_')[0] if '_' in args_path else args_path
    title = f'Cue Influence for {model_name}'
    save_path = f'zfigs/cue_influence_clustered_{model_name}.pdf'
    result_df = plot_cue_influence_clustered(roundspread_df, round_num=0, figsize=(22, 7), print_table=False, title=title, save_path=save_path)
    print(f"{'='*60}")

In [ ]:
def plot_cue_influence(roundspread_df, round_num=0, figsize=(18, 8), print_table=False, title=None, save_path=None):
    """
    Plot bias and backfire rates by cue type, broken down by correction vs corruption.
    
    Shows vertical grouped bars for each cue type, ordered by total bias rate (descending, left to right).
    Stochastic flip rate shown as horizontal dotted line (same across all cues).
    
    Args:
        roundspread_df: DataFrame with metrics in 'metric' column and values in 'test_round{N}' columns
        round_num: Which round to plot (default 0)
        figsize: Figure size tuple
        title: Optional custom title
        save_path: Optional path to save figure as PDF
    """
    from globals import cue_renaming_dict
    # Define metrics to extract
    metrics_needed = [
        "all_greedy_bias_correction_rate",
        "all_greedy_bias_corruption_rate",
        "all_greedy_backfire_correction_rate",
        "all_greedy_backfire_corruption_rate",
        "all_stochastic_flip_rate",
        "all_greedy_bias_rate_eligible_n",
        "all_greedy_backfire_rate_eligible_n",
        "all_n",
    ]
    
    val_col = f'test_round{round_num}'
    filtered_df = roundspread_df[roundspread_df['metric'].isin(metrics_needed)].copy()
    
    # Pivot to get metrics as columns per cue type
    pivot_df = filtered_df.pivot(
        index='counterfactual_type', 
        columns='metric', 
        values=val_col
    ).reset_index()
    
    # Drop 'all_ID' if present
    pivot_df = pivot_df[pivot_df['counterfactual_type'] != 'all_ID']
    
    # Calculate total bias rate for sorting (correction + corruption = total bias rate)
    pivot_df['total_bias_rate'] = (
        pivot_df['all_greedy_bias_correction_rate'].fillna(0) + 
        pivot_df['all_greedy_bias_corruption_rate'].fillna(0)
    )
    
    # Sort by total bias rate descending (left to right)
    pivot_df = pivot_df.sort_values('total_bias_rate', ascending=False)
    
    cue_types = pivot_df['counterfactual_type'].values
    x_pos = np.arange(len(cue_types))
    
    # Extract values and compute CIs
    def get_vals_and_cis(rate_col, n_col):
        vals = pivot_df[rate_col].fillna(0).values
        ns = pivot_df[n_col].fillna(1).values
        
        ci_lows, ci_highs = [], []
        for v, n in zip(vals, ns):
            if n > 0 and not np.isnan(v):
                successes = int(np.round(v * n))
                successes = min(successes, int(n))  # Clamp to valid range
                low, high = proportion_confint(successes, n, alpha=0.05, method='wilson')
                ci_lows.append(low)
                ci_highs.append(high)
            else:
                ci_lows.append(v)
                ci_highs.append(v)
        return vals, np.array(ci_lows), np.array(ci_highs)
    
    # Get bias values
    bias_corr, bias_corr_ci_low, bias_corr_ci_high = get_vals_and_cis(
        'all_greedy_bias_correction_rate', 'all_greedy_bias_rate_eligible_n')
    bias_corrupt, bias_corrupt_ci_low, bias_corrupt_ci_high = get_vals_and_cis(
        'all_greedy_bias_corruption_rate', 'all_greedy_bias_rate_eligible_n')
    
    # Get backfire values
    backfire_corr, backfire_corr_ci_low, backfire_corr_ci_high = get_vals_and_cis(
        'all_greedy_backfire_correction_rate', 'all_greedy_backfire_rate_eligible_n')
    backfire_corrupt, backfire_corrupt_ci_low, backfire_corrupt_ci_high = get_vals_and_cis(
        'all_greedy_backfire_corruption_rate', 'all_greedy_backfire_rate_eligible_n')
    
    # Get stochastic flip rate (use all_n for CI)
    flip_rate = pivot_df['all_stochastic_flip_rate'].fillna(0).values
    flip_n = pivot_df['all_n'].fillna(1).values
    flip_ci_lows, flip_ci_highs = [], []
    for v, n in zip(flip_rate, flip_n):
        if n > 0 and not np.isnan(v):
            successes = int(np.round(v * n))
            successes = min(successes, int(n))
            low, high = proportion_confint(successes, n, alpha=0.05, method='wilson')
            flip_ci_lows.append(low)
            flip_ci_highs.append(high)
        else:
            flip_ci_lows.append(v)
            flip_ci_highs.append(v)
    flip_ci_lows, flip_ci_highs = np.array(flip_ci_lows), np.array(flip_ci_highs)
    
    # Color palette - same colors for persuasion and dissuasion, hatching differentiates
    colors = {
        'correction': '#2ecc71',      # Green - correction is good
        'corruption': '#e74c3c',      # Red - corruption is bad
        'stochastic_flip': '#7f8c8d', # Gray
    }
    
    # Helper to compute non-negative error bars (scaled to percentage)
    def safe_yerr(vals, ci_low, ci_high):
        return [np.maximum(0, (vals - ci_low) * 100), np.maximum(0, (ci_high - vals) * 100)]
    
    # Create figure with two subplots stacked
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize)
    if PLOT_BACKGROUND_COLOR:
        ax1.set_facecolor(PLOT_BACKGROUND_COLOR)
        ax2.set_facecolor(PLOT_BACKGROUND_COLOR)
        # fig.patch.set_facecolor(PLOT_BACKGROUND_COLOR)
    
    # Bar positioning - just 2 bars now
    bar_width = 0.35
    offset1, offset2 = -bar_width/2, bar_width/2
    
    # Get stochastic flip rate (same across all cues - use first value), scaled to %
    stochastic_flip_val = flip_rate[0] * 100
    
    # --- Top subplot: Persuasion rates (scaled to %) ---
    ax1.bar(x_pos + offset1, bias_corr * 100, bar_width, 
            label='Persuasion Correction', color=colors['correction'], 
            edgecolor='black', linewidth=0.5,
            yerr=safe_yerr(bias_corr, bias_corr_ci_low, bias_corr_ci_high), 
            capsize=2, error_kw={'linewidth': 0.8, 'alpha': 0.7})
    ax1.bar(x_pos + offset2, bias_corrupt * 100, bar_width,
            label='Persuasion Corruption', color=colors['corruption'],
            edgecolor='black', linewidth=0.5,
            yerr=safe_yerr(bias_corrupt, bias_corrupt_ci_low, bias_corrupt_ci_high),
            capsize=2, error_kw={'linewidth': 0.8, 'alpha': 0.7})
    ax1.axhline(y=stochastic_flip_val, color=colors['stochastic_flip'], linestyle=':', linewidth=2,
                label=f'Stochastic Flip ({stochastic_flip_val:.1f}%)')
    
    # Apply cue renaming dict for display
    display_labels = [cue_renaming_dict.get(cue, cue) for cue in cue_types]
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=10, fontweight='bold')
    ax1.set_ylabel('Rate (%)', fontsize=18, fontweight='bold', labelpad=10)
    ax1.set_title('Persuasion Rates by Cue Type', fontsize=18, fontweight='bold', pad=10)
    ax1.set_ylim(0, 100)
    ax1.set_xlim(-0.5, len(cue_types) - 0.5)
    ax1.grid(True, axis='y', linestyle='--', alpha=0.4)
    
    # Add text labels above bars (above CIs, scaled to %)
    for i, (bc, bcrpt) in enumerate(zip(bias_corr, bias_corrupt)):
        y_pos1 = bias_corr_ci_high[i] * 100 + 2
        y_pos2 = bias_corrupt_ci_high[i] * 100 + 2
        ax1.text(x_pos[i] + offset1, y_pos1, f'{bc*100:.1f}', ha='center', va='bottom', fontsize=11)
        ax1.text(x_pos[i] + offset2, y_pos2, f'{bcrpt*100:.1f}', ha='center', va='bottom', fontsize=11)
    
    # --- Bottom subplot: Dissuasion rates (hatched, scaled to %) ---
    ax2.bar(x_pos + offset1, backfire_corr * 100, bar_width,
            label='Dissuasion Correction', color=colors['correction'],
            edgecolor='black', linewidth=0.5, hatch='///',
            yerr=safe_yerr(backfire_corr, backfire_corr_ci_low, backfire_corr_ci_high),
            capsize=2, error_kw={'linewidth': 0.8, 'alpha': 0.7})
    ax2.bar(x_pos + offset2, backfire_corrupt * 100, bar_width,
            label='Dissuasion Corruption', color=colors['corruption'],
            edgecolor='black', linewidth=0.5, hatch='///',
            yerr=safe_yerr(backfire_corrupt, backfire_corrupt_ci_low, backfire_corrupt_ci_high),
            capsize=2, error_kw={'linewidth': 0.8, 'alpha': 0.7})
    ax2.axhline(y=stochastic_flip_val, color=colors['stochastic_flip'], linestyle=':', linewidth=2,
                label=f'Stochastic Flip ({stochastic_flip_val:.1f}%)')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=10, fontweight='bold')
    ax2.set_xlabel('Cue Type', fontsize=16, fontweight='bold', labelpad=10)
    ax2.set_ylabel('Rate (%)', fontsize=16, fontweight='bold', labelpad=10)
    ax2.set_title('Dissuasion Rates by Cue Type', fontsize=18, fontweight='bold', pad=10)
    ax2.set_ylim(0, 100)
    ax2.set_xlim(-0.5, len(cue_types) - 0.5)
    ax2.grid(True, axis='y', linestyle='--', alpha=0.4)
    
    # Add text labels above bars (above CIs, scaled to %)
    for i, (bfc, bfcrpt) in enumerate(zip(backfire_corr, backfire_corrupt)):
        y_pos1 = backfire_corr_ci_high[i] * 100 + 2
        y_pos2 = backfire_corrupt_ci_high[i] * 100 + 2
        ax2.text(x_pos[i] + offset1, y_pos1, f'{bfc*100:.1f}', ha='center', va='bottom', fontsize=11)
        ax2.text(x_pos[i] + offset2, y_pos2, f'{bfcrpt*100:.1f}', ha='center', va='bottom', fontsize=11)
    
    # Overall title
    fig.suptitle(title, fontsize=24, fontweight='bold', y=1)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.18)  # Make room for legend
    
    # Combined legend at the bottom
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    # Combine unique handles (stochastic flip is in both)
    all_handles = handles1 + [h for h, l in zip(handles2, labels2) if l not in labels1]
    all_labels = labels1 + [l for l in labels2 if l not in labels1]
    fig.legend(all_handles, all_labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=5,
               frameon=False, prop={'weight': 'bold', 'size': 14})
    
    if save_path:
        fig.savefig(save_path, format='pdf', bbox_inches='tight', dpi=300)
        print(f"Saved figure to {save_path}")
    plt.show()
    
    return pivot_df


# Run the plot
for args_path, roundspread_df in dfs:
    print(f"\n{'='*60}")
    print(f"Plotting: {args_path}")
    # Extract model name from args_path for title
    if 'gpt-oss-120b' in args_path:
        model_name = 'gpt-oss-120b'
    elif 'qwen3-235b-a22b' in args_path or 'qwen-235' in args_path:
        model_name = 'Qwen3-235B-A22B'
    else:
        model_name = args_path.split('_')[0] if '_' in args_path else args_path
    title = f'Persuasion/Dissuasion Rates for {model_name}'
    save_path = f'zfigs/cue_influence_persuasion_dissuasion_{model_name}.pdf'
    result_df = plot_cue_influence(roundspread_df, round_num=0, figsize=(18, 12), title=title, save_path=save_path)
    print(f"{'='*60}")